# CoSched-GNN: consensus-aware offloading and validator co-selection

**How to run on Kaggle**
1. *Add Input* → the Alibaba cluster-trace dataset (it mounts under `/kaggle/input`).
2. *Settings → Accelerator → GPU*.
3. Edit the command in the **Run** cell (epochs, seed, …), then *Run All*.

The first cell(s) only write the code to files; nothing is trained until the **Run** cell.
The loader finds `batch_task*.csv` (or `batch_instance*.csv`) in `/kaggle/input` automatically;
add `--data /kaggle/input/<folder>/batch_task.csv` to the command if it doesn't.

**Command-line options (both scripts)**

| Option | Meaning |
|---|---|
| `--epochs N` | training epochs per algorithm (**required**; 1 epoch = one 300-slot simulation run) |
| `--seed S` | random seed (default 42) |
| `--vehicles N` | vehicles for training and Fig. 3 (default 10) |
| `--horizon T` | time slots per epoch (default 300) |
| `--eval-windows W` | evaluation windows averaged per value (default 3) |
| `--data PATH` | path to the trace file (default: auto-detect) |
| `--out DIR` | output folder |

**CoSched-GNN options (`cosched_gnn.py`)**

| Option | Meaning |
|---|---|
| `--experiments main,ablation,pqc,hidden,scaling` | which experiments to run (default all) |
| `--methods CoSched-GNN,TaBRL,...` | methods in the main comparison (default: CoSched-GNN, TaBRL, FE-CTDE-style, HGAT-style, MLP-Joint, HEFT+Random, DAG-Local; CP-Greedy+Post-load (ablation A6), TaBRL-GNN, MA-LHTO, IOPO, DECO can be added) |
| `--seeds 43,44` | repeat everything per seed and report mean +- std and a t-test vs TaBRL |
| `--servers M` | edge servers (default 8; executor exclusion needs M >= n + servers used) |
| `--pbft-f F` | faults tolerated, committee n = 3F + 1 (default 1) |
| `--sig dilithium2` | signature scheme: ecdsa (classical), dilithium2/3/5 = ML-DSA-44/65/87 (post-quantum, default ML-DSA-44) |
| `--set crypto=model` | skip the real cryptographic operations (same results, faster; costs still measured) |
| `--hidden 128 --layers 3` | GNN size |
| `--rollout 2048` | PPO decisions per update |

## Plan for the final results (one Kaggle notebook copy per run, all with the same N and M)

| Run | `RUN` | `SEEDS` | What it produces | Approx. time (150 epochs, N = 15) |
|---|---|---|---|---|
| A | `"main"` | `"43"` | main comparison, Fig. 3/4, consensus & security, post-quantum, consensus-delay sensitivity | 6-8 h |
| B | `"ablation"` | `"43"` | ablations A1-A6 | 6-9 h |
| C | `"seed"` | `"44,45"` | main comparison only, two more seeds | 7-10 h |
| D | `"seed"` | `"46,47"` | main comparison only, two more seeds | 7-10 h |
| E | merge cell (no GPU) | | `seeds_summary.csv` = mean +- std over seeds 43-47 + t-test vs TaBRL | 1 min |

Seeds are merged only if they use the same number of vehicles, servers and f (the merge script checks).
Use **Save Version -> Save & Run All (Commit)** so the run continues in the background (up to 12 h)
and its files are kept in the version's *Output* tab. Settings: *Accelerator: GPU*, *Internet: On*,
input: the Alibaba cluster-trace dataset.

## Settings for this run (change only these lines)

In [ ]:
RUN = "main"       # "main" (Run A), "ablation" (Run B), "seed" (Runs C, D: extra seeds)
SEEDS = "43"       # one seed, or several separated by commas, e.g. "44,45"
VEHICLES = 15      # number of vehicles N
SERVERS = 10       # number of edge servers M
EPOCHS = 150
EXPS = {"main": "main,pqc,hidden", "ablation": "ablation", "seed": "main"}[RUN]
OUT = f"/kaggle/working/cosched_{RUN}_N{VEHICLES}_M{SERVERS}"
print(RUN, SEEDS, VEHICLES, SERVERS, EPOCHS, EXPS, OUT)

## Post-quantum library
Installs liboqs (Open Quantum Safe; built on first import, needs *Settings -> Internet -> On*, about 3-5 min) and the `pqcrypto` wheel as a fallback.

In [ ]:
!pip install -q liboqs-python pqcrypto
import oqs; print('liboqs', oqs.oqs_version())

## Code
Writes the simulator, the post-quantum layer, CoSched-GNN with all baselines and experiments, the unit tests, the seed-merging script and the figure-redrawing script.

In [ ]:
%%writefile tabrl_iov.py
# %% [markdown]
# # TaBRL: Trust-Aware Blockchain Reinforcement Learning for Secure Task Offloading in Edge-Assisted IoV
#
# Re-implementation of Yin et al., IEEE Communications Standards Magazine (2026),
# DOI 10.1109/MCOMSTD.2026.3653017, driven by the **Alibaba Cluster Trace v2018**
# (`batch_task.csv`, e.g. the Kaggle upload "alibaba cluster trace part 2").
#
# Nothing in this file is a hard-coded result: every number in the figures comes
# from simulating the trained policies on DAG jobs taken from the trace.
#
# Pipeline
# 1. Load `batch_task.csv` (or `batch_instance.csv`), rebuild each job's DAG from the
#    task names (`M1`, `R2_1`, `J3_1_2` -> task 3 depends on tasks 1 and 2).
# 2. Scale CPU work / memory into subtask CPU cycles and input bits, discretise the
#    arrival timestamps into time slots and split jobs chronologically: 70 % train, 30 % test.
# 3. Simulate the blockchain-assisted IoV system: N vehicles, M roadside edge servers (RES)
#    with a multi-level feedback queue (MFQ), V2I channel, consortium-blockchain confirmation
#    delay, vehicle task and energy queues (Lyapunov), and compromised RESs that tamper with
#    results and misreport their load.
# 4. Train TaBRL and the baselines MA-LHTO, IOPO and DECO; DAG-Local is the all-local reference.
# 5. Evaluate on train and test windows -> Fig. 3 (completion time + processing capacity bars)
#    and Fig. 4 (backlog + energy over time for N = 5 and N = 15).

# %%
import os, re, glob, math, time, json, copy, random
from collections import deque

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

CFG = dict(
    # ---------------- data ----------------
    data_path=None,            # None -> search /kaggle/input recursively for batch_task*.csv
    max_rows=4_000_000,        # rows read from the trace (batch_task has ~14M rows)
    max_jobs=40_000,           # earliest valid DAG jobs kept
    allow_synthetic=False,     # True only for offline smoke tests (never for reported results)
    k_max=8, k_min=2,          # subtasks per DAG job
    train_frac=0.7,
    cyc_range=(0.2e9, 2.4e9),  # CPU cycles per subtask after scaling
    bits_range=(0.5e6, 6e6),   # input bits per subtask after scaling
    # ---------------- system ----------------
    n_veh=10, n_res=4, res_cores=2,
    res_freq=(2.0e9, 3.0e9, 4.0e9, 5.0e9),   # heterogeneous RES CPU (Hz per core)
    f_local_max=2.5e9, f_local_min=0.5e9, kappa=1e-27,
    slot=1.0, dt=0.1, horizon=300, drain_max=900,
    arrival_rate=0.30,                 # mean DAG jobs per vehicle per slot
    mfq_slices=(0.2, 0.5, 1.0),        # Low / Mid / High queue time slices (s)
    road_len=2000.0, rsu_offset=15.0, speed=(10.0, 25.0),
    bandwidth=2e6, p_tx=0.2, noise_dbm=-174.0, shadow_db=4.0, overhead=1.2,
    backhaul_bps=100e6, backhaul_hop_s=0.004,
    blk_interval=0.2, blk_capacity=24, blk_size=1e6, blk_rate=20e6,
    # compromised RES: which RES is compromised is re-drawn every episode, so a policy
    # cannot memorise it and has to detect it online (only TaBRL sees on-chain trust)
    n_malicious=1, p_tamper=0.35, report_factor=0.25,
    dispute_blocks=5,          # extra confirmation blocks before a failed proof is finalised
    energy_replenish=1.5, energy_v=1.0,
    # ---------------- Lyapunov reward ----------------
    V=1.0, alpha_q=0.05, alpha_e=0.03, beta_trust=0.3,
    trust_lambda=2.0, trust_min=0.8,   # TaBRL: trust-weighted RES selection / filtering
    # ---------------- MA-LHTO (He et al., IEEE TMC 2026) ----------------
    malhto_hidden=128, malhto_entropy=0.1, malhto_xi=0.01,   # Θ = 0.1, ξ = 0.01 as selected in the paper
    malhto_reset_every=1000, malhto_reset_std=0.01,          # δ_reset, σ: not given in the paper
    # ---------------- learning ----------------
    train_episodes=None,       # set on the command line: --epochs N
    batch=128, buffer=50_000, lr_actor=3e-4, lr_critic=1e-3,
    hidden=256, train_every=1, tau=0.01, n_candidates=32,
    eps_start=0.5, eps_end=0.03,
    # ---------------- evaluation ----------------
    eval_windows=3, fig4_horizon=150, fig4_veh=(5, 15),
    seed=42, out_dir="results",
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# %% [markdown]
# ## 1. Alibaba Cluster Trace v2018 -> DAG jobs

# %%
TASK_COLS = ["task_name", "instance_num", "job_name", "task_type", "status",
             "start_time", "end_time", "plan_cpu", "plan_mem"]
INST_COLS = ["instance_name", "task_name", "job_name", "task_type", "status", "start_time",
             "end_time", "machine_id", "seq_no", "total_seq_no", "cpu_avg", "cpu_max",
             "mem_avg", "mem_max"]
NAME_RE = r"^[A-Za-z]+(\d+)((?:_\d+)*)$"


def _looks_like_trace(fp):
    """True if the file has the batch_task (9 columns) or batch_instance (14 columns) layout."""
    try:
        head = pd.read_csv(fp, header=None, nrows=3, low_memory=False)
    except Exception:
        return False
    first = [str(v).strip().lower() for v in head.iloc[0].values]
    return head.shape[1] in (9, 14) or "task_name" in first


def find_trace_file(path=None):
    if path and os.path.isfile(path):
        return path
    roots = [path] if path else ["/kaggle/input", ".", "./data"]
    files = []
    for r in roots:
        if r and os.path.isdir(r):
            for ext in ("*.csv", "*.csv.gz", "*.tar.gz"):
                files += glob.glob(os.path.join(r, "**", ext), recursive=True)
    for key in ("batch_task", "batch_instance"):
        hit = [f for f in files if key in os.path.basename(f).lower()]
        if hit:
            return sorted(hit, key=os.path.getsize)[-1]
    # dataset uploaded under another file name: only look in the input folder (never in our own
    # output folders) and only accept a file with the trace's column layout
    inputs = [f for f in files if f.startswith(path or "/kaggle/input") and f.endswith(".csv")]
    inputs = [f for f in sorted(inputs, key=os.path.getsize, reverse=True) if _looks_like_trace(f)]
    return inputs[0] if inputs else None


def _read_trace(fp, max_rows):
    head = pd.read_csv(fp, header=None, nrows=3, low_memory=False)
    has_header = any(str(v).strip().lower() in ("task_name", "job_name", "instance_name")
                     for v in head.iloc[0].values)
    ncol = head.shape[1]
    if has_header:
        df = pd.read_csv(fp, nrows=max_rows, low_memory=False)
        df.columns = [c.strip().lower() for c in df.columns]
        kind = "instance" if "instance_name" in df.columns else "task"
    else:
        kind = "instance" if ncol == 14 else "task"
        cols = INST_COLS if kind == "instance" else TASK_COLS
        if ncol != len(cols):
            raise ValueError(f"{fp}: unexpected column count {ncol}")
        df = pd.read_csv(fp, header=None, names=cols, nrows=max_rows, low_memory=False)
    if kind == "instance":   # aggregate instances into tasks
        df = (df.groupby(["job_name", "task_name"], as_index=False)
                .agg(instance_num=("start_time", "size"), task_type=("task_type", "first"),
                     status=("status", "first"), start_time=("start_time", "min"),
                     end_time=("end_time", "max"), plan_cpu=("cpu_avg", "mean"),
                     plan_mem=("mem_avg", "mean")))
    return df, kind


def _log_scale(v, lo, hi):
    lv = np.log1p(v)
    a, b = np.nanpercentile(lv, 1), np.nanpercentile(lv, 99)
    z = np.clip((lv - a) / max(b - a, 1e-9), 0, 1)
    return lo + z * (hi - lo)


def load_dag_jobs(cfg):
    fp = find_trace_file(cfg["data_path"])
    if fp is None:
        if not cfg["allow_synthetic"]:
            raise FileNotFoundError(
                "Alibaba trace not found. Add the Kaggle dataset to the notebook "
                "(it appears under /kaggle/input) or set CFG['data_path'] to batch_task.csv.")
        print("!! No trace found - using a SYNTHETIC trace for a smoke test only.")
        df, kind = synthetic_batch_task(cfg["max_jobs"], cfg["seed"], cfg["k_max"]), "synthetic"
    else:
        print(f"Loading trace: {fp}")
        df, kind = _read_trace(fp, cfg["max_rows"])
    print(f"  {len(df):,} task rows ({kind})")

    df = df.dropna(subset=["task_name", "job_name", "start_time", "end_time"])
    if "status" in df.columns and (df["status"] == "Terminated").any():
        df = df[df["status"] == "Terminated"]
    df = df[(df["start_time"] > 0) & (df["end_time"] >= df["start_time"])].copy()
    ext = df["task_name"].astype(str).str.extract(NAME_RE)
    df["tid"] = pd.to_numeric(ext[0], errors="coerce")
    df["deps"] = ext[1].fillna("")
    ok = df["tid"].notna().groupby(df["job_name"]).transform("all")
    df = df[ok]
    size = df.groupby("job_name")["tid"].transform("size")
    df = df[(size >= cfg["k_min"]) & (size <= cfg["k_max"])].copy()
    # keep only the earliest jobs (with margin for ones rejected below) before building DAGs
    first = df.groupby("job_name")["start_time"].min().nsmallest(int(cfg["max_jobs"] * 1.2))
    df = df[df["job_name"].isin(first.index)].copy()

    inst = pd.to_numeric(df["instance_num"], errors="coerce").fillna(1).clip(lower=1)
    cpu = pd.to_numeric(df["plan_cpu"], errors="coerce").fillna(100).clip(lower=1)
    mem = pd.to_numeric(df["plan_mem"], errors="coerce").fillna(0.5).clip(lower=0.01)
    dur = (df["end_time"] - df["start_time"]).clip(lower=1)
    df["cycles"] = _log_scale((inst * cpu / 100.0 * dur).values, *cfg["cyc_range"])
    df["bits"] = _log_scale((inst * mem).values, *cfg["bits_range"])

    jobs = []
    for jn, g in df.groupby("job_name", sort=False):
        ids = g["tid"].astype(int).tolist()
        if len(set(ids)) != len(ids):
            continue
        idx = {t: i for i, t in enumerate(ids)}
        preds = [[idx[int(d)] for d in dep.split("_") if d and int(d) in idx and int(d) != t]
                 for t, dep in zip(ids, g["deps"])]
        order = topo_order(preds)
        if order is None:
            continue
        pos = {o: i for i, o in enumerate(order)}   # re-index in topological order
        jobs.append(dict(
            start=float(g["start_time"].min()),
            b=[float(g["cycles"].iloc[o]) for o in order],
            d=[float(g["bits"].iloc[o]) for o in order],
            preds=[sorted(pos[p] for p in preds[o]) for o in order]))
    jobs.sort(key=lambda j: j["start"])
    jobs = jobs[: cfg["max_jobs"]]
    n_tr = int(len(jobs) * cfg["train_frac"])
    ks = [len(j["b"]) for j in jobs]
    print(f"  {len(jobs):,} DAG jobs ({np.mean(ks):.2f} subtasks avg), train {n_tr:,} / test {len(jobs) - n_tr:,}")
    return jobs[:n_tr], jobs[n_tr:]


def topo_order(preds):
    n = len(preds)
    indeg = [len(p) for p in preds]
    succ = [[] for _ in range(n)]
    for i, ps in enumerate(preds):
        for p in ps:
            succ[p].append(i)
    q = deque(i for i in range(n) if indeg[i] == 0)
    out = []
    while q:
        u = q.popleft(); out.append(u)
        for v in succ[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return out if len(out) == n else None


def synthetic_batch_task(n_jobs, seed, k_hi=8):
    """Trace-shaped stand-in used ONLY when allow_synthetic=True (offline smoke test)."""
    rng = np.random.default_rng(seed)
    rows, t = [], 86400
    for j in range(n_jobs):
        t += int(rng.exponential(3.0))
        k = int(rng.integers(2, 9)) if k_hi <= 8 else int(min(k_hi, 2 + rng.geometric(1 / 6)))
        for i in range(1, k + 1):
            deps = sorted(rng.choice(np.arange(1, i), size=min(i - 1, int(rng.integers(0, 3))),
                                     replace=False)) if i > 1 else []
            name = rng.choice(list("MRJ")) + str(i) + "".join(f"_{d}" for d in deps)
            dur = int(rng.lognormal(3.5, 1.2)) + 1
            rows.append([name, int(rng.lognormal(2.0, 1.5)) + 1, f"j_{j}", 1, "Terminated",
                         t, t + dur, float(rng.choice([50, 100, 200])), float(rng.uniform(0.1, 1.5))])
    return pd.DataFrame(rows, columns=TASK_COLS)


def make_window(jobs, n_veh, horizon, cfg, rng):
    """Discretise trace arrivals into slots and give each job to a free vehicle.
    Trace time is compressed so the mean load is arrival_rate jobs / vehicle / slot,
    while the burstiness of the trace is kept."""
    need = int(horizon * n_veh * cfg["arrival_rate"] * 1.3) + 10
    s0 = int(rng.integers(0, max(1, len(jobs) - need)))
    seg = jobs[s0: s0 + need]
    starts = np.array([j["start"] for j in seg])
    gap = max((starts[-1] - starts[0]) / max(len(seg) - 1, 1), 1e-6)
    trace_slot = gap * n_veh * cfg["arrival_rate"]
    slots = ((starts - starts[0]) / trace_slot).astype(int)
    sched = [[] for _ in range(horizon)]
    carry = deque()
    ji = 0
    for s in range(horizon):
        while ji < len(seg) and slots[ji] <= s:
            carry.append(seg[ji]); ji += 1
        free = list(rng.permutation(n_veh))
        while carry and free:
            sched[s].append((int(free.pop()), carry.popleft()))
    return sched


# %% [markdown]
# ## 2. Blockchain-assisted IoV environment

# %%
class Blockchain:
    """Consortium chain: a transaction waits for the next block with free space,
    then pays D_blk / R_blk to propagate and reach consensus."""
    def __init__(self, cfg):
        self.tau, self.cap = cfg["blk_interval"], cfg["blk_capacity"]
        self.prop = cfg["blk_size"] / cfg["blk_rate"]
        self.used = {}

    def confirm(self, t):
        b = int(math.ceil(t / self.tau + 1e-9))
        while self.used.get(b, 0) >= self.cap:
            b += 1
        self.used[b] = self.used.get(b, 0) + 1
        return b * self.tau + self.prop - t

    def est_delay(self, t):
        b = int(math.ceil(t / self.tau + 1e-9))
        n = 0
        while self.used.get(b + n, 0) >= self.cap:
            n += 1
        return (b + n) * self.tau + self.prop - t


class Sub:
    __slots__ = ("task", "k", "b", "d", "preds", "x", "res", "rem", "level", "done_at",
                 "avail_at", "f")

    def __init__(self, task, k, b, d):
        self.task, self.k, self.b, self.d = task, k, b, d
        self.preds, self.x, self.res, self.rem, self.level = [], 0, -1, b, 0
        self.done_at, self.avail_at, self.f = None, 0.0, 1e9

    def ready(self, t):
        return all(p.done_at is not None and p.done_at <= t + 1e-9 for p in self.preds)


class Task:
    def __init__(self, vid, arr, job):
        self.vid, self.arr = vid, arr
        self.subs = [Sub(self, k, b, d) for k, (b, d) in enumerate(zip(job["b"], job["d"]))]
        for s, ps in zip(self.subs, job["preds"]):
            s.preds = [self.subs[p] for p in ps]
        self.size = sum(job["d"])
        self.n_done, self.complete, self.energy = 0, None, 0.0
        self.obs = self.act = None
        self.info = {}


class RES:
    def __init__(self, j, cfg, malicious):
        self.j, self.cfg, self.mal = j, cfg, malicious
        fr = cfg["res_freq"]
        self.F = float(fr[j % len(fr)]) if isinstance(fr, (list, tuple)) else float(fr)
        self.slices = cfg["mfq_slices"]
        self.incoming, self.pending = [], []
        self.q = [deque(), deque(), deque()]
        self.cores = [[None, 0.0] for _ in range(cfg["res_cores"])]   # [job, slice left]
        self.ok = self.fail = 0

    def backlog_cycles(self):
        c = sum(s.rem for s in self.incoming) + sum(s.rem for s in self.pending)
        c += sum(s.rem for q in self.q for s in q)
        return c + sum(core[0].rem for core in self.cores if core[0] is not None)

    def n_jobs(self):
        return (len(self.incoming) + len(self.pending) + sum(len(q) for q in self.q)
                + sum(core[0] is not None for core in self.cores))

    def trust(self):
        # reputation with an optimistic prior: 1.0 until verification failures are recorded
        return (self.ok + 2.0) / (self.ok + self.fail + 2.0)

    def step(self, t, dt, env):
        if self.incoming:
            keep = []
            for s in self.incoming:
                (self.pending if s.avail_at <= t else keep).append(s)
            self.incoming = keep
        if self.pending:
            keep = []
            for s in self.pending:
                if s.ready(t):
                    s.level = 0; self.q[0].append(s)
                else:
                    keep.append(s)
            self.pending = keep
        for core in self.cores:
            budget = dt
            while budget > 1e-9:
                if core[0] is None:
                    lvl = next((l for l in range(3) if self.q[l]), None)
                    if lvl is None:
                        break
                    core[0], core[1] = self.q[lvl].popleft(), self.slices[lvl]
                s = core[0]
                run = min(budget, s.rem / self.F, core[1])
                s.rem -= run * self.F; budget -= run; core[1] -= run
                if s.rem <= 1.0:
                    core[0] = None
                    env.res_finished(self, s, t + dt - budget)
                elif core[1] <= 1e-9:          # time slice used up -> demote (MFQ)
                    s.level = min(s.level + 1, 2)
                    self.q[s.level].append(s); core[0] = None


class Vehicle:
    def __init__(self, vid, cfg, rng):
        self.vid, self.cfg = vid, cfg
        self.x = float(rng.uniform(0, cfg["road_len"]))
        self.y = float(rng.uniform(0, 10))
        self.v = float(rng.uniform(*cfg["speed"])) * (1 if rng.random() < 0.5 else -1)
        self.local = deque()
        self.tx_free, self.tx_bits = 0.0, []    # (start, end, bits) of uplink transfers
        self.Q = self.E = 0.0
        self.slot_energy = self.total_energy = 0.0

    def step(self, t, dt, env):
        budget, now = dt, t
        while budget > 1e-9 and self.local:
            s = self.local[0]
            if s.avail_at > now + 1e-9 or not s.ready(now):
                break
            run = min(budget, s.rem / s.f)
            s.rem -= run * s.f; budget -= run; now += run
            e = self.cfg["kappa"] * s.f ** 3 * run
            self.slot_energy += e; s.task.energy += e
            if s.rem <= 1.0:
                self.local.popleft()
                env.sub_done(s, now)

    def queued_bits(self, t):
        q = sum(s.d * s.rem / s.b for s in self.local)
        for a, b, bits in self.tx_bits:
            if b > t:
                q += bits * min(1.0, (b - t) / max(b - a, 1e-9))
        return q


class IoVEnv:
    def __init__(self, cfg, n_veh, seed):
        self.cfg, self.N, self.M = cfg, n_veh, cfg["n_res"]
        self.K = cfg["k_max"]
        self.rng = np.random.default_rng(seed)
        mal = set(self.rng.choice(self.M, cfg["n_malicious"], replace=False).tolist())
        self.res = [RES(j, cfg, j in mal) for j in range(self.M)]
        self.veh = [Vehicle(i, cfg, self.rng) for i in range(n_veh)]
        self.rsu = np.array([[cfg["road_len"] * (j + 0.5) / self.M, cfg["rsu_offset"]] for j in range(self.M)])
        self.chain = Blockchain(cfg)
        self.t, self.tasks, self.completed = 0.0, [], []
        self.shadow = self.rng.normal(0, cfg["shadow_db"], n_veh)

    # ---- channel ----
    def nearest(self, v):
        d = np.hypot(self.rsu[:, 0] - v.x, self.rsu[:, 1] - v.y)
        j = int(np.argmin(d)); return j, float(d[j])

    def rate(self, v):
        c = self.cfg
        _, d = self.nearest(v)
        pl = 128.1 + 37.6 * math.log10(max(d, 10.0) / 1000.0) + self.shadow[v.vid]
        noise = 10 ** ((c["noise_dbm"] - 30) / 10) * c["bandwidth"]
        snr = c["p_tx"] * 10 ** (-pl / 10) / noise
        return c["bandwidth"] * math.log2(1 + snr)

    # ---- observation ----
    def obs_dim(self):
        return 6 * self.K + 6 + 5 * self.M + 2

    def _obs_parts(self, vid, task, mode):
        """Per-subtask features, DAG depth, vehicle, per-RES and global features."""
        v, c = self.veh[vid], self.cfg
        depth, tasks = [], []
        for k, s in enumerate(task.subs):
            dp = 1 + max((depth[p.k] for p in s.preds), default=-1)
            depth.append(dp)
            n_succ = sum(s in o.preds for o in task.subs)
            tasks.append((s.b / 1e9, s.d / 1e6, dp, len(s.preds), n_succ))
        near, _ = self.nearest(v)
        vf = [v.Q / 1e7, v.E / 10.0, sum(s.rem for s in v.local) / c["f_local_max"] / 10.0,
              max(v.tx_free - self.t, 0) / 5.0, self.rate(v) / 2e7, near / self.M]
        rf = []
        for j, r in enumerate(self.res):
            load = r.backlog_cycles() / (r.F * len(r.cores))
            if mode != "audited" and r.mal:     # self-reported load of a compromised RES
                load *= c["report_factor"]
            tr = r.trust() if mode == "audited" else 0.5
            rf.append((load / 10.0, r.n_jobs() / 20.0, tr, abs(near - j), r.F / 5e9))
        gf = [self.chain.est_delay(self.t), sum(x.Q for x in self.veh) / 1e8]
        return tasks, vf, rf, gf

    def build_obs(self, vid, task, mode):
        """Flat, fixed-size observation for the FC agents (padded to k_max subtasks)."""
        K = self.K
        tasks, vf, rf, gf = self._obs_parts(vid, task, mode)
        tf = np.zeros((K, 6), np.float32)
        for k, (b, d, dp, npred, nsucc) in enumerate(tasks):
            tf[k] = [b, d, dp / K, npred / K, nsucc / K, 1.0]
        rflat = [x for (l, n, tr, hop, f) in rf for x in (l, n, tr, hop / self.M, f)]
        return np.concatenate([tf.ravel(), np.array(vf + rflat + gf, np.float32)]).astype(np.float32)

    def build_graph_obs(self, vid, task, mode):
        """Graph observation for the GNN agent: any number of subtasks and RESs.
        xt: subtask nodes, A[i, j] = 1 if subtask j is a predecessor of i, xs: RES nodes."""
        tasks, vf, rf, gf = self._obs_parts(vid, task, mode)
        k = len(task.subs)
        xt = np.array([(b, d, dp / 8.0, npred / 4.0, nsucc / 4.0) for b, d, dp, npred, nsucc in tasks],
                      np.float32)
        A = np.zeros((k, k), np.float32)
        for s in task.subs:
            for p in s.preds:
                A[s.k, p.k] = 1.0
        xs = np.array([(l, n, tr, hop / 4.0, f) for l, n, tr, hop, f in rf], np.float32)
        vf = list(vf); vf[-1] = vf[-1] * self.M / 4.0     # nearest-RSU index, independent of M
        return dict(xt=xt, A=A, xs=xs, xv=np.array(vf, np.float32), xg=np.array(gf, np.float32))

    # ---- actions ----
    def submit(self, task, x, y, f):
        c, v = self.cfg, self.veh[task.vid]
        f = float(np.clip(f, c["f_local_min"], c["f_local_max"]))
        near, _ = self.nearest(v)
        rate = self.rate(v)
        task.info.update(Q=v.Q, E=v.E, trust=[r.trust() for r in self.res], x=list(x), y=list(y))
        for k, s in enumerate(task.subs):
            s.x, s.f = int(x[k]), f
            if s.x:
                j = int(y[k]); s.res = j
                st = max(self.t, v.tx_free)
                dur = c["overhead"] * s.d / rate
                v.tx_free = st + dur
                v.tx_bits.append((st, st + dur, s.d))
                e = c["p_tx"] * dur
                v.slot_energy += e; task.energy += e
                hop = abs(near - j) * (c["backhaul_hop_s"] + s.d / c["backhaul_bps"])
                arrive = st + dur + hop
                s.avail_at = arrive + self.chain.confirm(arrive)     # on-chain registration
                self.res[j].incoming.append(s)
            else:
                v.local.append(s)
        self.tasks.append(task)

    def res_finished(self, r, s, t):
        blk = self.chain.confirm(t)                   # proof-of-computation
        if r.mal and self.rng.random() < self.cfg["p_tamper"]:
            r.fail += 1                               # verification fails -> dispute, then recompute locally
            v = self.veh[s.task.vid]
            detect = t + blk + self.cfg["dispute_blocks"] * self.cfg["blk_interval"]
            s.x, s.rem, s.avail_at, s.f = 0, s.b, detect, self.cfg["f_local_max"]
            v.local.appendleft(s)
        else:
            r.ok += 1
            self.sub_done(s, t + blk)

    def sub_done(self, s, t):
        s.done_at = t
        task = s.task
        task.n_done += 1
        if task.n_done == len(task.subs):
            task.complete = max(x.done_at for x in task.subs)
            self.completed.append(task)

    # ---- time ----
    def begin_slot(self):
        c = self.cfg
        for v in self.veh:
            v.x = (v.x + v.v * c["slot"]) % c["road_len"]
        self.shadow = 0.7 * self.shadow + 0.3 * self.rng.normal(0, c["shadow_db"], self.N)

    def run_slot(self):
        c = self.cfg
        n = int(round(c["slot"] / c["dt"]))
        for i in range(n):
            t = self.t + i * c["dt"]
            for r in self.res:
                r.step(t, c["dt"], self)
            for v in self.veh:
                v.step(t, c["dt"], self)
        self.t += c["slot"]
        for v in self.veh:
            v.tx_bits = [z for z in v.tx_bits if z[1] > self.t]
            v.Q = v.queued_bits(self.t)
            v.E = max(v.E + c["energy_v"] * v.slot_energy - c["energy_v"] * c["energy_replenish"], 0.0)
            v.total_energy += v.slot_energy; v.slot_energy = 0.0
        done, self.completed = self.completed, []
        return done

    def all_done(self):
        return all(t.complete is not None for t in self.tasks)


# %% [markdown]
# ## 3. Rewards (Lyapunov drift-plus-penalty and baseline objectives)

# %%
def task_delay(task):
    return max(task.complete - task.arr, 1e-3)


def lyapunov_reward(task, cfg, trust_aware):
    """Per-slot problem P2: V * throughput - Q_i * (bits kept local) - E_i * energy."""
    T = task_delay(task)
    local_bits = sum(s.d for s, x in zip(task.subs, task.info["x"]) if not x) / 1e6
    r = cfg["V"] * (task.size / 1e6) / T
    r -= cfg["alpha_q"] * (task.info["Q"] / 1e7) * local_bits
    r -= cfg["alpha_e"] * (task.info["E"] / 10.0) * task.energy
    if trust_aware:   # penalise sending work to RESs the chain rates as unreliable
        tr = task.info["trust"]
        r -= cfg["beta_trust"] * sum((1 - tr[y]) for x, y in zip(task.info["x"], task.info["y"]) if x)
    return float(np.clip(r, -20, 20))


def cost_reward(task, w_t=1.0, w_e=0.1):
    return float(np.clip(-(w_t * task_delay(task) + w_e * task.energy), -40, 0))


# %% [markdown]
# ## 4. Agents

# %%
def mlp(i, o, h, n=2):
    layers, d = [], i
    for _ in range(n):
        layers += [nn.Linear(d, h), nn.ReLU()]; d = h
    layers.append(nn.Linear(d, o))
    return nn.Sequential(*layers)


def soft_update(tgt, src, tau):
    with torch.no_grad():
        for a, b in zip(tgt.parameters(), src.parameters()):
            a.mul_(1 - tau).add_(tau * b)


class Replay:
    def __init__(self, cap, od, ad):
        self.o = np.zeros((cap, od), np.float32); self.a = np.zeros((cap, ad), np.float32)
        self.r = np.zeros(cap, np.float32); self.n = self.p = 0; self.cap = cap

    def add(self, o, a, r):
        self.o[self.p], self.a[self.p], self.r[self.p] = o, a, r
        self.p = (self.p + 1) % self.cap; self.n = min(self.n + 1, self.cap)

    def sample(self, b):
        i = np.random.randint(0, self.n, b)
        f = lambda z: torch.as_tensor(z[i], device=DEVICE)
        return f(self.o), f(self.a), f(self.r)


class BaseAgent:
    name, obs_mode = "base", "reported"

    def __init__(self, env, cfg):
        self.K, self.M, self.cfg = env.K, env.M, cfg
        self.od = env.obs_dim()
        self.eps, self.steps = cfg["eps_start"], 0

    def set_progress(self, frac):
        c = self.cfg
        self.eps = c["eps_end"] + (c["eps_start"] - c["eps_end"]) * max(0.0, 1 - frac / 0.7)

    def f_of(self, u):
        c = self.cfg
        return c["f_local_min"] + float(np.clip(u, 0, 1)) * (c["f_local_max"] - c["f_local_min"])

    def reward(self, task):
        raise NotImplementedError

    def train(self):
        pass


class DAGLocal(BaseAgent):
    """All subtasks executed on the vehicle at full CPU frequency (no learning)."""
    name = "DAG-Local"

    def act(self, obs, task, explore):
        k = len(task.subs)
        return [0] * k, [0] * k, self.cfg["f_local_max"], None

    def store(self, task, r):
        pass

    def reward(self, task):
        return 0.0


class TaBRL(BaseAgent):
    """Dual-FC Actor (offloading {x,y} + computation {f}) with target Actors,
    noise-preserving quantisation (NOP) candidates, Critic that scores candidates on
    blockchain-audited state, Lyapunov + trust reward, replay buffer and soft updates."""
    name, obs_mode = "TaBRL", "audited"

    def __init__(self, env, cfg):
        super().__init__(env, cfg)
        K, M, h = self.K, self.M, cfg["hidden"]
        self.ad = K + K * M + 1
        self.actor_off = mlp(self.od, K + K * M, h).to(DEVICE)
        self.actor_f = mlp(self.od, 1, h // 2).to(DEVICE)
        self.t_off, self.t_f = copy.deepcopy(self.actor_off), copy.deepcopy(self.actor_f)
        self.critic = mlp(self.od + self.ad, 1, h, 3).to(DEVICE)
        self.opt_a = torch.optim.Adam(list(self.actor_off.parameters()) + list(self.actor_f.parameters()),
                                      lr=cfg["lr_actor"])
        self.opt_c = torch.optim.Adam(self.critic.parameters(), lr=cfg["lr_critic"])
        self.buf = Replay(cfg["buffer"], self.od, self.ad)
        self.rng = np.random.default_rng(cfg["seed"] + 1)

    def encode(self, x, yprob, u):
        # x: (B,K) offload prob, yprob: (B,K,M) RES distribution, u: (B,1) freq in [0,1]
        return torch.cat([x, (x.unsqueeze(-1) * yprob).flatten(1), u], 1)

    def _forward(self, off_net, f_net, o):
        z = off_net(o)
        p = torch.sigmoid(z[:, : self.K])
        logit = z[:, self.K:].view(-1, self.K, self.M)
        u = torch.sigmoid(f_net(o))
        return p, logit, u

    def res_view(self, obs):
        """Blockchain-audited RES state from the observation: load (s), trust, CPU (Hz)."""
        r = obs[6 * self.K + 6: 6 * self.K + 6 + 5 * self.M].reshape(self.M, 5)
        return r[:, 0] * 10.0, np.clip(r[:, 2], 1e-3, 1.0), r[:, 4] * 5e9

    def candidates(self, p, logit, u, tp, tlogit, tu, k, load, trust, freq, bvec):
        rng, cands, M = self.rng, [], logit.shape[-1]
        # trust-weighted policy: pi'(j) ~ pi(j) * trust_j^lambda (on-chain verification record)
        bias = self.cfg["trust_lambda"] * np.log(trust)
        def add(pp, ll, uu, b=bias):
            x = (pp[:k] > 0.5).astype(np.float32)
            y = (ll[:k] + b).argmax(-1)
            cands.append((x, y, float(np.clip(uu, 0, 1))))
        add(p, logit, u); add(tp, tlogit, tu)
        add(p, logit, u, 0.0)                              # raw Actor choice, no trust bias
        # trust-filtered candidates: work on RESs below the trust threshold is moved to the
        # best trusted RES (trust * speed / backlog), or kept on the vehicle
        x0 = (p[:k] > 0.5).astype(np.float32); y0 = (logit[:k] + bias).argmax(-1)
        bad = trust[y0] < self.cfg["trust_min"]
        if bad.any():
            # spread the moved subtasks over trusted RESs in order of trust * speed / backlog
            ok = [j for j in np.argsort(-(trust * freq / (1.0 + load))) if trust[j] >= self.cfg["trust_min"]]
            ok = ok or [int(np.argmax(trust))]
            y1 = y0.copy(); y1[bad] = [ok[i % len(ok)] for i in range(int(bad.sum()))]
            cands.append((x0, y1, float(np.clip(u, 0, 1))))
            x2 = x0.copy(); x2[bad] = 0
            cands.append((x2, y0, float(np.clip(u, 0, 1))))
        order = np.argsort(np.abs(p[:k] - 0.5))           # order-preserving flips
        for m in range(1, min(k, 3) + 1):
            pp = p.copy(); pp[order[:m]] = 1 - pp[order[:m]]; add(pp, logit, u)
        # structured quantisation: consolidated placements on trusted RESs
        # (whole DAG to one RES, heavy subtasks to one RES, all local)
        heavy = (bvec[:k] >= np.median(bvec[:k])).astype(np.float32)
        uu = float(np.clip(u, 0, 1))
        for j in range(M):
            if trust[j] >= self.cfg["trust_min"]:
                cands.append((np.ones(k, np.float32), np.full(k, j), uu))
                cands.append((heavy, np.full(k, j), uu))
        cands.append((np.zeros(k, np.float32), np.zeros(k, int), 1.0))
        i = 0
        while len(cands) < self.cfg["n_candidates"]:      # Gaussian + uniform perturbation
            pp = np.clip(p + rng.normal(0, 0.25, p.shape), 0, 1)
            ll = logit + rng.normal(0, 1.0, logit.shape)
            if rng.random() < 0.3:
                ll[:, rng.integers(M)] += rng.uniform(0, 3)
            add(pp, ll, u + rng.uniform(-0.25, 0.25), bias if i % 2 == 0 else 0.0)
            i += 1
        return cands

    def enc_np(self, x, y, u):
        K, M = self.K, self.M
        a = np.zeros(self.ad, np.float32)
        a[: len(x)] = x
        oh = np.zeros((K, M), np.float32)
        oh[np.arange(len(y)), y] = 1
        oh[: len(x)] *= x[:, None]; oh[len(x):] = 0
        a[K: K + K * M] = oh.ravel(); a[-1] = u
        return a

    @torch.no_grad()
    def act(self, obs, task, explore):
        k = len(task.subs)
        o = torch.as_tensor(obs, device=DEVICE).unsqueeze(0)
        p, l, u = [z[0].cpu().numpy() for z in self._forward(self.actor_off, self.actor_f, o)]
        tp, tl, tu = [z[0].cpu().numpy() for z in self._forward(self.t_off, self.t_f, o)]
        load, trust, freq = self.res_view(obs)
        cands = self.candidates(p, l, float(u[0]), tp, tl, float(tu[0]), k, load, trust, freq,
                                obs[0: 6 * self.K: 6])
        if explore and self.rng.random() < self.eps:
            x = self.rng.integers(0, 2, k).astype(np.float32)
            y = self.rng.integers(0, self.M, k)
            best = (x, y, float(self.rng.random()))
        else:
            enc = torch.as_tensor(np.stack([self.enc_np(*c) for c in cands]), device=DEVICE)
            q = self.critic(torch.cat([o.expand(len(cands), -1), enc], 1)).squeeze(1)
            best = cands[int(q.argmax())]
        x, y, uu = best
        return x.astype(int).tolist(), y.tolist(), self.f_of(uu), self.enc_np(x, y, uu)

    def reward(self, task):
        return lyapunov_reward(task, self.cfg, trust_aware=True)

    def store(self, task, r):
        self.buf.add(task.obs, task.act, r)

    def train(self):
        if self.buf.n < self.cfg["batch"]:
            return
        K, M = self.K, self.M
        o, a, r = self.buf.sample(self.cfg["batch"])
        q = self.critic(torch.cat([o, a], 1)).squeeze(1)
        lc = F.mse_loss(q, r)                                   # TD loss (per-slot, gamma = 0)
        self.opt_c.zero_grad(); lc.backward(); self.opt_c.step()

        p, logit, u = self._forward(self.actor_off, self.actor_f, o)
        valid = o[:, 5: 6 * K: 6]                               # subtask-valid flags
        relaxed = self.encode(p * valid, torch.softmax(logit, -1), u)
        q_pi = self.critic(torch.cat([o, relaxed], 1)).squeeze(1)
        with torch.no_grad():                                   # advantage-weighted imitation
            w = torch.exp((r - q_pi) / 1.0).clamp(max=10.0); w = w / w.mean()
        x_t = a[:, :K]
        y_t = a[:, K: K + K * M].view(-1, K, M)
        bce = (F.binary_cross_entropy(p.clamp(1e-5, 1 - 1e-5), x_t, reduction="none") * valid).sum(1)
        ce = (-(torch.log_softmax(logit, -1) * y_t).sum(-1)).sum(1)
        mse = (u.squeeze(1) - a[:, -1]) ** 2
        la = (w * (bce + ce + mse)).mean() - 0.1 * q_pi.mean()
        self.opt_a.zero_grad(); la.backward(); self.opt_a.step()
        soft_update(self.t_off, self.actor_off, self.cfg["tau"])
        soft_update(self.t_f, self.actor_f, self.cfg["tau"])


class DQN:
    def __init__(self, od, na, cfg, h):
        self.q = mlp(od, na, h).to(DEVICE)
        self.opt = torch.optim.Adam(self.q.parameters(), lr=cfg["lr_critic"])
        self.buf = Replay(cfg["buffer"], od, 1)
        self.na, self.bs = na, cfg["batch"]

    @torch.no_grad()
    def greedy(self, obs):
        return int(self.q(torch.as_tensor(obs, device=DEVICE).unsqueeze(0)).argmax())

    def train(self):
        if self.buf.n < self.bs:
            return
        o, a, r = self.buf.sample(self.bs)
        q = self.q(o).gather(1, a.long()).squeeze(1)
        loss = F.smooth_l1_loss(q, r)
        self.opt.zero_grad(); loss.backward(); self.opt.step()


def pattern_to_xy(pat, task, M):
    """0: all local | 1..M: all to RES j | M+1..2M: heavy subtasks (>= median cycles) to RES j."""
    k = len(task.subs)
    if pat == 0:
        return [0] * k, [0] * k
    j = (pat - 1) % M
    if pat <= M:
        return [1] * k, [j] * k
    med = np.median([s.b for s in task.subs])
    return [int(s.b >= med) for s in task.subs], [j] * k


class _MALHTOActor(nn.Module):
    """FC actor with two hidden layers and two output heads (offloading target, CPU frequency)."""
    def __init__(self, n_in, n_opt, h):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(n_in, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU())
        self.head_x = nn.Linear(h, n_opt)
        self.head_f = nn.Linear(h, 1)

    def forward(self, s):
        z = self.body(s)
        return self.head_x(z), torch.sigmoid(self.head_f(z))


class _MALHTODevice:
    """One device agent: primary + target actor, optimiser and its own replay buffer."""
    def __init__(self, n_in, n_opt, cfg):
        self.net = _MALHTOActor(n_in, n_opt, cfg["malhto_hidden"]).to(DEVICE)
        self.target = copy.deepcopy(self.net)
        self.opt = torch.optim.Adam(self.net.parameters(), lr=cfg["lr_critic"])
        cap = cfg["buffer"]
        self.s = np.zeros((cap, n_in), np.float32); self.a = np.zeros(cap, np.int64)
        self.u = np.zeros(cap, np.float32); self.n = self.p = 0; self.cap = cap; self.steps = 0

    def add(self, s, a, u):
        self.s[self.p], self.a[self.p], self.u[self.p] = s, a, u
        self.p = (self.p + 1) % self.cap; self.n = min(self.n + 1, self.cap)


class MALHTO(BaseAgent):
    """MA-LHTO as described in X. He et al., "Real-Time Scheduling of CPU/GPU Heterogeneous Tasks
    in Dynamic IoT Systems", IEEE TMC 25(1), 2026, and its released code
    (github.com/HE-xiao4333/MA-LHTO), mapped onto this vehicular environment.

    * One agent per vehicle (device); FC actor with two hidden layers and two output heads:
      offloading target in {local, RES 1..M} and CPU frequency. The GPU-node head and the
      base-station resource-allocation agents are dropped: this system has no GPU types and
      RESs share their CPU through the MFQ, so there is no per-task allocation to learn.
    * Local state (eq. 22): the subtask, the other subtasks generated with it, the device
      queue, and the queue length / pending-task count of every RES (self-reported, no trust).
      Each subtask is decided on its own; the DAG edges are not observed.
    * Centralised training, decentralised execution: during training the top-K ranked actions
      (K = 3 / 2 / 1 for slots < 50 / < 200 / later, as in the authors' code) are scored by a
      centralised model-based critic with the team reward (eq. 28, value = 10*0.7*sigmoid(psi)
      - sigmoid(stable) as in the released code, GPU-fragmentation term = 0); the best one is
      executed and stored. Loss (eq. 36): cross-entropy to the stored action - Θ * entropy,
      plus Huber loss for the frequency head. Soft-updated target actor (ξ = 0.01) and a
      periodic reset of the last hidden layer (eq. 38).
    * Execution (evaluation): every vehicle takes its own actor's most probable action."""
    name, obs_mode = "MA-LHTO", "reported"

    def __init__(self, env, cfg):
        super().__init__(env, cfg)
        self.n_in = 7 + 2 * self.M
        self.agents = [_MALHTODevice(self.n_in, self.M + 1, cfg) for _ in range(env.N)]
        self._last = 0

    def _reported_load(self, r):
        load = r.backlog_cycles() / (r.F * len(r.cores))
        return load * self.cfg["report_factor"] if r.mal else load

    def _states(self, task):
        env, c = self.env, self.cfg
        v, subs = env.veh[task.vid], task.subs
        B, D = sum(s.b for s in subs), sum(s.d for s in subs)
        common = [v.Q / 1e7, sum(s.rem for s in v.local) / c["f_local_max"] / 10.0]
        for r in env.res:
            common += [self._reported_load(r) / 10.0, r.n_jobs() / 20.0]
        return np.array([[s.b / 1e9, s.d / 1e6, (len(subs) - 1) / 8.0, (B - s.b) / 1e9, (D - s.d) / 1e6]
                         + common for s in subs], np.float32)

    def _value(self, task, opt, f):
        """Centralised critic: estimated completion time of the job under `opt`
        (0 = local, j + 1 = RES j) and the RES backlog it leaves, from the reported state."""
        env, c = self.env, self.cfg
        v, now = env.veh[task.vid], env.t
        local_free = now + sum(s.rem / s.f for s in v.local)
        tx_free, rate = max(now, v.tx_free), env.rate(v)
        near, _ = env.nearest(v)
        blk = env.chain.est_delay(now)
        load = [self._reported_load(r) for r in env.res]
        free = [now + l for l in load]
        fin = []
        for s, o in zip(task.subs, opt):
            ready = max((fin[p.k] for p in s.preds), default=now)
            if o == 0:
                start = max(ready, local_free); local_free = start + s.b / f; fin.append(local_free)
            else:
                j = int(o) - 1; r = env.res[j]
                tx_free += c["overhead"] * s.d / rate
                arrive = tx_free + abs(near - j) * (c["backhaul_hop_s"] + s.d / c["backhaul_bps"]) + blk
                start = max(arrive, ready, free[j])
                free[j] += s.b / (r.F * len(r.cores)); load[j] += s.b / (r.F * len(r.cores))
                fin.append(start + s.b / r.F + blk)
        T = max(max(fin) - now, 1e-3)
        psi = (task.size / 1e6) / T
        stable = sum(load) / (10.0 * len(load))
        sig = lambda z: 1.0 / (1.0 + math.exp(-z))
        return 10.0 * 0.7 * sig(psi) - sig(stable)

    @torch.no_grad()
    def act(self, obs, task, explore):
        ag = self.agents[task.vid % len(self.agents)]
        st = self._states(task)
        logits, fu = ag.net(torch.as_tensor(st, device=DEVICE))
        order = torch.argsort(logits, dim=1, descending=True).cpu().numpy()
        u = float(fu.mean())
        if explore:                                     # centralised training: pick best of top-K
            slot = int(self.env.t)
            K = min(3 if slot < 50 else 2 if slot < 200 else 1, self.M + 1)
            cands = [order[:, i] for i in range(K)]
            vals = [self._value(task, cand, self.f_of(u)) for cand in cands]
            opt = cands[int(np.argmax(vals))]
            for k in range(len(task.subs)):
                ag.add(st[k], opt[k], u)
        else:                                           # decentralised execution
            opt = order[:, 0]
        x = [int(o > 0) for o in opt]
        y = [max(int(o) - 1, 0) for o in opt]
        return x, y, self.f_of(u), np.array([1.0], np.float32)

    def reward(self, task):                             # logged only; learning uses the critic above
        return lyapunov_reward(task, self.cfg, trust_aware=False)

    def store(self, task, r):
        self._last = task.vid % len(self.agents)

    def train(self):
        c, ag = self.cfg, self.agents[self._last]
        if ag.n < c["batch"]:
            return
        i = np.random.randint(0, ag.n, c["batch"])
        s = torch.as_tensor(ag.s[i], device=DEVICE)
        a = torch.as_tensor(ag.a[i], device=DEVICE)
        u = torch.as_tensor(ag.u[i], device=DEVICE)
        logits, f = ag.net(s)
        p = torch.softmax(logits, 1)
        entropy = -(p * torch.log(p + 1e-8)).sum(1).mean()
        loss = F.cross_entropy(logits, a) + F.smooth_l1_loss(f.squeeze(1), u) - c["malhto_entropy"] * entropy
        ag.opt.zero_grad(); loss.backward(); ag.opt.step()
        soft_update(ag.target, ag.net, c["malhto_xi"])
        ag.steps += 1
        if ag.steps % c["malhto_reset_every"] == 0:     # eq. (38): reset the last hidden layer
            with torch.no_grad():
                last = ag.net.body[2]
                last.weight.normal_(0.0, c["malhto_reset_std"]); last.bias.zero_()


class IOPO(BaseAgent):
    """Two-stage DRL: stage 1 DQN picks the offloading pattern, stage 2 sets the local
    CPU frequency in closed form to minimise w_t*delay + w_e*energy."""
    name = "IOPO"
    W_T, W_E = 1.0, 0.1

    def __init__(self, env, cfg):
        super().__init__(env, cfg)
        self.na = 1 + 2 * self.M
        self.dqn = DQN(self.od, self.na, cfg, cfg["hidden"])
        self.rng = np.random.default_rng(cfg["seed"] + 3)
        # argmin_f  W_T*b/f + W_E*kappa*b*f^2  ->  f* = (W_T / (2 W_E kappa))^(1/3)
        self.f_star = float(np.clip((self.W_T / (2 * self.W_E * cfg["kappa"])) ** (1 / 3),
                                    cfg["f_local_min"], cfg["f_local_max"]))

    def act(self, obs, task, explore):
        a = int(self.rng.integers(self.na)) if explore and self.rng.random() < self.eps else self.dqn.greedy(obs)
        x, y = pattern_to_xy(a, task, self.M)
        return x, y, self.f_star, np.array([a], np.float32)

    def reward(self, task):
        return cost_reward(task, self.W_T, self.W_E)

    def store(self, task, r):
        self.dqn.buf.add(task.obs, task.act, r)

    def train(self):
        self.dqn.train()


class DECO(BaseAgent):
    """TD3 offloading: continuous action [offload(K), RES(K), f] quantised to decisions;
    twin critics, target policy smoothing and delayed actor updates."""
    name = "DECO"

    def __init__(self, env, cfg):
        super().__init__(env, cfg)
        K, h = self.K, cfg["hidden"]
        self.ad = 2 * K + 1
        self.actor = mlp(self.od, self.ad, h).to(DEVICE)
        self.q1 = mlp(self.od + self.ad, 1, h).to(DEVICE)
        self.q2 = mlp(self.od + self.ad, 1, h).to(DEVICE)
        self.actor_t = copy.deepcopy(self.actor)
        self.opt_a = torch.optim.Adam(self.actor.parameters(), lr=cfg["lr_actor"])
        self.opt_c = torch.optim.Adam(list(self.q1.parameters()) + list(self.q2.parameters()), lr=cfg["lr_critic"])
        self.buf = Replay(cfg["buffer"], self.od, self.ad)
        self.rng = np.random.default_rng(cfg["seed"] + 4)
        self.it = 0

    @torch.no_grad()
    def act(self, obs, task, explore):
        a = torch.sigmoid(self.actor(torch.as_tensor(obs, device=DEVICE).unsqueeze(0)))[0].cpu().numpy()
        if explore:
            a = np.clip(a + self.rng.normal(0, 0.1 + 0.4 * self.eps, a.shape), 0, 1)
        k, K = len(task.subs), self.K
        x = (a[:k] > 0.5).astype(int).tolist()
        y = np.minimum((a[K: K + k] * self.M).astype(int), self.M - 1).tolist()
        return x, y, self.f_of(a[-1]), a.astype(np.float32)

    def reward(self, task):
        return cost_reward(task, 1.0, 0.1)

    def store(self, task, r):
        self.buf.add(task.obs, task.act, r)

    def train(self):
        if self.buf.n < self.cfg["batch"]:
            return
        o, a, r = self.buf.sample(self.cfg["batch"])
        oa = torch.cat([o, a], 1)
        lc = F.mse_loss(self.q1(oa).squeeze(1), r) + F.mse_loss(self.q2(oa).squeeze(1), r)
        self.opt_c.zero_grad(); lc.backward(); self.opt_c.step()
        self.it += 1
        if self.it % 2 == 0:                                   # delayed policy update
            pi = torch.sigmoid(self.actor(o))
            la = -self.q1(torch.cat([o, pi], 1)).mean()
            self.opt_a.zero_grad(); la.backward(); self.opt_a.step()
            soft_update(self.actor_t, self.actor, self.cfg["tau"])


ALGOS = {"TaBRL": TaBRL, "MA-LHTO": MALHTO, "IOPO": IOPO, "DECO": DECO, "DAG-Local": DAGLocal}



# %% [markdown]
# ## 5. Episode runner, training and evaluation

# %%
def run_episode(agent, sched, cfg, n_veh, seed, train, horizon=None, trace=False):
    env = IoVEnv(cfg, n_veh, seed)
    agent.env = env
    horizon = horizon or len(sched)
    backlog, energy, rewards = [], [], []
    stores = 0

    def handle(done):
        nonlocal stores
        for task in done:
            r = agent.reward(task); rewards.append(r)
            if train and task.act is not None:
                agent.store(task, r); stores += 1
                if stores % getattr(agent, "train_every", cfg["train_every"]) == 0:
                    agent.train()

    for s in range(horizon):
        env.begin_slot()
        for vid, job in sched[s]:
            task = Task(vid, env.t, job)
            build = env.build_graph_obs if getattr(agent, "graph_obs", False) else env.build_obs
            task.obs = build(vid, task, agent.obs_mode)
            x, y, f, task.act = agent.act(task.obs, task, explore=train)
            env.submit(task, x, y, f)
        handle(env.run_slot())
        if trace:
            backlog.append(sum(v.Q for v in env.veh))
            energy.append(sum(v.total_energy for v in env.veh))
    extra = 0
    while not env.all_done() and extra < cfg["drain_max"]:
        env.begin_slot(); handle(env.run_slot()); extra += 1

    fin = [t for t in env.tasks if t.complete is not None]
    delays = [task_delay(t) for t in fin]
    # unfinished tasks (only if drain_max hit) are counted with their age so they are not hidden
    delays += [env.t - t.arr for t in env.tasks if t.complete is None]
    makespan = max([t.complete for t in fin] + [env.t if len(fin) < len(env.tasks) else 0, 1.0])
    return dict(
        completion=float(np.mean(delays)) if delays else 0.0,
        # processing capacity = mean per-DAG processing rate (bits / completion time),
        # i.e. the throughput term size/T of problem P2
        throughput=float(np.mean([t.size / task_delay(t) for t in fin])) if fin else 0.0,
        sys_throughput=float(sum(t.size for t in fin) / makespan),
        reward=float(np.mean(rewards)) if rewards else 0.0,
        energy=float(sum(v.total_energy for v in env.veh)),
        offload=float(np.mean([np.mean(t.info["x"]) for t in env.tasks])) if env.tasks else 0.0,
        tampered=int(sum(r.fail for r in env.res)),
        n_tasks=len(env.tasks), backlog=backlog, energy_t=energy)


def train_agent(name, train_jobs, cfg, n_veh):
    set_seed(cfg["seed"])
    probe = IoVEnv(cfg, n_veh, 0)
    agent = ALGOS[name](probe, cfg)
    hist = []
    if name == "DAG-Local":
        return agent, hist
    rng = np.random.default_rng(cfg["seed"] + 100)
    E = cfg["train_episodes"]
    if not E:
        raise ValueError("Number of training epochs not set: run with --epochs N "
                         "(e.g. python tabrl_iov.py --epochs 150)")
    t0 = time.time()
    for ep in range(E):
        agent.set_progress(ep / E)
        sched = make_window(train_jobs, n_veh, cfg["horizon"], cfg, rng)
        m = run_episode(agent, sched, cfg, n_veh, seed=10_000 + ep, train=True)
        hist.append(dict(ep=ep, completion=m["completion"], throughput=m["throughput"], reward=m["reward"]))
        if ep % 10 == 0 or ep == E - 1:
            print(f"  [{name:9s}] epoch {ep + 1:3d}/{E}  T_c={m['completion']:.3f}s  "
                  f"thr={m['throughput'] / 1e6:.2f} Mbit/s  r={m['reward']:.3f}  "
                  f"off={m['offload']:.2f}  eps={agent.eps:.2f}  ({time.time() - t0:.0f}s)")
    return agent, hist


def evaluate(agent, windows, cfg, n_veh, horizon=None, trace=False):
    ms = [run_episode(agent, sched, cfg, n_veh, seed=seed, train=False, horizon=horizon, trace=trace)
          for seed, sched in windows]
    out = {k: float(np.mean([m[k] for m in ms])) for k in
           ("completion", "throughput", "sys_throughput", "reward", "energy", "offload", "tampered")}
    out["completion_std"] = float(np.std([m["completion"] for m in ms]))
    out["throughput_std"] = float(np.std([m["throughput"] for m in ms]))
    if trace:
        out["backlog"] = np.mean([m["backlog"] for m in ms], 0).tolist()
        out["energy_t"] = np.mean([m["energy_t"] for m in ms], 0).tolist()
    return out


def fixed_windows(jobs, n_veh, horizon, cfg, n, seed):
    rng = np.random.default_rng(seed)
    return [(seed * 100 + i, make_window(jobs, n_veh, horizon, cfg, rng)) for i in range(n)]


# %% [markdown]
# ## 6. Figures

# %%
ORDER = ["TaBRL", "MA-LHTO", "IOPO", "DECO", "DAG-Local"]
C_TIME, C_CAP = "#B08FAE", "#7FB8D8"      # mauve / light blue as in the paper
C_LINES = {"TaBRL": "#1a1a1a", "MA-LHTO": "#E03C31", "IOPO": "#9B59B6", "DECO": "#4FB6E0"}
# Axis ranges as multiples of each panel's largest value: (completion-time axis, capacity axis).
# They only set how tall the bars look, never the numbers. The ratios are read off the paper's
# Fig. 3: (a) training, time axis 0-20 for a max of 18.19, capacity axis 0-5000 for a max of
# 2693; (b) test, time axis ~0-17.5 for a max of 16.67, capacity axis ~0-3500 for a max of 3128.
AXIS_HEADROOM = {"train": (1.10, 1.86), "test": (1.05, 1.12)}


def _bar_panel(ax, data, title=None, xlabel=None, headroom=(1.10, 1.86)):
    """Grouped bars on twin axes: completion time (left) and processing capacity (right)."""
    algs = [a for a in ORDER if a in data]
    ct = np.array([data[a]["completion"] for a in algs])
    cap = np.array([data[a]["throughput"] / 1e3 for a in algs])        # kbit/s
    x, w = np.arange(len(algs)), 0.4
    b1 = ax.bar(x - w / 2, ct, w, color=C_TIME, label="Completion time")
    ax2 = ax.twinx()
    b2 = ax2.bar(x + w / 2, cap, w, color=C_CAP, label="Processing capacity")
    for b, v in zip(b1, ct):
        ax.text(b.get_x() + w / 2, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    for b, v in zip(b2, cap):
        ax2.text(b.get_x() + w / 2, v, f"{v:.0f}", ha="center", va="bottom", fontsize=8)
    ax.set_ylim(0, ct.max() * headroom[0]); ax2.set_ylim(0, cap.max() * headroom[1])
    ax.set_xticks(x); ax.set_xticklabels(algs, rotation=15)
    ax.set_ylabel("Mean completion time (s)"); ax2.set_ylabel("Processing capacity (kbit/s)")
    if title:
        ax.set_title(title)
    if xlabel:
        ax.set_xlabel(xlabel)
    return b1, b2


def plot_fig3(res, path, title="Impact of different algorithms on train and test set"):
    """Side-by-side layout: Training set | Test set."""
    plt.rcParams.update(plt.rcParamsDefault)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    for ax, split, name in zip(axes, ("train", "test"), ("Training set", "Test set")):
        b1, b2 = _bar_panel(ax, res[split], title=name, headroom=AXIS_HEADROOM[split])
    fig.legend([b1, b2], ["Completion time", "Processing capacity"], loc="lower center",
               ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(title, fontsize=14)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)


def plot_fig3_paper_style(res, path):
    """Stacked layout as printed in the paper: (a) training set over (b) test set."""
    plt.rcParams.update({"font.family": "serif", "font.size": 11})
    fig, axes = plt.subplots(2, 1, figsize=(7.2, 8.4))
    for ax, split, tag in zip(axes, ("train", "test"), ("(a)", "(b)")):
        b1, b2 = _bar_panel(ax, res[split], xlabel=f"Algorithms\n{tag}", headroom=AXIS_HEADROOM[split])
        ax.legend([b1, b2], ["Completion time", "Processing capacity"], loc="upper center",
                  ncol=2, fontsize=8, frameon=False)
    fig.tight_layout()
    fig.savefig(path, dpi=300); plt.close(fig)
    plt.rcParams.update(plt.rcParamsDefault)


def replot_fig3(out_dir, train=None, test=None):
    """Redraw Fig. 3 of a finished run with other y-axis ranges, without retraining.

    Display only: every plotted value is read unchanged from <out_dir>/results.json, and
    comparisons between algorithms on the same axis are not affected. train / test are
    (completion-time axis, capacity axis) ranges as multiples of the panel's largest bar,
    e.g. train=(1.10, 2.2). Writes fig3_completion_capacity_custom.png and
    fig3_paper_style_custom.png next to the original figures."""
    res = json.load(open(os.path.join(out_dir, "results.json")))["fig3"]
    saved, saved_order = dict(AXIS_HEADROOM), list(ORDER)
    if train:
        AXIS_HEADROOM["train"] = tuple(train)
    if test:
        AXIS_HEADROOM["test"] = tuple(test)
    ORDER.extend(a for a in res["train"] if a not in ORDER)     # e.g. TaBRL-GNN
    try:
        plot_fig3(res, os.path.join(out_dir, "fig3_completion_capacity_custom.png"))
        plot_fig3_paper_style(res, os.path.join(out_dir, "fig3_paper_style_custom.png"))
        for split in ("train", "test"):
            d, (ht, hc) = res[split], AXIS_HEADROOM[split]
            ct_max = max(m["completion"] for m in d.values())
            cap_max = max(m["throughput"] for m in d.values())
            t, c = d["TaBRL"]["completion"], d["TaBRL"]["throughput"]
            brown, blue = t / (ht * ct_max), c / (hc * cap_max)
            equal_at = ht * (c / cap_max) * (ct_max / t)
            print(f"{split:5s}: axes (time {ht:.2f}, capacity {hc:.2f})  TaBRL bar heights: "
                  f"completion {brown:.0%}, capacity {blue:.0%}  -> "
                  f"{'brown taller' if brown > blue else 'blue taller'}; "
                  f"the two are equal at capacity axis {equal_at:.2f}")
    finally:
        AXIS_HEADROOM.clear(); AXIS_HEADROOM.update(saved)
        ORDER[:] = saved_order


def plot_security(res, path):
    """Tampered results returned by compromised RESs (caught by on-chain verification)."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, split, name in zip(axes, ("train", "test"), ("Training set", "Test set")):
        algs = [a for a in ORDER if a in res[split] and a != "DAG-Local"]
        v = [res[split][a]["tampered"] for a in algs]
        bars = ax.bar(algs, v, 0.55, color=[C_TIME if a == "TaBRL" else C_CAP for a in algs])
        for b, x in zip(bars, v):
            ax.text(b.get_x() + b.get_width() / 2, x, f"{x:.0f}", ha="center", va="bottom", fontsize=9)
        ax.set_title(name)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axes[0].set_ylabel("Tampered subtask results per episode")
    fig.suptitle("Exposure to compromised edge servers", fontsize=13)
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_fig4(res4, path):
    fig, axes = plt.subplots(2, 1, figsize=(7.2, 8.0))
    for ax, (nv, data), tag in zip(axes, res4.items(), ("(a)", "(b)")):
        ax2 = ax.twinx()
        for a in [a for a in ORDER if a != "DAG-Local"]:
            if a not in data:
                continue
            t = np.arange(1, len(data[a]["backlog"]) + 1)
            ax.plot(t, np.cumsum(data[a]["backlog"]), color=C_LINES.get(a), lw=1.4, label=a)
            ax2.plot(t, data[a]["energy_t"], color=C_LINES.get(a), lw=1.0, ls="--", alpha=0.8)
        ax.set_xlabel(f"Time (slot)\n{tag} Vehicles = {nv}")
        ax.set_ylabel("Accumulated awaiting task size (bit·slot)")
        ax2.set_ylabel("Energy consumption (J, dashed)")
        ax.legend(loc="upper left", fontsize=8, frameon=False)
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_training(hists, path):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for a, h in hists.items():
        if not h:
            continue
        ep = [z["ep"] for z in h]
        sm = lambda v: pd.Series(v).rolling(8, min_periods=1).mean()
        axes[0].plot(ep, sm([z["completion"] for z in h]), label=a, color=C_LINES.get(a))
        axes[1].plot(ep, sm([z["throughput"] / 1e3 for z in h]), label=a, color=C_LINES.get(a))
    axes[0].set_ylabel("Completion time (s)"); axes[1].set_ylabel("Processing capacity (kbit/s)")
    for ax in axes:
        ax.set_xlabel("Training epoch"); ax.legend(frameon=False, fontsize=8)
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


# %% [markdown]
# ## 7. Run everything

# %%
SETTING_KEYS = ["train_episodes", "seed", "n_veh", "n_res", "horizon", "eval_windows",
                "arrival_rate", "n_malicious", "data_path", "out_dir"]


def print_settings(cfg):
    names = {"train_episodes": "epochs"}
    print("Settings: " + ", ".join(f"{names.get(k, k)}={cfg[k]}" for k in SETTING_KEYS if k in cfg))


def results_table(res):
    """All Fig. 3 values as a table. Two throughput measures are reported:
    processing capacity = mean over DAGs of size / completion time (the bars in Fig. 3),
    system throughput   = all input bits processed / total time until the last DAG finished."""
    return pd.DataFrame([{
        "Split": s, "Algorithm": a,
        "Completion time (s)": round(m["completion"], 3),
        "Processing capacity (kbit/s)": round(m["throughput"] / 1e3, 1),
        "System throughput (Mbit/s)": round(m["sys_throughput"] / 1e6, 3),
        "Energy (J)": round(m["energy"], 1),
        "Offload ratio": round(m["offload"], 3),
        "Tampered results": round(m["tampered"], 1)}
        for s in res for a, m in res[s].items()])


def fig4_table(res4):
    """Fig. 4 values at the end of the horizon: backlog and cumulative energy."""
    return pd.DataFrame([{
        "Vehicles": nv, "Algorithm": a,
        "Accumulated awaiting task size (bit*slot)": float(np.sum(m["backlog"])),
        "Mean awaiting task size (bit)": round(float(np.mean(m["backlog"])), 1),
        "Energy consumption (J)": round(m["energy_t"][-1], 1)}
        for nv, d in res4.items() for a, m in d.items()])


def print_table(title, df):
    print(f"\n{'=' * 20} {title} {'=' * 20}")
    print(df.to_string(index=False))


def apply_cli_args(cfg):
    """Command-line overrides for the script version, e.g.
    python tabrl_iov.py --epochs 100 --seed 7 --data /path/batch_task.csv"""
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", "--episodes", dest="epochs", type=int, required=True,
                    help="training epochs per algorithm (1 epoch = one simulation run of `horizon` slots)")
    ap.add_argument("--seed", type=int)
    ap.add_argument("--vehicles", type=int)
    ap.add_argument("--horizon", type=int, help="time slots per episode")
    ap.add_argument("--eval-windows", type=int)
    ap.add_argument("--data", help="path to batch_task.csv")
    ap.add_argument("--out", help="output directory")
    args, _ = ap.parse_known_args()
    for flag, key in [("epochs", "train_episodes"), ("seed", "seed"), ("vehicles", "n_veh"),
                      ("horizon", "horizon"), ("eval_windows", "eval_windows"), ("data", "data_path"),
                      ("out", "out_dir")]:
        if getattr(args, flag) is not None:
            cfg[key] = getattr(args, flag)
    return cfg


class _Tee:
    """Send every printed line to the notebook/terminal and to a log file, flushing each
    write so no line is held back or lost if the notebook display drops output."""
    def __init__(self, path, stream):
        self.f, self.stream = open(path, "a", encoding="utf-8"), stream

    def write(self, text):
        self.stream.write(text); self.stream.flush()
        self.f.write(text); self.f.flush()

    def flush(self):
        self.stream.flush(); self.f.flush()


def logged(fn, log_name):
    """Run fn(cfg, ...) with all output also saved to <out_dir>/<log_name>."""
    import sys

    def wrapper(cfg=CFG, *args, **kwargs):
        os.makedirs(cfg["out_dir"], exist_ok=True)
        old, tee = sys.stdout, _Tee(os.path.join(cfg["out_dir"], log_name), sys.stdout)
        sys.stdout = tee
        try:
            return fn(cfg, *args, **kwargs)
        finally:
            sys.stdout = old; tee.f.close()
    wrapper.__name__, wrapper.__doc__ = fn.__name__, fn.__doc__
    return wrapper


def main(cfg=CFG, return_agents=None):
    os.makedirs(cfg["out_dir"], exist_ok=True)
    set_seed(cfg["seed"])
    print(f"Device: {DEVICE}")
    print_settings(cfg)
    train_jobs, test_jobs = load_dag_jobs(cfg)
    N = cfg["n_veh"]
    win_train = fixed_windows(train_jobs, N, cfg["horizon"], cfg, cfg["eval_windows"], seed=1)
    win_test = fixed_windows(test_jobs, N, cfg["horizon"], cfg, cfg["eval_windows"], seed=2)

    agents, hists, res = {}, {}, {"train": {}, "test": {}}
    for name in ORDER:
        print(f"\n=== {name} ===")
        agents[name], hists[name] = train_agent(name, train_jobs, cfg, N)
        for split, win in (("train", win_train), ("test", win_test)):
            res[split][name] = evaluate(agents[name], win, cfg, N)
            m = res[split][name]
            print(f"  eval {split:5s}: completion {m['completion']:.3f}s  "
                  f"capacity {m['throughput'] / 1e3:.0f} kbit/s  energy {m['energy']:.0f} J  "
                  f"offload {m['offload']:.2f}  tampered {m['tampered']:.0f}")

    if return_agents is not None:
        return_agents.update(agents)
    res4 = {}
    for nv in cfg["fig4_veh"]:
        wins = fixed_windows(test_jobs, nv, cfg["fig4_horizon"], cfg, cfg["eval_windows"], seed=3 + nv)
        res4[nv] = {a: evaluate(agents[a], wins, cfg, nv, trace=True)
                    for a in ORDER if a != "DAG-Local"}

    plot_fig3(res, os.path.join(cfg["out_dir"], "fig3_completion_capacity.png"))
    plot_fig3_paper_style(res, os.path.join(cfg["out_dir"], "fig3_paper_style.png"))
    plot_fig4(res4, os.path.join(cfg["out_dir"], "fig4_backlog_energy.png"))
    plot_security(res, os.path.join(cfg["out_dir"], "security_tampered.png"))
    plot_training(hists, os.path.join(cfg["out_dir"], "training_curves.png"))

    table, t4 = results_table(res), fig4_table(res4)
    table.to_csv(os.path.join(cfg["out_dir"], "results.csv"), index=False)
    t4.to_csv(os.path.join(cfg["out_dir"], "fig4_values.csv"), index=False)
    with open(os.path.join(cfg["out_dir"], "results.json"), "w") as fh:
        json.dump(dict(fig3=res, fig4={str(k): v for k, v in res4.items()}, history=hists, cfg=cfg),
                  fh, default=float)
    print_table("Fig. 3 values (train and test)", table)
    print("Processing capacity = mean of size/completion time per DAG (Fig. 3 bars); "
          "system throughput = all bits processed / total time.")
    print_table(f"Fig. 4 values after {cfg['fig4_horizon']} slots", t4)
    print(f"\nFigures and tables written to {cfg['out_dir']}/")
    return res, res4, hists


main = logged(main, "train_log.txt")


if __name__ == "__main__":
    main(apply_cli_args(CFG))


In [ ]:
%%writefile tabrl_gnn.py
# %% [markdown]
# # Thesis extension: TaBRL-GNN (graph neural network actor-critic)
#
# TaBRL's dual-FC Actor sees each DAG as a flat, zero-padded vector of at most `k_max`
# subtasks, and its output layer is sized for exactly `n_res` edge servers. It cannot see
# the dependency edges, and it cannot run on a larger DAG or a different number of servers
# without retraining.
#
# **TaBRL-GNN** swaps the dual-FC Actor and the FC Critic for graph networks and keeps
# everything else identical: NOP candidates, trust coupling, Lyapunov reward, replay buffer,
# soft-updated target actors and the environment. Any difference in the results therefore
# comes from the network architecture.
#
# * Subtask nodes pass messages along the DAG edges (from predecessors and successors).
# * RES nodes carry the audited load, trust, speed and hop distance; they exchange messages
#   with the subtask nodes.
# * Server choice = a score for every (subtask, RES) pair, so any number of RESs works.
# * Dual Actor as in the paper: one GNN for offloading + RES selection, one for CPU frequency.
# * The Critic is a GNN whose subtask nodes also receive the candidate action.
#
# Experiments
# * **E1:** the base scenario (Fig. 3 style), TaBRL-GNN next to TaBRL and the baselines.
# * **E2:** larger DAGs (up to 20 subtasks).
# * **E3:** zero-shot scalability. Policies trained with N = 10 vehicles and M = 4 RESs are
#   tested with more vehicles, and TaBRL-GNN also with a different number of RESs, which
#   the fixed-size FC networks cannot handle.

# %%
from tabrl_iov import *  # notebook: inlined
from tabrl_iov import _bar_panel  # notebook: inlined

FT, FS, FV, FG = 5, 5, 6, 2          # feature sizes: subtask, RES, vehicle, global


def batch_graphs(obs_list):
    """Pad a list of graph observations into dense tensors with node masks."""
    B = len(obs_list)
    K = max(o["xt"].shape[0] for o in obs_list)
    M = max(o["xs"].shape[0] for o in obs_list)
    xt = np.zeros((B, K, FT), np.float32); A = np.zeros((B, K, K), np.float32)
    mt = np.zeros((B, K), np.float32)
    xs = np.zeros((B, M, FS), np.float32); ms = np.zeros((B, M), np.float32)
    xv = np.zeros((B, FV), np.float32); xg = np.zeros((B, FG), np.float32)
    for i, o in enumerate(obs_list):
        k, m = o["xt"].shape[0], o["xs"].shape[0]
        xt[i, :k] = o["xt"]; A[i, :k, :k] = o["A"]; mt[i, :k] = 1
        xs[i, :m] = o["xs"]; ms[i, :m] = 1
        xv[i] = o["xv"]; xg[i] = o["xg"]
    t = lambda z: torch.as_tensor(z, device=DEVICE)
    return dict(xt=t(xt), A=t(A), mt=t(mt), xs=t(xs), ms=t(ms), xv=t(xv), xg=t(xg))


def repeat_graph(g, n):
    return {k: v.expand(n, *v.shape[1:]) for k, v in g.items()}


def masked_mean(h, m):
    return (h * m.unsqueeze(-1)).sum(1) / m.sum(1, keepdim=True).clamp(min=1)


def masked_max(h, m):
    return h.masked_fill(m.unsqueeze(-1) == 0, -1e4).max(1).values


class GNNLayer(nn.Module):
    """One round of message passing: subtask <- predecessors, successors, RESs, context;
    RES <- subtasks."""
    def __init__(self, h):
        super().__init__()
        self.self_t, self.pred, self.succ = nn.Linear(h, h), nn.Linear(h, h), nn.Linear(h, h)
        self.srv, self.ctx = nn.Linear(h, h), nn.Linear(h, h)
        self.self_s, self.task = nn.Linear(h, h), nn.Linear(h, h)
        self.norm_t, self.norm_s = nn.LayerNorm(h), nn.LayerNorm(h)

    def forward(self, ht, hs, g, c):
        A, mt, ms = g["A"], g["mt"], g["ms"]
        At = A.transpose(1, 2)
        m_pred = (A @ ht) / A.sum(-1, keepdim=True).clamp(min=1)
        m_succ = (At @ ht) / At.sum(-1, keepdim=True).clamp(min=1)
        upd_t = F.relu(self.self_t(ht) + self.pred(m_pred) + self.succ(m_succ)
                       + self.srv(masked_mean(hs, ms)).unsqueeze(1) + self.ctx(c).unsqueeze(1))
        ht = self.norm_t(ht + upd_t) * mt.unsqueeze(-1)
        upd_s = F.relu(self.self_s(hs) + self.task(masked_mean(ht, mt)).unsqueeze(1))
        hs = self.norm_s(hs + upd_s) * ms.unsqueeze(-1)
        return ht, hs


class GraphEncoder(nn.Module):
    """Embeds subtask nodes, RES nodes and the vehicle/global context. With act_in=True
    (Critic) each subtask node also receives its offload decision and the embedding of the
    RES it is sent to."""
    def __init__(self, h, layers, act_in=False):
        super().__init__()
        self.act_in = act_in
        self.t_in = nn.Sequential(nn.Linear(FT + int(act_in), h), nn.ReLU(), nn.Linear(h, h))
        self.s_in = nn.Sequential(nn.Linear(FS, h), nn.ReLU(), nn.Linear(h, h))
        self.c_in = nn.Sequential(nn.Linear(FV + FG, h), nn.ReLU(), nn.Linear(h, h))
        if act_in:
            self.a_proj = nn.Linear(h, h)
        self.layers = nn.ModuleList([GNNLayer(h) for _ in range(layers)])

    def forward(self, g, x=None, ypi=None):
        c = self.c_in(torch.cat([g["xv"], g["xg"]], -1))
        hs = self.s_in(g["xs"]) * g["ms"].unsqueeze(-1)
        if self.act_in:
            ht = self.t_in(torch.cat([g["xt"], x.unsqueeze(-1)], -1))
            ht = ht + self.a_proj(x.unsqueeze(-1) * (ypi @ hs))       # chosen RES embedding
        else:
            ht = self.t_in(g["xt"])
        ht = (ht + c.unsqueeze(1)) * g["mt"].unsqueeze(-1)
        for layer in self.layers:
            ht, hs = layer(ht, hs, g, c)
        return ht, hs, c


class GNNOffloadActor(nn.Module):
    """Actor 1: offload probability per subtask + score per (subtask, RES) pair."""
    def __init__(self, h, layers):
        super().__init__()
        self.enc = GraphEncoder(h, layers)
        self.off = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
        self.pair = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, 1))

    def forward(self, g):
        ht, hs, _ = self.enc(g)
        B, K, H = ht.shape; M = hs.shape[1]
        p = torch.sigmoid(self.off(ht).squeeze(-1)) * g["mt"]
        pair = torch.cat([ht.unsqueeze(2).expand(B, K, M, H), hs.unsqueeze(1).expand(B, K, M, H)], -1)
        logit = self.pair(pair).squeeze(-1)
        logit = logit.masked_fill(g["ms"].unsqueeze(1) == 0, -1e4)
        return p, logit


class GNNFreqActor(nn.Module):
    """Actor 2: local CPU frequency from the pooled DAG embedding and the context."""
    def __init__(self, h, layers):
        super().__init__()
        self.enc = GraphEncoder(h, layers)
        self.head = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, 1))

    def forward(self, g):
        ht, _, c = self.enc(g)
        return torch.sigmoid(self.head(torch.cat([masked_mean(ht, g["mt"]), c], -1)))


class GNNCritic(nn.Module):
    def __init__(self, h, layers):
        super().__init__()
        self.enc = GraphEncoder(h, layers, act_in=True)
        self.head = nn.Sequential(nn.Linear(4 * h + 1, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(),
                                  nn.Linear(h, 1))

    def forward(self, g, x, ypi, u):
        ht, hs, c = self.enc(g, x, ypi)
        z = torch.cat([masked_mean(ht, g["mt"]), masked_max(ht, g["mt"]),
                       masked_mean(hs, g["ms"]), c, u], -1)
        return self.head(z).squeeze(-1)


class TaBRLGNN(TaBRL):
    """TaBRL with graph networks instead of the dual-FC Actor and FC Critic. Candidate
    generation (NOP + trust coupling), reward and update rules are inherited unchanged."""
    name, obs_mode, graph_obs = "TaBRL-GNN", "audited", True

    def __init__(self, env, cfg):
        BaseAgent.__init__(self, env, cfg)
        h, L = cfg["gnn_hidden"], cfg["gnn_layers"]
        self.actor_off, self.actor_f = GNNOffloadActor(h, L).to(DEVICE), GNNFreqActor(h, L).to(DEVICE)
        self.t_off, self.t_f = copy.deepcopy(self.actor_off), copy.deepcopy(self.actor_f)
        self.critic = GNNCritic(h, L).to(DEVICE)
        self.opt_a = torch.optim.Adam(list(self.actor_off.parameters()) + list(self.actor_f.parameters()),
                                      lr=cfg["lr_actor"])
        self.opt_c = torch.optim.Adam(self.critic.parameters(), lr=cfg["lr_critic"])
        self.buf, self.cap, self.pos = [], cfg["buffer"], 0
        self.rng = np.random.default_rng(cfg["seed"] + 5)
        self.train_every = cfg["gnn_train_every"]    # fewer updates than FC TaBRL (runtime)

    @torch.no_grad()
    def act(self, obs, task, explore):
        k, M = len(task.subs), obs["xs"].shape[0]
        g = batch_graphs([obs])
        p, l = [z[0].cpu().numpy() for z in self.actor_off(g)]
        tp, tl = [z[0].cpu().numpy() for z in self.t_off(g)]
        u, tu = float(self.actor_f(g)[0]), float(self.t_f(g)[0])
        load, trust, freq = obs["xs"][:, 0] * 10.0, np.clip(obs["xs"][:, 2], 1e-3, 1), obs["xs"][:, 4] * 5e9
        cands = self.candidates(p, l, u, tp, tl, tu, k, load, trust, freq, obs["xt"][:, 0])
        if explore and self.rng.random() < self.eps:
            best = (self.rng.integers(0, 2, k).astype(np.float32), self.rng.integers(0, M, k),
                    float(self.rng.random()))
        else:
            n = len(cands)
            x = torch.as_tensor(np.stack([c[0] for c in cands]), device=DEVICE)
            y = torch.as_tensor(np.stack([c[1] for c in cands]), device=DEVICE)
            u_c = torch.as_tensor([[c[2]] for c in cands], dtype=torch.float32, device=DEVICE)
            q = self.critic(repeat_graph(g, n), x, F.one_hot(y, M).float(), u_c)
            best = cands[int(q.argmax())]
        x, y, uu = best
        return x.astype(int).tolist(), [int(v) for v in y], self.f_of(uu), (x.astype(np.float32), np.asarray(y), uu)

    def store(self, task, r):
        item = (task.obs, task.act, r)
        if len(self.buf) < self.cap:
            self.buf.append(item)
        else:
            self.buf[self.pos] = item
        self.pos = (self.pos + 1) % self.cap

    def train(self):
        B = self.cfg["batch"]
        if len(self.buf) < B:
            return
        batch = [self.buf[i] for i in np.random.randint(0, len(self.buf), B)]
        g = batch_graphs([b[0] for b in batch])
        K, M = g["xt"].shape[1], g["xs"].shape[1]
        x = np.zeros((B, K), np.float32); y = np.zeros((B, K), np.int64)
        for i, (_, (xi, yi, _), _) in enumerate(batch):
            x[i, :len(xi)] = xi; y[i, :len(yi)] = yi
        x = torch.as_tensor(x, device=DEVICE)
        y_oh = F.one_hot(torch.as_tensor(y, device=DEVICE), M).float()
        u = torch.as_tensor([[b[1][2]] for b in batch], dtype=torch.float32, device=DEVICE)
        r = torch.as_tensor([b[2] for b in batch], dtype=torch.float32, device=DEVICE)
        mt = g["mt"]

        lc = F.mse_loss(self.critic(g, x, y_oh, u), r)                # TD loss (per-slot, gamma = 0)
        self.opt_c.zero_grad(); lc.backward(); self.opt_c.step()

        p, logit = self.actor_off(g)
        uu = self.actor_f(g)
        q_pi = self.critic(g, p, torch.softmax(logit, -1), uu)
        with torch.no_grad():                                         # advantage-weighted imitation
            w = torch.exp(r - q_pi).clamp(max=10.0); w = w / w.mean()
        bce = (F.binary_cross_entropy(p.clamp(1e-5, 1 - 1e-5), x, reduction="none") * mt).sum(1)
        ce = (-(torch.log_softmax(logit, -1) * y_oh).sum(-1) * x * mt).sum(1)
        mse = (uu.squeeze(1) - u.squeeze(1)) ** 2
        la = (w * (bce + ce + mse)).mean() - 0.1 * q_pi.mean()
        self.opt_a.zero_grad(); la.backward(); self.opt_a.step()
        soft_update(self.t_off, self.actor_off, self.cfg["tau"])
        soft_update(self.t_f, self.actor_f, self.cfg["tau"])


ALGOS["TaBRL-GNN"] = TaBRLGNN
if "TaBRL-GNN" not in ORDER:
    ORDER.insert(1, "TaBRL-GNN")
C_LINES["TaBRL-GNN"] = "#E69F00"
CFG.update(gnn_hidden=64, gnn_layers=2, gnn_train_every=2)


# %% [markdown]
# ## Thesis experiments

# %%
def plot_bars(res, algs, path, title):
    """Single-panel grouped bars (completion time, processing capacity) for a test result."""
    fig, ax = plt.subplots(figsize=(1.6 * len(algs) + 3, 4.8))
    _bar_panel(ax, {a: res[a] for a in algs}, title=title, headroom=(1.2, 1.2))
    fig.legend(["Completion time", "Processing capacity"], loc="lower center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, 0.06, 1, 1)); fig.savefig(path, dpi=300); plt.close(fig)


def plot_scaling(xs, curves, xlabel, path, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    for a, pts in curves.items():
        xv = [x for x in xs if x in pts]
        axes[0].plot(xv, [pts[x]["completion"] for x in xv], marker="o", lw=2, label=a, color=C_LINES.get(a))
        axes[1].plot(xv, [pts[x]["throughput"] / 1e3 for x in xv], marker="o", lw=2, label=a,
                     color=C_LINES.get(a))
    axes[0].set_ylabel("Mean completion time (s, log scale)"); axes[1].set_ylabel("Processing capacity (kbit/s)")
    axes[0].set_yscale("log")        # overloaded points (100+ s) would flatten the rest
    for ax in axes:
        ax.set_xlabel(xlabel); ax.set_xticks(xs); ax.grid(alpha=0.3)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axes[0].legend(frameon=False)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def thesis_main(cfg=CFG, run_e1=True, run_e2=True, run_e3=True):
    out = cfg["out_dir"]; os.makedirs(out, exist_ok=True)
    summary = {}

    # ---- E1: base scenario, all algorithms (produces the same figures as the base notebook)
    agents = {}
    if run_e1 or run_e3:
        print("\n######## E1: base scenario ########")
        res, res4, hists = main(dict(cfg, out_dir=os.path.join(out, "E1_base")), return_agents=agents)
        summary["E1"] = res

    # ---- E2: larger DAGs (2..k_large subtasks); load kept comparable by scaling arrivals
    if run_e2:
        print("\n######## E2: larger DAGs ########")
        c2 = dict(cfg, k_max=cfg["k_large"])
        tr2, te2 = load_dag_jobs(c2)
        mean_k = np.mean([len(j["b"]) for j in tr2])
        c2["arrival_rate"] = cfg["arrival_rate"] * cfg["base_mean_k"] / mean_k
        print(f"  mean subtasks {mean_k:.2f} -> arrival_rate {c2['arrival_rate']:.3f}")
        N = c2["n_veh"]
        win = fixed_windows(te2, N, c2["horizon"], c2, c2["eval_windows"], seed=2)
        res2 = {}
        for a in ORDER:                                   # every algorithm, as in E1
            ag, _ = train_agent(a, tr2, c2, N)
            res2[a] = evaluate(ag, win, c2, N)
            print(f"  E2 {a:10s} completion {res2[a]['completion']:.3f}s  capacity {res2[a]['throughput']/1e3:.0f} kbit/s")
        plot_bars(res2, list(res2), os.path.join(out, "E2_large_dags.png"),
                  f"Test set, DAGs with up to {cfg['k_large']} subtasks")
        t2 = results_table({"test": res2}).drop(columns="Split")
        t2.to_csv(os.path.join(out, "E2_values.csv"), index=False)
        print_table(f"E2 values: DAGs with up to {cfg['k_large']} subtasks (test set)", t2)
        summary["E2"] = res2

    # ---- E3: zero-shot scalability of the E1 policies (trained with N=10, M=4)
    if run_e3:
        print("\n######## E3: zero-shot scalability ########")
        _, te = load_dag_jobs(cfg)
        veh = {a: {} for a in ORDER}
        for nv in cfg["scale_veh"]:
            win = fixed_windows(te, nv, cfg["horizon"], cfg, cfg["eval_windows"], seed=50 + nv)
            for a in ORDER:
                veh[a][nv] = evaluate(agents[a], win, cfg, nv)
            print(f"  N={nv:2d}  " + "  ".join(f"{a} {veh[a][nv]['completion']:.2f}s" for a in ORDER))
        plot_scaling(list(cfg["scale_veh"]), veh, "Number of vehicles N",
                     os.path.join(out, "E3_scale_vehicles.png"),
                     f"Zero-shot: trained with N = {cfg['n_veh']}, tested with N = {list(cfg['scale_veh'])}")
        srv = {"TaBRL-GNN": {}, "DAG-Local": {}}
        for m in cfg["scale_res"]:
            cm = dict(cfg, n_res=m)
            win = fixed_windows(te, cfg["n_veh"], cfg["horizon"], cm, cfg["eval_windows"], seed=80 + m)
            for a in ("TaBRL-GNN", "DAG-Local"):          # size-agnostic policies
                srv[a][m] = evaluate(agents[a], win, cm, cfg["n_veh"])
            if m == cfg["n_res"]:                  # FC networks only run with the training M
                for a in ORDER:
                    srv.setdefault(a, {})[m] = evaluate(agents[a], win, cm, cfg["n_veh"])
            print(f"  M={m}  GNN {srv['TaBRL-GNN'][m]['completion']:.2f}s  "
                  f"Local {srv['DAG-Local'][m]['completion']:.2f}s")
        plot_scaling(list(cfg["scale_res"]), srv, "Number of edge servers M",
                     os.path.join(out, "E3_scale_servers.png"),
                     f"Zero-shot: trained with M = {cfg['n_res']} (FC agents only run at M = {cfg['n_res']})")
        rows = [dict(Sweep="vehicles", Size=n, Algorithm=a, **{
                    "Completion time (s)": round(m["completion"], 3),
                    "Processing capacity (kbit/s)": round(m["throughput"] / 1e3, 1),
                    "System throughput (Mbit/s)": round(m["sys_throughput"] / 1e6, 3)})
                for a, pts in veh.items() for n, m in pts.items()]
        rows += [dict(Sweep="servers", Size=n, Algorithm=a, **{
                    "Completion time (s)": round(m["completion"], 3),
                    "Processing capacity (kbit/s)": round(m["throughput"] / 1e3, 1),
                    "System throughput (Mbit/s)": round(m["sys_throughput"] / 1e6, 3)})
                 for a, pts in srv.items() for n, m in pts.items()]
        t3 = pd.DataFrame(rows).sort_values(["Sweep", "Size", "Algorithm"])
        t3.to_csv(os.path.join(out, "E3_values.csv"), index=False)
        print_table("E3 values: zero-shot scalability (test set)", t3)
        summary["E3"] = dict(vehicles=veh, servers=srv)

    with open(os.path.join(out, "thesis_results.json"), "w") as fh:
        json.dump(summary, fh, default=float)
    print(f"\nThesis figures written to {out}/")
    return summary


thesis_main = logged(thesis_main, "thesis_log.txt")


CFG.update(k_large=20, base_mean_k=5.0, scale_veh=(5, 10, 15, 20), scale_res=(3, 4, 6, 8))

def apply_thesis_args(cfg):
    """Thesis-only command-line options (the common ones, including the required --epochs,
    are read by apply_cli_args)."""
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--experiments", default="e1,e2,e3", help="which experiments to run, e.g. e1,e3")
    ap.add_argument("--gnn-train-every", type=int, help="TaBRL-GNN updates once every N finished tasks")
    ap.add_argument("--k-large", type=int, help="E2: largest DAG size")
    ap.add_argument("--scale-veh", help="E3: vehicle counts, e.g. 5,10,15,20")
    ap.add_argument("--scale-res", help="E3: edge-server counts, e.g. 3,4,6,8")
    args, _ = ap.parse_known_args()
    if args.gnn_train_every is not None:
        cfg["gnn_train_every"] = args.gnn_train_every
    if args.k_large is not None:
        cfg["k_large"] = args.k_large
    if args.scale_veh:
        cfg["scale_veh"] = tuple(int(v) for v in args.scale_veh.split(","))
    if args.scale_res:
        cfg["scale_res"] = tuple(int(v) for v in args.scale_res.split(","))
    run = {e.strip().lower() for e in args.experiments.split(",")}
    return cfg, dict(run_e1="e1" in run, run_e2="e2" in run, run_e3="e3" in run)


if __name__ == "__main__":
    cfg, runs = apply_thesis_args(apply_cli_args(CFG))
    thesis_main(cfg, **runs)


In [ ]:
%%writefile pq_crypto.py
"""Post-quantum cryptography for the PBFT consensus of CoSched-GNN.

Real cryptography, not a model:
  * signatures: ML-DSA-44 / 65 / 87 (CRYSTALS-Dilithium, NIST FIPS 204), and ECDSA P-256 as
    the classical comparison;
  * key exchange: ML-KEM-768 (CRYSTALS-Kyber, NIST FIPS 203) for every vehicle-to-server
    session, followed by AES-256-GCM encryption of the uploaded task descriptors;
  * hashes: SHA3-256.

Backends: liboqs (Open Quantum Safe, `pip install liboqs-python`) is used when available,
otherwise the `pqcrypto` wheel (slower signing); ECDSA and AES-GCM come from `cryptography`.

The time each operation takes is measured on the running machine (`measure`) and fed into
the simulated consensus delay, so the delay uses measured, not assumed, costs.

Quantum adversary: a cryptographically relevant quantum computer runs Shor's algorithm and
recovers an ECDSA secret key from its public key. `Signer.recover_secret` models exactly that
capability (it returns the secret key of a classical scheme); no such attack is known for
ML-DSA, so it returns None there.
"""
import hashlib
import os
import time

SCHEMES = {                       # name used in the code -> (algorithm, pqcrypto module, level)
    "ecdsa": ("ECDSA-P256", None, "classical (broken by Shor's algorithm)"),
    "dilithium2": ("ML-DSA-44", "ml_dsa_44", "post-quantum, NIST level 2"),
    "dilithium3": ("ML-DSA-65", "ml_dsa_65", "post-quantum, NIST level 3"),
    "dilithium5": ("ML-DSA-87", "ml_dsa_87", "post-quantum, NIST level 5"),
}
KEM_ALG, KEM_MOD = "ML-KEM-768", "ml_kem_768"
INSTALL_HINT = ("Post-quantum library missing. On Kaggle run first:\n"
                "  !pip install -q liboqs-python pqcrypto\n"
                "(liboqs-python builds liboqs on its first import, needs Internet on), "
                "or run with --set crypto=model to skip the real cryptography.")


def _load_oqs():
    try:
        import oqs                                          # noqa: F401
        return oqs
    except (Exception, SystemExit):                         # liboqs missing or failed to build
        return None


def _load_pqcrypto():
    try:
        import pqcrypto.sign                                # noqa: F401
        import pqcrypto.kem                                 # noqa: F401
        return True
    except Exception:
        return False


def backend_name():
    if _load_oqs() is not None:
        return "liboqs"
    if _load_pqcrypto():
        return "pqcrypto"
    return None


def sha3(*parts):
    h = hashlib.sha3_256()
    for p in parts:
        h.update(p if isinstance(p, bytes) else str(p).encode())
        h.update(b"|")
    return h.digest()


class Signer:
    """keygen / sign / verify for one signature scheme."""

    def __init__(self, scheme):
        self.scheme = scheme
        self.alg, self.mod_name, self.level = SCHEMES[scheme]
        self.post_quantum = scheme != "ecdsa"
        self._secret_of = {}                    # pk -> secret handle (only used by recover_secret)
        if scheme == "ecdsa":
            from cryptography.hazmat.primitives.asymmetric import ec
            from cryptography.hazmat.primitives import hashes, serialization
            from cryptography.exceptions import InvalidSignature
            self._ec, self._hashes, self._ser, self._bad = ec, hashes, serialization, InvalidSignature
            self.backend = "cryptography"
            self._pub_cache = {}
            self.sig_bytes = 72                 # DER-encoded ECDSA P-256 (at most)
            return
        oqs = _load_oqs()
        if oqs is not None:
            self.backend, self._oqs = "liboqs", oqs
            self._verifier = oqs.Signature(self.alg)
            self.sig_bytes = self._verifier.details["length_signature"]
        elif _load_pqcrypto():
            import importlib
            self.backend = "pqcrypto"
            self._m = importlib.import_module(f"pqcrypto.sign.{self.mod_name}")
            self.sig_bytes = self._m.SIGNATURE_SIZE
        else:
            raise RuntimeError(INSTALL_HINT)

    def keygen(self):
        if self.scheme == "ecdsa":
            sk = self._ec.generate_private_key(self._ec.SECP256R1())
            pk = sk.public_key().public_bytes(self._ser.Encoding.X962,
                                              self._ser.PublicFormat.UncompressedPoint)
        elif self.backend == "liboqs":
            sk = self._oqs.Signature(self.alg)
            pk = sk.generate_keypair()
        else:
            pk, sk = self._m.keygen()
        self._secret_of[pk] = sk
        return pk, sk

    def sign(self, sk, msg):
        if self.scheme == "ecdsa":
            return sk.sign(msg, self._ec.ECDSA(self._hashes.SHA256()))
        if self.backend == "liboqs":
            return sk.sign(msg)
        return self._m.sign(sk, msg)

    def verify(self, pk, msg, sig):
        try:
            if self.scheme == "ecdsa":
                key = self._pub_cache.get(pk)
                if key is None:
                    key = self._pub_cache[pk] = self._ec.EllipticCurvePublicKey.from_encoded_point(
                        self._ec.SECP256R1(), pk)
                key.verify(sig, msg, self._ec.ECDSA(self._hashes.SHA256()))
                return True
            if self.backend == "liboqs":
                return bool(self._verifier.verify(msg, sig, pk))
            self._m.verify(pk, msg, sig)
            return True
        except Exception:
            return False

    def recover_secret(self, pk):
        """Quantum adversary (Shor): recovers classical secret keys, not ML-DSA ones."""
        return None if self.post_quantum else self._secret_of.get(pk)


class KEM:
    """ML-KEM-768 key encapsulation + AES-256-GCM for vehicle-to-server sessions."""

    def __init__(self):
        from cryptography.hazmat.primitives.ciphers.aead import AESGCM
        self._aead = AESGCM
        oqs = _load_oqs()
        if oqs is not None:
            self.backend, self._oqs = "liboqs", oqs
            self.ct_bytes = oqs.KeyEncapsulation(KEM_ALG).details["length_ciphertext"]
        elif _load_pqcrypto():
            import importlib
            self.backend = "pqcrypto"
            self._m = importlib.import_module(f"pqcrypto.kem.{KEM_MOD}")
            self.ct_bytes = self._m.CIPHERTEXT_SIZE
        else:
            raise RuntimeError(INSTALL_HINT)

    def keygen(self):
        if self.backend == "liboqs":
            k = self._oqs.KeyEncapsulation(KEM_ALG)
            return k.generate_keypair(), k
        return self._m.keygen()

    def encaps(self, pk):
        if self.backend == "liboqs":
            ct, ss = self._oqs.KeyEncapsulation(KEM_ALG).encap_secret(pk)
            return ct, ss
        return self._m.encaps(pk)

    def decaps(self, sk, ct):
        if self.backend == "liboqs":
            return sk.decap_secret(ct)
        return self._m.decaps(sk, ct)

    def seal(self, key, plaintext, aad=b""):
        nonce = os.urandom(12)
        return nonce + self._aead(key[:32]).encrypt(nonce, plaintext, aad)

    def open(self, key, blob, aad=b""):
        return self._aead(key[:32]).decrypt(blob[:12], blob[12:], aad)


_SIGNERS, _COSTS = {}, {}


def signer(scheme):
    if scheme not in _SIGNERS:
        _SIGNERS[scheme] = Signer(scheme)
    return _SIGNERS[scheme]


def measure(scheme, reps=300):
    """Mean wall-clock seconds of sign / verify on this machine, plus sizes (cached)."""
    if scheme in _COSTS:
        return _COSTS[scheme]
    s = signer(scheme)
    pk, sk = s.keygen()
    msg = os.urandom(96)
    for _ in range(20):                                     # warm-up
        sig = s.sign(sk, msg); s.verify(pk, msg, sig)
    t = time.perf_counter()
    for _ in range(reps):
        sig = s.sign(sk, msg)
    t_sign = (time.perf_counter() - t) / reps
    t = time.perf_counter()
    for _ in range(reps):
        s.verify(pk, msg, sig)
    t_verify = (time.perf_counter() - t) / reps
    r3 = lambda v: float(f"{v:.3g}")                        # 3 significant digits
    _COSTS[scheme] = dict(sign_s=r3(t_sign), verify_s=r3(t_verify), sig_bytes=s.sig_bytes,
                          backend=s.backend, alg=s.alg, level=s.level)
    return _COSTS[scheme]


In [ ]:
%%writefile cosched_gnn.py
# %% [markdown]
# # CoSched-GNN: consensus-aware DAG offloading and PBFT validator co-selection
#
# Extension of TaBRL (Yin et al., IEEE Comm. Standards Mag. 2026). One heterogeneous-GNN
# agent decides, in a single action, where every DAG subtask runs and which edge servers
# form the PBFT committee that verifies the offloaded results.
#
# * **N1 consensus-aware critical path:** consensus delay is part of the DAG timing, so only
#   the part of it that exceeds a subtask's slack lengthens completion time.
# * **N2 post-offloading load:** validators are chosen from the load each server will have
#   after this decision's subtasks and this slot's verification work are added.
# * **N3 executor exclusion:** a server that executes any subtask of a DAG is never one of
#   that DAG's validators.
# * **N4 one joint action:** offloading head + validator head on one graph encoder, PPO.
# * **P3 post-quantum signatures:** the signature size and sign/verify cycles of the chosen
#   scheme (ECDSA, Dilithium-2/3/5) enter the consensus delay.
#
# Everything the TaBRL environment already models (V2I channel, multi-level feedback
# queues, vehicle queues, Lyapunov reward, Alibaba trace, compromised server) is reused
# from tabrl_iov.py unchanged. Only the consensus model, the action and the agent change.
#
# Verification work is real work in the simulator: each validator gets a verification job
# of c_cons cycles in its MFQ, so it competes with offloaded subtasks for the CPU. A result
# commits when q = 2f + 1 validators have accepted it, plus the PBFT message rounds.

# %%
from tabrl_gnn import *  # notebook: inlined
from tabrl_gnn import _bar_panel, masked_mean, masked_max  # notebook: inlined
import itertools
import os

import pq_crypto as PQ  # notebook: inlined

# Signature schemes: ecdsa (classical), dilithium2/3/5 = ML-DSA-44/65/87 (post-quantum).
# With crypto="real" (default) every result, vote and quorum certificate is really signed
# and verified, and every vehicle-server session uses ML-KEM-768 + AES-256-GCM (pq_crypto.py).
# The consensus delay uses the sign/verify times measured on this machine.
SIG = PQ.SCHEMES

CO_CFG = dict(
    n_res=8,                     # needs M >= n + (servers used by one DAG); spec default 8
    pbft_f=1,                    # n = 3f + 1 validators, quorum q = 2f + 1
    c_chk=2e6,                   # result-check cycles per verified subtask
    sig_scheme="dilithium2",
    crypto="real",               # "real": real ML-DSA / ML-KEM operations; "model": costs only
    host_ghz=3.0,                # converts measured host seconds into CPU cycles
    p_forge=0.5,                 # a malicious executor whose tampered result is rejected tries
                                 # to forge a quorum certificate with this probability
    quantum_adversary=False,     # attacker can run Shor's algorithm (breaks ECDSA keys)
    p_mitm=0.0,                  # probability a result is altered on the backhaul in transit
    s_hdr=200, s_res=1024,       # PBFT header and result-proof size (bytes)
    r_bh=1e9,                    # RES-RES backhaul rate (bit/s) for consensus messages
    rtt_range=(0.002, 0.010),    # RES-RES round-trip time, grows with distance (s)
    load_horizon=1.0,            # H in u_post (s)
    cpu_floor=0.05,              # epsilon in tau_k
    q_max=5e9,                   # RES queue cap for the offloading mask (cycles)
    t_max_blk=1.0,               # consensus-delay limit (s)
    dead_range=(2.0, 6.0),       # DAG deadline drawn per DAG (s)
    lam_dl=1.0, lam_blk=0.5,     # deadline / consensus-limit penalties
    cons_scale=1.0,              # multiplies RTT and c_chk (hidden-fraction experiment)
    # CoSched-GNN network and PPO
    co_hidden=128, co_layers=3, co_heads=4,
    freq_levels=(0.5e9, 1.0e9, 1.5e9, 2.0e9, 2.5e9),
    ppo_rollout=512, ppo_epochs=4, ppo_minibatch=64, ppo_clip=0.2,
    # gamma = 0: the Lyapunov drift-plus-penalty decouples the problem into independent
    # per-slot problems (TaBRL, P2), so each DAG's reward is credited to its own decision.
    # The spec's 0.99 / 0.95 learned far slower in our tests; set them with --set if needed.
    ppo_gamma=0.0, ppo_lambda=0.0, ppo_lr=3e-4,
    ppo_ent=(0.01, 0.001), ppo_vf=0.5, ppo_grad_clip=0.5,
    reward_norm=5.0,
    bc_epochs=0,                 # optional warm start: epochs imitating CP-Greedy before PPO
)


def co_cfg(base=None, **kw):
    c = dict(base if base is not None else CFG)
    c.update(CO_CFG)
    c.update(kw)
    return c


# %% [markdown]
# ## 1. DAG timing, slack and rank (section 4.5 and 5.3)

# %%
def children_of(preds):
    ch = [[] for _ in preds]
    for v, ps in enumerate(preds):
        for p in ps:
            ch[p].append(v)
    return ch


def rank_slack(preds, dur):
    """Upward rank (HEFT), earliest/latest finish and slack of every node, from the expected
    durations `dur` (consensus included for edge options)."""
    n = len(preds)
    order = topo_order(preds)
    ch = children_of(preds)
    ef = np.zeros(n)
    for v in order:
        ef[v] = max((ef[p] for p in preds[v]), default=0.0) + dur[v]
    mk = float(ef.max()) if n else 0.0
    lf = np.full(n, mk)
    for v in reversed(order):
        if ch[v]:
            lf[v] = min(lf[c] - dur[c] for c in ch[v])
    rank = np.zeros(n)
    for v in reversed(order):
        rank[v] = dur[v] + max((rank[c] for c in ch[v]), default=0.0)
    return rank, lf - ef, ef, lf, mk


def static_schedule(preds, place, t_loc, t_edge, t_cons):
    """Reference DAG timing used by the unit tests and the Gantt figure.
    place[v] = 0 (vehicle) or 1 (edge). One vehicle CPU runs local subtasks non-preemptively,
    highest upward rank first; edge subtasks run in parallel. A child is ready only when every
    parent is done and, if the parent was offloaded, committed (safety invariant).
    Returns makespan, (start, fin, done) per subtask and the exposed consensus delay."""
    n = len(preds)
    dur = [t_loc[v] if place[v] == 0 else t_edge[v] + t_cons[v] for v in range(n)]
    rank = rank_slack(preds, dur)[0]
    start, fin, done = [None] * n, [None] * n, [None] * n
    cpu, left = 0.0, set(range(n))
    while left:
        prog = False
        for v in sorted(left, key=lambda z: -rank[z]):
            if place[v] == 1 and all(done[p] is not None for p in preds[v]):
                start[v] = max((done[p] for p in preds[v]), default=0.0)
                fin[v] = start[v] + t_edge[v]; done[v] = fin[v] + t_cons[v]
                left.discard(v); prog = True
        loc = [v for v in left if place[v] == 0 and all(done[p] is not None for p in preds[v])]
        if loc:
            ready = {v: max((done[p] for p in preds[v]), default=0.0) for v in loc}
            t0 = max(cpu, min(ready.values()))
            cand = [v for v in loc if ready[v] <= t0 + 1e-12]
            v = max(cand, key=lambda z: rank[z])
            start[v] = t0; fin[v] = done[v] = t0 + t_loc[v]; cpu = fin[v]
            left.discard(v); prog = True
        if not prog:
            raise ValueError("cycle in DAG")
    mk = max(done)
    return mk, start, fin, done


def exposed_delay(preds, place, t_loc, t_edge, t_cons, v):
    """How much the makespan grows because of subtask v's consensus delay."""
    tc = list(t_cons)
    full = static_schedule(preds, place, t_loc, t_edge, tc)[0]
    tc[v] = 0.0
    return full - static_schedule(preds, place, t_loc, t_edge, tc)[0]


# %% [markdown]
# ## 2. PBFT consensus model (section 4.4)

# %%
# Used only with crypto="model" when no PQC library is installed (approximate values).
_FALLBACK_COSTS = {"ecdsa": (72, 4e-5, 1.1e-4), "dilithium2": (2420, 7e-5, 2.2e-5),
                   "dilithium3": (3309, 1e-4, 3.4e-5), "dilithium5": (4627, 1.2e-4, 4.9e-5)}


def pq_costs(cfg):
    """Signature size and sign / verify time of the configured scheme, measured on this
    machine with the real library (fallback table only in model mode without a library)."""
    sch = cfg["sig_scheme"]
    try:
        return PQ.measure(sch)
    except RuntimeError:
        if cfg["crypto"] == "real":
            raise
        b, ts, tv = _FALLBACK_COSTS[sch]
        return dict(sign_s=ts, verify_s=tv, sig_bytes=b, backend="none (fallback table)",
                    alg=PQ.SCHEMES[sch][0], level=PQ.SCHEMES[sch][2])


def c_cons_cycles(cfg):
    """CPU cycles one validator spends per verified subtask: result check, verify the
    executor's signature, sign its prepare and commit votes, verify the other n-1 validators'
    prepare and commit votes."""
    n = 3 * cfg["pbft_f"] + 1
    pc = pq_costs(cfg)
    hz = cfg["host_ghz"] * 1e9
    return (cfg["c_chk"] * cfg["cons_scale"] + 2 * pc["sign_s"] * hz
            + (1 + 2 * (n - 1)) * pc["verify_s"] * hz)


def t_msg(cfg, rtt, C):
    """One PBFT phase: half the largest RTT inside the committee + (n - 1) messages."""
    n = len(C)
    s_msg = (cfg["s_hdr"] + pq_costs(cfg)["sig_bytes"]) * 8
    worst = max((rtt[a, b] for a in C for b in C), default=0.0)
    return 0.5 * worst + (n - 1) * s_msg / cfg["r_bh"]


def t_cons_formula(cfg, rtt, F, u, C):
    """Closed-form T_cons(v, C) = S_res/R_bh + 3 T_msg(C) + tau_(q) used for features and the
    heuristics. The simulator itself runs verification as CPU jobs (see CoEnv)."""
    q = 2 * cfg["pbft_f"] + 1
    cc = c_cons_cycles(cfg)
    tau = sorted(cc / (F[k] * max(cfg["cpu_floor"], 1.0 - u[k])) for k in C)
    return cfg["s_res"] * 8 / cfg["r_bh"] + 3 * t_msg(cfg, rtt, C) + tau[q - 1]


class _ModelSig(bytes):
    """Signature placeholder in model mode: carries whether it would verify."""
    def __new__(cls, valid):
        o = super().__new__(cls, b"model")
        o.model_valid = valid
        return o


class VJob:
    """Verification job of one validator for one offloaded subtask; it sits in the
    validator's MFQ like any subtask, so validation and computation share the CPU."""
    __slots__ = ("rnd", "k", "accept", "rem", "b", "level", "avail_at", "done_at", "preds")

    def __init__(self, rnd, k, accept, cycles, avail_at):
        self.rnd, self.k, self.accept = rnd, k, accept
        self.rem = self.b = cycles
        self.level, self.avail_at, self.done_at, self.preds = 0, avail_at, None, []

    def ready(self, t):
        return True


class Round:
    """One PBFT instance: the result of subtask `s` executed on RES `ex`. `msg`/`sig` are the
    executor's signed result as received (possibly altered in transit), `expected` is the
    digest a correct execution gives, `cert` collects the signed accept votes."""
    __slots__ = ("s", "ex", "tampered", "t0", "C", "tmsg", "acc", "rej", "votes", "jobs", "decided",
                 "msg", "sig", "digest", "expected", "altered", "cert", "sig_bad")

    def __init__(self, s, ex, tampered, t0, C, tmsg):
        self.s, self.ex, self.tampered, self.t0, self.C, self.tmsg = s, ex, tampered, t0, C, tmsg
        self.acc = self.rej = self.votes = self.jobs = 0
        self.decided = self.altered = self.sig_bad = False
        self.msg = self.sig = self.digest = self.expected = None
        self.cert = []


# %% [markdown]
# ## 3. Environment with committee-based consensus

# %%
class CoRES(RES):
    """RES of tabrl_iov with one fix: a job demoted when its MFQ slice ends goes back to the
    queues only after every core has finished the current step, so another core cannot run
    the same job in the same step (it used to run on two cores at once)."""

    def step(self, t, dt, env):
        if self.incoming:
            keep = []
            for s in self.incoming:
                (self.pending if s.avail_at <= t else keep).append(s)
            self.incoming = keep
        if self.pending:
            keep = []
            for s in self.pending:
                if s.ready(t):
                    s.level = 0; self.q[0].append(s)
                else:
                    keep.append(s)
            self.pending = keep
        demoted = []
        for core in self.cores:
            budget = dt
            while budget > 1e-9:
                if core[0] is None:
                    lvl = next((l for l in range(3) if self.q[l]), None)
                    if lvl is None:
                        break
                    core[0], core[1] = self.q[lvl].popleft(), self.slices[lvl]
                s = core[0]
                run = min(budget, s.rem / self.F, core[1])
                s.rem -= run * self.F; budget -= run; core[1] -= run
                if s.rem <= 1.0:
                    core[0] = None
                    env.res_finished(self, s, t + dt - budget)
                elif core[1] <= 1e-9:          # slice used up -> demote, requeue after this step
                    s.level = min(s.level + 1, 2)
                    demoted.append(s); core[0] = None
        for s in demoted:
            self.q[s.level].append(s)


class CoEnv(IoVEnv):
    """TaBRL environment with the fixed D_blk/R_blk confirmation replaced by a per-DAG PBFT
    committee. Agents that do not choose validators use a random committee that is fixed for
    the whole episode (spec section 10.1)."""

    def __init__(self, cfg, n_veh, seed):
        super().__init__(cfg, n_veh, seed)
        for r in self.res:
            r.__class__ = CoRES
        self.n_cmt, self.q = 3 * cfg["pbft_f"] + 1, 2 * cfg["pbft_f"] + 1
        if self.M < self.n_cmt:
            raise ValueError(f"need at least n = {self.n_cmt} edge servers")
        lo, hi = cfg["rtt_range"]
        x = self.rsu[:, 0]
        d = np.abs(x[:, None] - x[None, :]) / max(cfg["road_len"], 1.0)
        self.rtt = (lo + (hi - lo) * d) * cfg["cons_scale"]
        np.fill_diagonal(self.rtt, 0.0)
        self.Fk = np.array([r.F for r in self.res])
        self.cores = np.array([len(r.cores) for r in self.res], float)
        self.fixed_committee = sorted(self.rng.choice(self.M, self.n_cmt, replace=False).tolist())
        self.cc = c_cons_cycles(cfg)
        self.ver_slot = np.zeros(self.M)       # verification cycles committed this slot
        self.served = np.zeros(self.M)         # committees each RES serves this slot
        self.n_off = self.n_selfval = self.tamper_committed = self.liveness_fail = 0
        # ---- post-quantum security layer ----
        self.real = cfg["crypto"] == "real"
        self.pc = pq_costs(cfg)
        self.post_quantum = cfg["sig_scheme"] != "ecdsa"
        self.forge_attempts = self.forge_accepted = self.mitm_detected = 0
        self.n_sign = self.n_verify = self.n_kem = 0
        self.sessions, self.uid = {}, 0
        self.kem_ct = 1088
        if self.real:
            self.signer = PQ.signer(cfg["sig_scheme"])
            self.sig_keys = [self.signer.keygen() for _ in range(self.M)]      # (pk, sk) per RES
            self.kem = PQ.KEM()
            self.kem_ct = self.kem.ct_bytes
            self.kem_keys = [self.kem.keygen() for _ in range(self.M)]

    # ---- cryptographic primitives (real, or their outcome in model mode) ----
    def _sign(self, j, msg):
        self.n_sign += 1
        return self.signer.sign(self.sig_keys[j][1], msg) if self.real else b"model"

    def _verify(self, j, msg, sig, valid_in_model=True):
        self.n_verify += 1
        return self.signer.verify(self.sig_keys[j][0], msg, sig) if self.real else valid_in_model

    def _session(self, vid, j):
        """ML-KEM-768 handshake between vehicle vid and RES j (once per pair and episode);
        returns True the first time, so the handshake ciphertext is added to the upload."""
        if (vid, j) in self.sessions:
            return False
        self.n_kem += 1
        if self.real:
            pk, sk = self.kem_keys[j]
            ct, ss_vehicle = self.kem.encaps(pk)
            ss_res = self.kem.decaps(sk, ct)
            assert ss_res == ss_vehicle
            self.sessions[(vid, j)] = ss_vehicle
        else:
            self.sessions[(vid, j)] = b"model"
        return True

    # ---- load views ----
    def backlog(self):
        return np.array([r.backlog_cycles() for r in self.res])

    def capacity(self):
        return self.Fk * self.cores * self.cfg["load_horizon"]

    def u_pre(self):
        return np.minimum(1.0, (self.backlog() + self.ver_slot) / self.capacity())

    def u_post(self, new_cycles):
        return np.minimum(1.0, (self.backlog() + new_cycles + self.ver_slot) / self.capacity())

    def tcons_est(self, C, u):
        return t_cons_formula(self.cfg, self.rtt, self.Fk, u, C)

    def begin_slot(self):
        super().begin_slot()
        self.ver_slot[:] = 0.0; self.served[:] = 0.0

    # ---- actions ----
    def submit(self, task, x, y, f, committee=None):
        c, v = self.cfg, self.veh[task.vid]
        f = float(np.clip(f, c["f_local_min"], c["f_local_max"]))
        near, _ = self.nearest(v)
        rate = self.rate(v)
        C = sorted(int(k) for k in (committee if committee is not None else self.fixed_committee))
        assert len(C) == self.n_cmt and len(set(C)) == self.n_cmt
        new = np.zeros(self.M)
        for k, s in enumerate(task.subs):
            if x[k]:
                new[int(y[k])] += s.b
        u_post = self.u_post(new)
        task.info.update(Q=v.Q, E=v.E, trust=[r.trust() for r in self.res], x=list(x), y=list(y),
                         committee=C, u_val=float(np.mean(u_post[C])), cd={}, tc={}, ex={})
        task.committee = C
        self.uid += 1
        task.info["uid"] = self.uid
        n_off = 0
        for k, s in enumerate(task.subs):
            s.x, s.f = int(x[k]), f
            if s.x:
                j = int(y[k]); s.res = j; n_off += 1
                self.n_off += 1; self.n_selfval += int(j in C)
                st = max(self.t, v.tx_free)
                dur = c["overhead"] * s.d / rate
                v.tx_free = st + dur
                v.tx_bits.append((st, st + dur, s.d))
                e = c["p_tx"] * dur
                v.slot_energy += e; task.energy += e
                hop = abs(near - j) * (c["backhaul_hop_s"] + s.d / c["backhaul_bps"])
                hs = self.kem_ct * 8 / rate if self._session(task.vid, j) else 0.0   # KEM handshake
                if self.real:                     # task descriptor sent encrypted (AES-256-GCM)
                    key = self.sessions[(task.vid, j)]
                    desc = PQ.sha3(self.uid, k, s.b, s.d)
                    assert self.kem.open(key, self.kem.seal(key, desc, b"upload"), b"upload") == desc
                s.avail_at = st + dur + hs + hop      # no pre-execution confirmation any more
                self.res[j].incoming.append(s)
            else:
                v.local.append(s)
        self.ver_slot[C] += self.cc * n_off
        self.served[C] += 1
        self.tasks.append(task)

    # ---- consensus ----
    def res_finished(self, r, s, t):
        if isinstance(s, VJob):
            return self._vote(s, t)
        task = s.task
        task.info["cd"][s.k] = t
        tampered = bool(r.mal and self.rng.random() < self.cfg["p_tamper"])
        C = task.committee
        tm = t_msg(self.cfg, self.rtt, C)
        rnd = Round(s, r.j, tampered, t, C, tm)
        uid = task.info["uid"]
        rnd.expected = PQ.sha3(uid, s.k, "result")                 # what a correct run returns
        rnd.digest = os.urandom(32) if tampered else rnd.expected
        rnd.msg = PQ.sha3(b"RESULT", uid, s.k, rnd.digest, r.j)
        rnd.sig = self._sign(r.j, rnd.msg)                          # executor signs its result
        if self.rng.random() < self.cfg["p_mitm"]:                  # altered on the backhaul
            rnd.altered = True
            rnd.digest = os.urandom(32)
        arrive = t + self.cfg["s_res"] * 8 / self.cfg["r_bh"] + tm  # pre-prepare
        for k in C:
            if self.res[k].mal:
                if not tampered:
                    continue                          # Byzantine validator withholds its vote
                accept = True                         # ... or backs a tampered result
            else:
                accept = None                         # decided by real verification at vote time
            rnd.jobs += 1
            self.res[k].incoming.append(VJob(rnd, k, accept, self.cc, arrive))
        if rnd.jobs == 0:
            self._liveness(rnd, t)

    def _vote(self, job, t):
        rnd = job.rnd
        rnd.votes += 1
        if rnd.decided:
            return
        accept = job.accept
        if accept is None:                            # honest validator: check signature + result
            received = PQ.sha3(b"RESULT", rnd.s.task.info["uid"], rnd.s.k, rnd.digest, rnd.ex)
            sig_ok = self._verify(rnd.ex, received, rnd.sig, valid_in_model=not rnd.altered)
            rnd.sig_bad |= not sig_ok
            accept = sig_ok and rnd.digest == rnd.expected
        vote = PQ.sha3(b"VOTE", rnd.s.task.info["uid"], rnd.s.k, rnd.digest, job.k, accept)
        vsig = self._sign(job.k, vote)                # every vote is signed
        rnd.acc += accept; rnd.rej += not accept
        if accept:
            rnd.cert.append((job.k, vote, vsig))
        if rnd.acc >= self.q:                         # prepare + commit phases
            self._commit(rnd, t, rnd.cert[:self.q])
        elif rnd.rej >= self.n_cmt - self.q + 1:     # accept quorum impossible -> reject
            rnd.decided = True
            if rnd.sig_bad:
                self.mitm_detected += 1
            if rnd.tampered:
                self.res[rnd.ex].fail += 1
                if self.rng.random() < self.cfg["p_forge"] and self._forge(rnd, t):
                    return
            self._reexecute(rnd.s, t + 2 * rnd.tmsg)
        elif rnd.votes == rnd.jobs:                   # votes withheld: view-change timeout
            self._liveness(rnd, t)

    def _certificate_ok(self, rnd, cert):
        """The consumer of a result (child executor or vehicle) checks the quorum certificate:
        q distinct committee members, each vote signed under that member's public key."""
        ids = [k for k, _, _ in cert]
        if len(set(ids)) < self.q or not set(ids) <= set(rnd.C):
            return False
        ok = True
        for k, vote, vsig in cert:
            ok &= self._verify(k, vote, vsig, valid_in_model=getattr(vsig, "model_valid", True))
        return ok

    def _commit(self, rnd, t, cert, forged=False):
        rnd.decided = True
        t_commit = t + 2 * rnd.tmsg + self.q * self.pc["verify_s"]
        if rnd.tampered:
            self.tamper_committed += 1
        elif not forged:
            self.res[rnd.ex].ok += 1
        rnd.s.task.info["tc"][rnd.s.k] = t_commit - rnd.t0
        self.sub_done(rnd.s, t_commit)

    def _forge(self, rnd, t):
        """A rejected malicious executor forges accept votes for the honest committee members.
        It needs their secret keys: a quantum adversary recovers ECDSA keys with Shor's
        algorithm, nobody can recover ML-DSA keys. Returns True if the forgery was accepted."""
        self.forge_attempts += 1
        attacker = rnd.ex
        cert = []
        for k in rnd.C:
            vote = PQ.sha3(b"VOTE", rnd.s.task.info["uid"], rnd.s.k, rnd.digest, k, True)
            colluder = self.res[k].mal                        # Byzantine members sign for real
            if self.real:
                if colluder:
                    sk = self.sig_keys[k][1]
                else:
                    sk = self.signer.recover_secret(self.sig_keys[k][0]) if self.cfg["quantum_adversary"] else None
                    if sk is None:
                        sk = self.sig_keys[attacker][1]       # best it can do: its own key
                self.n_sign += 1
                cert.append((k, vote, self.signer.sign(sk, vote)))
            else:
                broken = self.cfg["quantum_adversary"] and not self.post_quantum
                cert.append((k, vote, _ModelSig(broken or colluder)))
            if len(cert) == self.q:
                break
        if self._certificate_ok(rnd, cert):
            self.forge_accepted += 1
            self._commit(rnd, t, cert, forged=True)
            return True
        return False

    def _liveness(self, rnd, t):
        rnd.decided = True
        self.liveness_fail += 1
        self._reexecute(rnd.s, t + self.cfg["t_max_blk"])

    def _reexecute(self, s, t):
        v = self.veh[s.task.vid]
        s.x, s.rem, s.avail_at, s.f = 0, s.b, t, self.cfg["f_local_max"]
        v.local.appendleft(s)


# %% [markdown]
# ## 4. Episode loop, metrics and reward

# %%
def task_consensus_stats(task):
    """T_cons and exposed consensus delay of every committed offloaded subtask. The exposed
    part is how much later a child became ready (or the DAG finished) because of the
    consensus round, measured on the realised schedule."""
    subs, cd, tc = task.subs, task.info["cd"], task.info["tc"]
    ch = children_of([[p.k for p in s.preds] for s in subs])
    tot = exp = 0.0
    for k, tcons in tc.items():
        s = subs[k]
        if ch[k]:
            need = []
            for c in ch[k]:
                other = [p.done_at for p in subs[c].preds if p is not s]
                need.append(max(other + [subs[c].avail_at if subs[c].x else task.arr]))
            e = max(max(0.0, s.done_at - max(cd[k], n)) for n in need)
        else:
            other = [u.done_at for u in subs if u is not s]
            e = max(0.0, s.done_at - max([cd[k]] + other))
        tot += tcons; exp += min(e, tcons)
    task.info["ex"] = dict(total=tot, exposed=exp, n=len(tc))
    return tot, exp


def co_reward(task, cfg, n1_off=False):
    """TaBRL's P2 term (throughput, task-queue and energy-queue pressure) credited when the
    DAG finishes, minus deadline and consensus-limit penalties (section 7.6). With n1_off
    (ablation A1) the completion time is replaced by the sum over subtasks of their own time
    plus their full consensus delay, i.e. TaBRL's additive view."""
    T = task_delay(task)
    if n1_off:
        tc = task.info["tc"]
        own = 0.0
        for s in task.subs:
            rd = max([task.arr] + [p.done_at for p in s.preds])
            end = task.info["cd"].get(s.k, s.done_at)
            own += max(end - rd, 1e-3) + tc.get(s.k, 0.0)
        T = own
    local_bits = sum(s.d for s, x in zip(task.subs, task.info["x"]) if not x) / 1e6
    r = cfg["V"] * (task.size / 1e6) / max(T, 1e-3)
    r -= cfg["alpha_q"] * (task.info["Q"] / 1e7) * local_bits
    r -= cfg["alpha_e"] * (task.info["E"] / 10.0) * task.energy
    r -= cfg["lam_dl"] * float(task_delay(task) > task.deadline)
    r -= cfg["lam_blk"] * float(max(task.info["tc"].values(), default=0.0) > cfg["t_max_blk"])
    return float(np.clip(r / cfg["reward_norm"], -10, 10))


def run_co_episode(agent, sched, cfg, n_veh, seed, train, horizon=None, trace=False):
    env = CoEnv(cfg, n_veh, seed)
    agent.env = env
    horizon = horizon or len(sched)
    drng = np.random.default_rng(seed + 7)
    backlog, energy, rewards, objective = [], [], [], []
    stores = 0

    def handle(done):
        nonlocal stores
        for task in done:
            task_consensus_stats(task)
            objective.append(co_reward(task, cfg))       # same P2 objective for every method
            r = agent.reward(task); rewards.append(r)
            if train and task.act is not None:
                agent.store(task, r); stores += 1
                if stores % getattr(agent, "train_every", cfg["train_every"]) == 0:
                    agent.train()

    for s in range(horizon):
        env.begin_slot()
        for vid, job in sched[s]:
            task = Task(vid, env.t, job)
            task.deadline = float(drng.uniform(*cfg["dead_range"]))
            if hasattr(agent, "act_joint"):
                x, y, f, C, task.act = agent.act_joint(env, task, explore=train)
            else:
                build = env.build_graph_obs if getattr(agent, "graph_obs", False) else env.build_obs
                task.obs = build(vid, task, agent.obs_mode)
                x, y, f, task.act = agent.act(task.obs, task, explore=train)
                C = None
            env.submit(task, x, y, f, C)
        handle(env.run_slot())
        if trace:
            backlog.append(sum(v.Q for v in env.veh))
            energy.append(sum(v.total_energy for v in env.veh))
    extra = 0
    while not env.all_done() and extra < cfg["drain_max"]:
        env.begin_slot(); handle(env.run_slot()); extra += 1
    if train and hasattr(agent, "end_episode"):
        agent.end_episode()

    fin = [t for t in env.tasks if t.complete is not None]
    delays = [task_delay(t) for t in fin] + [env.t - t.arr for t in env.tasks if t.complete is None]
    makespan = max([t.complete for t in fin] + [env.t if len(fin) < len(env.tasks) else 0, 1.0])
    tot = sum(t.info["ex"]["total"] for t in fin if "ex" in t.info and t.info["ex"])
    exp = sum(t.info["ex"]["exposed"] for t in fin if "ex" in t.info and t.info["ex"])
    ncons = sum(t.info["ex"]["n"] for t in fin if "ex" in t.info and t.info["ex"])
    return dict(
        completion=float(np.mean(delays)) if delays else 0.0,
        throughput=float(np.mean([t.size / task_delay(t) for t in fin])) if fin else 0.0,
        sys_throughput=float(sum(t.size for t in fin) / makespan),
        reward=float(np.mean(rewards)) if rewards else 0.0,
        objective=float(np.mean(objective)) if objective else 0.0,
        energy=float(sum(v.total_energy for v in env.veh)),
        offload=float(np.mean([np.mean(t.info["x"]) for t in env.tasks])) if env.tasks else 0.0,
        tampered=int(sum(r.fail for r in env.res)),                 # caught by the committee
        tamper_committed=int(env.tamper_committed),                  # accepted although wrong
        deadline_miss=float(np.mean([task_delay(t) > t.deadline for t in fin])) if fin else 0.0,
        cons_ms=1e3 * tot / max(ncons, 1),
        hidden=float(1.0 - exp / tot) if tot > 0 else 1.0,
        self_val=float(env.n_selfval / max(env.n_off, 1)),
        val_util=float(np.mean([t.info["u_val"] for t in env.tasks])) if env.tasks else 0.0,
        liveness=int(env.liveness_fail),
        forge_attempts=int(env.forge_attempts), forge_accepted=int(env.forge_accepted),
        mitm_detected=int(env.mitm_detected), sig_ops=int(env.n_sign + env.n_verify),
        kem_handshakes=int(env.n_kem),
        n_tasks=len(env.tasks), backlog=backlog, energy_t=energy)


METRICS = ("forge_attempts", "forge_accepted", "mitm_detected", "sig_ops", "kem_handshakes",
           "completion", "throughput", "sys_throughput", "reward", "objective", "energy", "offload", "tampered",
           "tamper_committed", "deadline_miss", "cons_ms", "hidden", "self_val", "val_util", "liveness")


def co_evaluate(agent, windows, cfg, n_veh, horizon=None, trace=False):
    ms = [run_co_episode(agent, sched, cfg, n_veh, seed=seed, train=False, horizon=horizon, trace=trace)
          for seed, sched in windows]
    out = {k: float(np.mean([m[k] for m in ms])) for k in METRICS}
    out["completion_std"] = float(np.std([m["completion"] for m in ms]))
    out["throughput_std"] = float(np.std([m["throughput"] for m in ms]))
    if trace:
        out["backlog"] = np.mean([m["backlog"] for m in ms], 0).tolist()
        out["energy_t"] = np.mean([m["energy_t"] for m in ms], 0).tolist()
    return out


# %% [markdown]
# ## 5. Graph built at every DAG arrival (section 7.1)

# %%
FT_C, FS_C, FV_C = 9, 9, 6          # subtask, RES and vehicle node features
FE_OPT = 5                          # option edge features in the offloading head (3 static + 2 EFT)
FV_VAL = 4                          # extra validator-head features per RES


def lowest(values, n, allowed=None):
    idx = [k for k in np.argsort(values, kind="stable") if allowed is None or allowed[k]]
    return sorted(int(k) for k in idx[:n])


def co_graph(env, task, n1_off=False):
    """Heterogeneous graph of one DAG arrival: subtask nodes, RES nodes, the vehicle node and
    the depends_on / can_run_on / local_on / backhaul edges, plus the numbers the heads use."""
    c, v = env.cfg, env.veh[task.vid]
    K, M = len(task.subs), env.M
    near, _ = env.nearest(v)
    rate = max(env.rate(v), 1.0)
    preds = [[p.k for p in s.preds] for s in task.subs]
    b = np.array([s.b for s in task.subs]); d = np.array([s.d for s in task.subs])
    F, cores = env.Fk, env.cores
    backlog = env.backlog()
    u_pre = env.u_pre()
    hop = np.abs(near - np.arange(M))[None, :] * (c["backhaul_hop_s"] + d[:, None] / c["backhaul_bps"])
    t_up = c["overhead"] * d[:, None] / rate + hop                         # [K, M]
    t_loc = b / c["f_local_max"]
    t_edge = t_up + b[:, None] / F[None, :]
    tcons = env.tcons_est(lowest(u_pre, env.n_cmt), u_pre)
    dur = 0.5 * t_loc + 0.5 * (t_edge.mean(1) + tcons)
    rank, slack, _, _, _ = rank_slack(preds, dur)
    depth = np.zeros(K)
    for k in topo_order(preds):
        depth[k] = 1 + max((depth[p] for p in preds[k]), default=-1)
    ch = children_of(preds)
    xt = np.stack([b / 1e9, d / 1e6, rank / 5.0, slack / 5.0, depth / 8.0,
                   np.array([float(not ch[k]) for k in range(K)]),
                   np.full(K, task.deadline / 6.0),
                   np.array([len(p) for p in preds]) / 4.0,
                   np.array([len(x) for x in ch]) / 4.0], 1).astype(np.float32)
    if n1_off:                                    # ablation A1: no critical-path features
        xt[:, 2:4] = 0.0
    A = np.zeros((K, K), np.float32); Ed = np.zeros((K, K, 1), np.float32)
    for k, ps in enumerate(preds):
        for p in ps:
            A[k, p] = 1.0; Ed[k, p, 0] = d[p] / 1e6
    xs = np.stack([F / 5e9, backlog / (F * cores) / 10.0,
                   [len(r.q[0]) / 10.0 for r in env.res], [len(r.q[1]) / 10.0 for r in env.res],
                   [len(r.q[2]) / 10.0 for r in env.res], u_pre, env.served / 5.0,
                   env.ver_slot / (F * cores),
                   [r.trust() for r in env.res]],      # on-chain verification record (TaBRL)
                  1).astype(np.float32)
    zone = c["road_len"] / M
    edge = (near + (1 if v.v > 0 else 0)) * zone
    dwell = abs(edge - v.x) / max(abs(v.v), 1e-3)
    xv = np.array([c["f_local_max"] / 2.5e9, v.Q / 1e7, v.E / 10.0,
                   sum(s.rem for s in v.local) / c["f_local_max"] / 10.0, abs(v.v) / 25.0,
                   min(dwell, 60.0) / 10.0], np.float32)
    Ers = np.stack([t_up, hop / 0.1], -1).astype(np.float32)
    Etv = (t_loc / 2.0)[:, None].astype(np.float32)
    Ess = np.stack([env.rtt / 0.01, np.ones((M, M))], -1).astype(np.float32)
    # option edge features for the offloading head: vehicle first, then RES 0..M-1
    opt = np.zeros((K, M + 1, 3), np.float32)
    opt[:, 0, 0] = t_loc / 2.0; opt[:, 0, 2] = 1.0
    opt[:, 1:, 0] = t_up; opt[:, 1:, 1] = b[:, None] / F[None, :]
    return dict(xt=xt, A=A, Ed=Ed, xs=xs, xv=xv, Ers=Ers, Etv=Etv, Ess=Ess, opt=opt,
                order=np.argsort(-rank, kind="stable"), b=b, backlog=backlog, cap=env.capacity(),
                qmask=(backlog > c["q_max"]), u_pre=u_pre, rank=rank, slack=slack, t_loc=t_loc,
                t_up=t_up, preds=preds)


def batch_cographs(gs):
    B = len(gs)
    K = max(g["xt"].shape[0] for g in gs); M = gs[0]["xs"].shape[0]
    z = lambda *s: np.zeros(s, np.float32)
    out = dict(xt=z(B, K, FT_C), mt=z(B, K), A=z(B, K, K), Ed=z(B, K, K, 1), xs=z(B, M, FS_C),
               ms=np.ones((B, M), np.float32), xv=z(B, FV_C), Ers=z(B, K, M, 2), Etv=z(B, K, 1),
               Ess=z(B, M, M, 2))
    for i, g in enumerate(gs):
        k = g["xt"].shape[0]
        out["xt"][i, :k] = g["xt"]; out["mt"][i, :k] = 1; out["A"][i, :k, :k] = g["A"]
        out["Ed"][i, :k, :k] = g["Ed"]; out["xs"][i] = g["xs"]; out["xv"][i] = g["xv"]
        out["Ers"][i, :k] = g["Ers"]; out["Etv"][i, :k] = g["Etv"]; out["Ess"][i] = g["Ess"]
    return {k: torch.as_tensor(v, device=DEVICE) for k, v in out.items()}


# %% [markdown]
# ## 6. Heterogeneous graph-attention encoder, heads and critic (section 7.2 - 7.5)

# %%
class GATv2Rel(nn.Module):
    """Dense GATv2 attention for one relation (edge type) with edge features."""
    def __init__(self, d, heads, fe):
        super().__init__()
        self.h, self.dh = heads, d // heads
        self.ws, self.wt, self.we = nn.Linear(d, d), nn.Linear(d, d, bias=False), nn.Linear(fe, d, bias=False)
        self.att = nn.Parameter(torch.randn(heads, self.dh) * 0.1)

    def forward(self, ht, hs, e, mask):
        B, Nt, Ns = mask.shape
        zs = self.ws(hs).view(B, 1, Ns, self.h, self.dh)
        zt = self.wt(ht).view(B, Nt, 1, self.h, self.dh)
        ze = self.we(e).view(B, Nt, Ns, self.h, self.dh)
        msg = zs + ze
        sc = (F.leaky_relu(msg + zt, 0.2) * self.att).sum(-1)
        sc = sc.masked_fill(mask.unsqueeze(-1) == 0, -1e9)
        a = torch.softmax(sc, 2) * mask.unsqueeze(-1)
        return (a.unsqueeze(-1) * msg).sum(2).reshape(B, Nt, -1)


class HeteroLayer(nn.Module):
    REL = {"dep": ("t", "t", 1), "rdep": ("t", "t", 1), "run": ("t", "s", 2), "run_r": ("s", "t", 2),
           "loc": ("t", "v", 1), "loc_r": ("v", "t", 1), "bh": ("s", "s", 2)}

    def __init__(self, d, heads):
        super().__init__()
        self.rel = nn.ModuleDict({k: GATv2Rel(d, heads, fe) for k, (_, _, fe) in self.REL.items()})
        self.norm = nn.ModuleDict({k: nn.LayerNorm(d) for k in "tsv"})

    def forward(self, h, g):
        mt, ms = g["mt"], g["ms"]
        B, K = mt.shape; M = ms.shape[1]
        mts = mt.unsqueeze(2) * ms.unsqueeze(1)
        mss = ms.unsqueeze(2) * ms.unsqueeze(1) * (1 - torch.eye(M, device=ms.device))
        upd = dict(
            t=self.rel["dep"](h["t"], h["t"], g["Ed"], g["A"])
            + self.rel["rdep"](h["t"], h["t"], g["Ed"].transpose(1, 2), g["A"].transpose(1, 2))
            + self.rel["run"](h["t"], h["s"], g["Ers"], mts)
            + self.rel["loc"](h["t"], h["v"], g["Etv"].unsqueeze(-1), mt.unsqueeze(2)),
            s=self.rel["run_r"](h["s"], h["t"], g["Ers"].transpose(1, 2), mts.transpose(1, 2))
            + self.rel["bh"](h["s"], h["s"], g["Ess"], mss),
            v=self.rel["loc_r"](h["v"], h["t"], g["Etv"].transpose(1, 2).unsqueeze(-1), mt.unsqueeze(1)))
        out = {k: self.norm[k](h[k] + F.relu(upd[k])) for k in "tsv"}
        out["t"] = out["t"] * mt.unsqueeze(-1); out["s"] = out["s"] * ms.unsqueeze(-1)
        return out


class CoNet(nn.Module):
    """Encoder (heterogeneous GNN, or a flat MLP for MLP-Joint / ablation A5), offloading head,
    validator head, CPU-frequency head and critic."""
    def __init__(self, cfg, encoder="gnn"):
        super().__init__()
        d, L, H = cfg["co_hidden"], cfg["co_layers"], cfg["co_heads"]
        self.encoder, self.d = encoder, d
        self.inp = nn.ModuleDict(dict(t=nn.Linear(FT_C, d), s=nn.Linear(FS_C, d), v=nn.Linear(FV_C, d)))
        if encoder == "gnn":
            self.layers = nn.ModuleList([HeteroLayer(d, H) for _ in range(L)])
        else:
            self.k_pad, self.m_fix = cfg["k_max"], cfg["n_res"]
            flat = self.k_pad * FT_C + self.m_fix * FS_C + FV_C
            self.ctx = nn.Sequential(nn.Linear(flat, 2 * d), nn.ReLU(), nn.Linear(2 * d, d), nn.ReLU())
            self.mix = nn.ModuleDict({k: nn.Sequential(nn.Linear(2 * d, d), nn.ReLU(), nn.Linear(d, d))
                                      for k in "tsv"})
        g_dim = 5 * d
        self.off = nn.Sequential(nn.Linear(2 * d + FE_OPT + 2, d), nn.ReLU(), nn.Linear(d, 1))
        self.val = nn.Sequential(nn.Linear(d + FV_VAL + g_dim, d), nn.ReLU(), nn.Linear(d, 1))
        self.freq = nn.Sequential(nn.Linear(g_dim, d), nn.ReLU(), nn.Linear(d, len(cfg["freq_levels"])))
        self.critic = nn.Sequential(nn.Linear(g_dim, d), nn.ReLU(), nn.Linear(d, d), nn.ReLU(), nn.Linear(d, 1))

    def encode(self, g):
        h = dict(t=F.relu(self.inp["t"](g["xt"])) * g["mt"].unsqueeze(-1),
                 s=F.relu(self.inp["s"](g["xs"])), v=F.relu(self.inp["v"](g["xv"])).unsqueeze(1))
        if self.encoder == "gnn":
            for layer in self.layers:
                h = layer(h, g)
        else:
            B, K = g["mt"].shape
            xt = F.pad(g["xt"], (0, 0, 0, self.k_pad - K))
            flat = torch.cat([xt.reshape(B, -1), g["xs"].reshape(B, -1), g["xv"]], -1)
            ctx = self.ctx(flat)
            h = {k: self.mix[k](torch.cat([h[k], ctx.unsqueeze(1).expand_as(h[k])], -1)) for k in "tsv"}
            h["t"] = h["t"] * g["mt"].unsqueeze(-1)
        gl = torch.cat([masked_mean(h["t"], g["mt"]), masked_max(h["t"], g["mt"]),
                        masked_mean(h["s"], g["ms"]), masked_max(h["s"], g["ms"]), h["v"][:, 0]], -1)
        return h, gl

    def off_logits(self, hv, ho, e, dl, used):
        """hv [..., d] subtask embedding, ho [..., O, d] option embeddings (vehicle + RESs)."""
        x = torch.cat([hv.unsqueeze(-2).expand_as(ho), ho, e, dl.unsqueeze(-1), used.unsqueeze(-1)], -1)
        return self.off(x).squeeze(-1)

    def val_logits(self, hs, vf, gl):
        return self.val(torch.cat([hs, vf, gl.unsqueeze(1).expand(-1, hs.shape[1], -1)], -1)).squeeze(-1)


def seq_logp(scores, mask, seq):
    """Exact log-probability of an ordered committee drawn by sequential softmax without
    replacement (section 6.3). scores/mask [B, M], seq [B, n]."""
    avail = mask.clone()
    lp = torch.zeros(scores.shape[0], device=scores.device)
    ent = None
    for m in range(seq.shape[1]):
        logit = scores.masked_fill(avail == 0, -1e9)
        ls = torch.log_softmax(logit, -1)
        if ent is None:
            ent = -(ls.exp() * ls * avail).sum(-1)
        lp = lp + ls.gather(1, seq[:, m:m + 1]).squeeze(1)
        avail = avail.scatter(1, seq[:, m:m + 1], 0.0)
    return lp, ent


# %% [markdown]
# ## 7. CoSched-GNN agent: masked joint action + PPO (section 6, 7.6 - 7.8)

# %%
class CoSchedGNN(BaseAgent):
    """One action per DAG: an offloading option for every subtask (decided in rank order,
    with load/used features updated after each choice), then a PBFT committee of n RESs,
    then the local CPU frequency. Variants (ablations / closest-work baselines):
      n1_off      A1  reward = sum of own time + full consensus, no rank/slack features
      val_feat    A2  'pre': the validator head sees load before offloading
      exec_mask   A3  False: no executor exclusion (and no committee-feasibility mask)
      val_mode    A4  'postload_rule': learned offloading, rule-based post-load committee
                      'preload_rule': HGAT-style (n lowest pre-offloading load, no mask)
                      'reliability_rule': FE-CTDE-style (n most reliable RESs, Feng et al. 2026)
      encoder     A5  'mlp': flat MLP instead of the graph encoder (MLP-Joint)"""
    name, obs_mode = "CoSched-GNN", "reported"

    def __init__(self, env, cfg, n1_off=False, val_feat="post", exec_mask=True, val_mode="learn",
                 encoder="gnn", name=None):
        BaseAgent.__init__(self, env, cfg)
        self.name = name or self.name
        self.n1_off, self.val_feat, self.exec_mask = n1_off, val_feat, exec_mask
        self.val_mode, self.encoder = val_mode, encoder
        self.net = CoNet(cfg, encoder).to(DEVICE)
        self.opt = torch.optim.Adam(self.net.parameters(), lr=cfg["ppo_lr"])
        self.rng = np.random.default_rng(cfg["seed"] + 11)
        self.buf = []
        self.frac, self.train_every = 0.0, 1
        self.n_updates, self.epoch, self.expert = 0, 0, None

    def set_progress(self, frac):
        self.frac = frac
        for gr in self.opt.param_groups:
            gr["lr"] = self.cfg["ppo_lr"] * max(0.05, 1.0 - frac)
        self.eps = 0.0

    def in_warmup(self):
        return self.epoch < self.cfg["bc_epochs"]

    @torch.no_grad()
    def act_joint(self, env, task, explore):
        c = self.cfg
        G = co_graph(env, task, self.n1_off)
        forced = None
        if explore and self.in_warmup():          # warm start: imitate the CP-Greedy rule
            if self.expert is None:
                self.expert = CPGreedy(env, c)
            xe, ye = self.expert.plan(env, task)
            forced = dict(off={v: (int(ye[v]) + 1 if xe[v] else 0) for v in range(len(task.subs))},
                          C=self.expert.committee(env, task, xe, ye))
        K, M, n = len(task.subs), env.M, env.n_cmt
        gt = batch_cographs([G])
        h, gl = self.net.encode(gt)
        ho = torch.cat([h["v"], h["s"]], 1)[0]                                  # [1+M, d]
        cap = np.concatenate([[c["f_local_max"] * c["load_horizon"]], G["cap"]])
        assigned = np.zeros(M + 1); used = np.zeros(M + 1)
        limit = M - n if self.exec_mask else M
        steps_e, steps_dl, steps_used, steps_mask, acts = [], [], [], [], []
        logp = 0.0
        eft = EFT(env, task, G)
        for v in G["order"]:
            fin = eft.options(v)
            cons = eft.tcons if self.n1_off else max(0.0, eft.tcons - G["slack"][v])
            ev = np.concatenate([G["opt"][v], np.stack([(fin - env.t) / 2.0,
                                                        np.r_[0.0, np.full(M, cons / 0.5)]], 1)], 1).astype(np.float32)
            mask = np.ones(M + 1, np.float32)
            mask[1:][G["qmask"]] = 0.0
            if used[1:].sum() >= limit:
                mask[1:][used[1:] == 0] = 0.0
            dl = (assigned / cap).astype(np.float32)
            e = torch.as_tensor(ev, device=DEVICE)
            lg = self.net.off_logits(h["t"][0, v], ho, e, torch.as_tensor(dl, device=DEVICE),
                                     torch.as_tensor(used.astype(np.float32), device=DEVICE))
            lg = lg.masked_fill(torch.as_tensor(mask, device=DEVICE) == 0, -1e9)
            ls = torch.log_softmax(lg, -1)
            if forced is not None and mask[forced["off"][v]] > 0:
                a = forced["off"][v]
            else:
                a = int(torch.multinomial(ls.exp(), 1)) if explore else int(ls.argmax())
            logp += float(ls[a])
            steps_e.append(ev); steps_dl.append(dl); steps_used.append(used.astype(np.float32).copy())
            steps_mask.append(mask); acts.append(a)
            assigned[a] += G["b"][v]; used[a] = 1.0
            eft.commit(v, a, fin)
        x = np.zeros(K, int); y = np.zeros(K, int)
        for v, a in zip(G["order"], acts):
            x[v], y[v] = int(a > 0), max(a - 1, 0)
        new = np.zeros(M)
        for v in range(K):
            if x[v]:
                new[y[v]] += G["b"][v]
        executors = set(int(y[v]) for v in range(K) if x[v])
        u_post = env.u_post(new)
        vmask = np.ones(M, np.float32)
        if self.exec_mask:
            vmask[list(executors)] = 0.0
        u_feat = u_post if self.val_feat == "post" else G["u_pre"]
        mean_rtt = np.array([env.rtt[k][vmask > 0].mean() for k in range(M)])
        vf = np.stack([u_feat, env.Fk / 5e9, mean_rtt / 0.01, env.served / 5.0], 1).astype(np.float32)
        seq = None
        if self.val_mode == "learn":
            s = self.net.val_logits(h["s"], torch.as_tensor(vf, device=DEVICE)[None], gl)[0]
            avail = torch.tensor(vmask, device=DEVICE)          # copy: vmask is stored
            seq = []
            want = None
            if forced is not None and all(vmask[k] > 0 for k in forced["C"]):
                want = sorted(forced["C"], key=lambda k: u_post[k])
            for i in range(n):
                ls = torch.log_softmax(s.masked_fill(avail == 0, -1e9), -1)
                if want is not None:
                    k = want[i]
                else:
                    k = int(torch.multinomial(ls.exp(), 1)) if explore else int(ls.argmax())
                logp += float(ls[k]); seq.append(k); avail[k] = 0.0
            C = sorted(seq)
        elif self.val_mode == "postload_rule":
            C = lowest(u_post, n, vmask > 0)
        elif self.val_mode == "reliability_rule":                  # FE-CTDE-style
            # dynamic BFT committee of the n most reliable RESs (verification record),
            # ties broken by computing capability; load and the executors are not considered
            rel = np.array([r.trust() for r in env.res]) + 1e-3 * env.Fk / 5e9
            C = lowest(-rel, n)
        else:                                                      # preload_rule (HGAT-style)
            C = lowest(G["u_pre"], n)
        lf = torch.log_softmax(self.net.freq(gl)[0], -1)
        if forced is not None:
            fa = len(c["freq_levels"]) - 1                       # the rule runs at f_max
        else:
            fa = int(torch.multinomial(lf.exp(), 1)) if explore else int(lf.argmax())
        logp += float(lf[fa])
        value = float(self.net.critic(gl)[0, 0])
        rec = None
        if explore:
            G_store = {k: G[k] for k in ("xt", "A", "Ed", "xs", "xv", "Ers", "Etv", "Ess")}
            rec = dict(G=G_store, order=G["order"].copy(), e=np.stack(steps_e), dl=np.stack(steps_dl),
                       used=np.stack(steps_used), mask=np.stack(steps_mask), acts=np.array(acts),
                       vf=vf, vmask=vmask, seq=None if seq is None else np.array(seq), fa=fa,
                       logp=logp, value=value, r=None, term=False, bc=forced is not None)
            self.buf.append(rec)
        return x.tolist(), y.tolist(), c["freq_levels"][fa], C, rec

    def reward(self, task):
        return co_reward(task, self.cfg, self.n1_off)

    def store(self, task, r):
        task.act["r"] = r

    def end_episode(self):
        for rec in self.buf:
            if rec["r"] is None:                       # unfinished at the end of the drain
                rec["r"] = -self.cfg["lam_dl"]
        if self.buf:
            self.buf[-1]["term"] = True
        if self.buf and self.buf[0].get("bc"):
            self.bc_update(self.buf); self.buf = []
        self.train()

    def bc_update(self, recs):
        """Warm start (cfg bc_epochs > 0): maximise the log-probability of the rule's joint
        action and fit the critic to the observed rewards."""
        c = self.cfg
        rew = np.array([r["r"] for r in recs], np.float32)
        for _ in range(c["ppo_epochs"]):
            idx = np.random.permutation(len(recs))
            for s0 in range(0, len(recs), c["ppo_minibatch"]):
                mb = idx[s0:s0 + c["ppo_minibatch"]]
                logp, _, val = self.evaluate_batch([recs[i] for i in mb])
                vl = F.mse_loss(val, torch.as_tensor(rew[mb], device=DEVICE))
                loss = -logp.mean() + c["ppo_vf"] * vl
                self.opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), c["ppo_grad_clip"])
                self.opt.step()

    def train(self):
        R = self.cfg["ppo_rollout"]
        if self.in_warmup():
            return
        if len(self.buf) < R or any(rec["r"] is None for rec in self.buf[:R]):
            return
        batch, self.buf = self.buf[:R], self.buf[R:]
        boot = self.buf[0]["value"] if self.buf and not batch[-1]["term"] else 0.0
        self.ppo_update(batch, boot)

    def evaluate_batch(self, recs):
        g = batch_cographs([r["G"] for r in recs])
        h, gl = self.net.encode(g)
        B, K = g["mt"].shape; M = g["ms"].shape[1]
        pad = lambda key, shape, dt=np.float32: np.stack(
            [np.pad(r[key], [(0, K - len(r[key]))] + [(0, 0)] * (len(shape) - 1)) for r in recs]).astype(dt)
        t = lambda z: torch.as_tensor(z, device=DEVICE)
        order = t(pad("order", (K,), np.int64)); acts = t(pad("acts", (K,), np.int64))
        sv = torch.zeros(B, K, device=DEVICE)
        for i, r in enumerate(recs):
            sv[i, :len(r["acts"])] = 1.0
        e = t(pad("e", (K, M + 1, FE_OPT))); dl = t(pad("dl", (K, M + 1)))
        used = t(pad("used", (K, M + 1))); mask = t(pad("mask", (K, M + 1)))
        mask = mask + (1 - sv).unsqueeze(-1)            # padded steps: anything allowed (ignored)
        hv = h["t"].gather(1, order.unsqueeze(-1).expand(-1, -1, self.net.d))
        ho = torch.cat([h["v"], h["s"]], 1).unsqueeze(1).expand(-1, K, -1, -1)
        lg = self.net.off_logits(hv, ho, e, dl, used).masked_fill(mask == 0, -1e9)
        ls = torch.log_softmax(lg, -1)
        logp = (ls.gather(2, acts.unsqueeze(-1)).squeeze(-1) * sv).sum(1)
        ent = (-(ls.exp() * ls).sum(-1) * sv).sum(1)
        if self.val_mode == "learn":
            vf = t(np.stack([r["vf"] for r in recs])); vmask = t(np.stack([r["vmask"] for r in recs]))
            seq = t(np.stack([r["seq"] for r in recs]).astype(np.int64))
            lpv, entv = seq_logp(self.net.val_logits(h["s"], vf, gl), vmask, seq)
            logp, ent = logp + lpv, ent + entv
        lf = torch.log_softmax(self.net.freq(gl), -1)
        fa = t(np.array([r["fa"] for r in recs], np.int64))
        logp = logp + lf.gather(1, fa.unsqueeze(1)).squeeze(1)
        ent = ent - (lf.exp() * lf).sum(-1)
        return logp, ent, self.net.critic(gl).squeeze(-1)

    def ppo_update(self, batch, boot):
        c = self.cfg
        R = len(batch)
        v = np.array([r["value"] for r in batch] + [boot])
        rew = np.array([r["r"] for r in batch]); term = np.array([r["term"] for r in batch], float)
        adv = np.zeros(R); last = 0.0
        for i in reversed(range(R)):
            nv = 0.0 if term[i] else v[i + 1]
            delta = rew[i] + c["ppo_gamma"] * nv - v[i]
            last = delta + c["ppo_gamma"] * c["ppo_lambda"] * (1 - term[i]) * last
            adv[i] = last
        ret = adv + v[:R]
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        old = np.array([r["logp"] for r in batch])
        e0, e1 = c["ppo_ent"]
        ent_coef = e0 + (e1 - e0) * min(1.0, self.frac)
        for _ in range(c["ppo_epochs"]):
            idx = np.random.permutation(R)
            for s0 in range(0, R, c["ppo_minibatch"]):
                mb = idx[s0:s0 + c["ppo_minibatch"]]
                logp, ent, val = self.evaluate_batch([batch[i] for i in mb])
                tt = lambda z: torch.as_tensor(z[mb], dtype=torch.float32, device=DEVICE)
                ratio = torch.exp(logp - tt(old))
                a = tt(adv)
                pl = -torch.min(ratio * a, ratio.clamp(1 - c["ppo_clip"], 1 + c["ppo_clip"]) * a).mean()
                vl = F.mse_loss(val, tt(ret))
                loss = pl + c["ppo_vf"] * vl - ent_coef * ent.mean()
                self.opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), c["ppo_grad_clip"])
                self.opt.step()
        self.n_updates += 1


# %% [markdown]
# ## 8. Heuristic baselines (sections 5.5 and 6.4)

# %%
class EFT:
    """Earliest-finish-time bookkeeping for one DAG decision (list scheduling in rank order).
    Used by the rule baselines to choose and by CoSched-GNN as option features, so the
    network sees at least the information the rules use."""

    def __init__(self, env, task, G):
        c = env.cfg
        self.env, self.task, self.G = env, task, G
        self.t0 = env.t
        self.tcons = env.tcons_est(lowest(G["u_pre"], env.n_cmt), G["u_pre"])
        v = env.veh[task.vid]
        self.cpu = env.t + sum(s.rem for s in v.local) / c["f_local_max"]
        self.res_free = env.t + G["backlog"] / (env.Fk * env.cores)
        self.done = np.zeros(len(task.subs))

    def options(self, k):
        """Estimated finish (before consensus) of subtask k on the vehicle and on every RES."""
        G, env = self.G, self.env
        ready = max([self.t0] + [self.done[p] for p in G["preds"][k]])
        loc = max(ready, self.cpu) + G["t_loc"][k]
        edge = np.maximum(np.maximum(ready, self.t0 + G["t_up"][k]), self.res_free) + G["b"][k] / env.Fk
        return np.concatenate([[loc], edge])

    def commit(self, k, a, fin):
        if a == 0:
            self.cpu = fin[0]; self.done[k] = fin[0]
        else:
            j = a - 1
            self.res_free[j] += self.G["b"][k] / (self.env.Fk[j] * self.env.cores[j])
            self.done[k] = fin[a] + self.tcons


class _Greedy(BaseAgent):
    """List scheduling in upward-rank order: each subtask goes to the option with the earliest
    estimated finish. HEFT counts the full consensus delay of an edge option; CP-Greedy counts
    only the part exceeding the subtask's slack (rule of section 5.2) and keeps enough
    non-executing RESs for the committee."""
    exposed_only = feasible = False

    def __init__(self, env, cfg):
        BaseAgent.__init__(self, env, cfg)
        self.rng = np.random.default_rng(cfg["seed"] + 13)

    def plan(self, env, task):
        G = co_graph(env, task)
        K, M = len(task.subs), env.M
        eft = EFT(env, task, G)
        x = np.zeros(K, int); y = np.zeros(K, int)
        used, limit = set(), (M - env.n_cmt if self.feasible else M)
        for k in G["order"]:
            fin = eft.options(k)
            extra = max(0.0, eft.tcons - G["slack"][k]) if self.exposed_only else eft.tcons
            best, b_cost = 0, fin[0]
            for j in range(M):
                if G["qmask"][j] or (j not in used and len(used) >= limit):
                    continue
                if fin[j + 1] + extra < b_cost:
                    best, b_cost = j + 1, fin[j + 1] + extra
            if best:
                x[k], y[k] = 1, best - 1; used.add(best - 1)
            eft.commit(k, best, fin)
        return x, y

    def act_joint(self, env, task, explore):
        x, y = self.plan(env, task)
        return x.tolist(), y.tolist(), self.cfg["f_local_max"], self.committee(env, task, x, y), None

    def reward(self, task):
        return co_reward(task, self.cfg)

    def store(self, task, r):
        pass


class HEFTRandom(_Greedy):
    name = "HEFT+Random"

    def committee(self, env, task, x, y):
        return sorted(self.rng.choice(env.M, env.n_cmt, replace=False).tolist())


class CPGreedy(_Greedy):
    name, exposed_only, feasible = "CP-Greedy+Post-load", True, True

    def committee(self, env, task, x, y):
        new = np.zeros(env.M); ok = np.ones(env.M, bool)
        for k, s in enumerate(task.subs):
            if x[k]:
                new[y[k]] += s.b; ok[y[k]] = False
        return lowest(env.u_post(new), env.n_cmt, ok)


# %% [markdown]
# ## 9. Training and the method registry (section 10.1 and 10.2)

# %%
CO_ALGOS = {
    "CoSched-GNN": lambda e, c: CoSchedGNN(e, c),
    "TaBRL-GNN": TaBRLGNN, "TaBRL": TaBRL,
    "HGAT-style": lambda e, c: CoSchedGNN(e, c, val_mode="preload_rule", exec_mask=False, name="HGAT-style"),
    "MLP-Joint": lambda e, c: CoSchedGNN(e, c, encoder="mlp", name="MLP-Joint"),
    # Feng et al., IEEE TMC 2026: joint offloading and consensus with CTDE learning (actors
    # shared by all vehicles, centralised critic) and a reliability-based dynamic BFT committee.
    # Re-implemented with a flat actor-critic, the same PPO budget and the same reward.
    "FE-CTDE-style": lambda e, c: CoSchedGNN(e, c, encoder="mlp", val_mode="reliability_rule",
                                             exec_mask=False, name="FE-CTDE-style"),
    "CP-Greedy+Post-load": CPGreedy, "HEFT+Random": HEFTRandom,
    "MA-LHTO": MALHTO, "IOPO": IOPO, "DECO": DECO, "DAG-Local": DAGLocal,
    # ablations of CoSched-GNN (A5 = MLP-Joint, A6 = CP-Greedy+Post-load, i.e. no learning)
    "A1 N1 off": lambda e, c: CoSchedGNN(e, c, n1_off=True, name="A1 N1 off"),
    "A2 pre-load": lambda e, c: CoSchedGNN(e, c, val_feat="pre", name="A2 pre-load"),
    "A3 no exec mask": lambda e, c: CoSchedGNN(e, c, exec_mask=False, name="A3 no exec mask"),
    "A4 two-stage": lambda e, c: CoSchedGNN(e, c, val_mode="postload_rule", name="A4 two-stage"),
}
NO_TRAIN = {"DAG-Local", "HEFT+Random", "CP-Greedy+Post-load"}
GRAPH_METHODS = {"CoSched-GNN", "TaBRL-GNN", "HGAT-style", "CP-Greedy+Post-load", "HEFT+Random",
                 "DAG-Local", "A1 N1 off", "A2 pre-load", "A3 no exec mask", "A4 two-stage"}
# Default comparison for the paper: the base paper (TaBRL, 2026), joint offloading + consensus
# (FE-CTDE-style, Feng et al. IEEE TMC 2026), GNN DAG offloading (HGAT-style, IEEE IoT-J 2025),
# joint offloading + learned committee with a flat network (MLP-Joint, improved-PBFT line of
# work), the classic HEFT list scheduler and the all-local reference used in TaBRL's Fig. 3.
CO_ORDER = ["CoSched-GNN", "TaBRL", "FE-CTDE-style", "HGAT-style", "MLP-Joint", "HEFT+Random",
            "DAG-Local"]
# Also available with --methods (not needed for the paper): TaBRL's own baselines and TaBRL-GNN.
CO_EXTRA = ["TaBRL-GNN", "MA-LHTO", "IOPO", "DECO"]
# CP-Greedy+Post-load is our own rule (CoSched-GNN's ideas without learning), so it is an ablation
# ("A6 no learning"), not a compared method.
ABLATIONS = ["CoSched-GNN", "A1 N1 off", "A2 pre-load", "A3 no exec mask", "A4 two-stage", "MLP-Joint",
             "CP-Greedy+Post-load"]
C_LINES.update({"FE-CTDE-style": "#7B3294", "CoSched-GNN": "#D55E00", "HGAT-style": "#009E73", "MLP-Joint": "#56B4E9",
                "CP-Greedy+Post-load": "#CC79A7", "HEFT+Random": "#999999", "DAG-Local": "#0072B2"})


def co_train_agent(name, jobs, cfg, n_veh):
    set_seed(cfg["seed"])
    agent = CO_ALGOS[name](CoEnv(cfg, n_veh, 0), cfg)
    hist = []
    if name in NO_TRAIN:
        return agent, hist
    E = cfg["train_episodes"]
    if not E:
        raise ValueError("Number of training epochs not set: run with --epochs N")
    rng = np.random.default_rng(cfg["seed"] + 100)
    t0 = time.time()
    for ep in range(E):
        agent.set_progress(ep / E)
        agent.epoch = ep
        sched = make_window(jobs, n_veh, cfg["horizon"], cfg, rng)
        m = run_co_episode(agent, sched, cfg, n_veh, seed=10_000 + ep, train=True)
        hist.append(dict(ep=ep, completion=m["completion"], throughput=m["throughput"], reward=m["reward"]))
        if ep % 10 == 0 or ep == E - 1:
            print(f"  [{name:19s}] epoch {ep + 1:3d}/{E}  T_c={m['completion']:.3f}s  "
                  f"cap={m['throughput'] / 1e3:.0f} kbit/s  r={m['reward']:.3f}  off={m['offload']:.2f}  "
                  f"cons={m['cons_ms']:.0f}ms  hidden={m['hidden']:.0%}  ({time.time() - t0:.0f}s)")
    return agent, hist


# %% [markdown]
# ## 10. Tables and figures (section 10.3 and 10.4)

# %%
def co_table(res):
    rows = []
    for split, d in res.items():
        for a, m in d.items():
            rows.append({"Split": split, "Algorithm": a,
                         "Completion time (s)": round(m["completion"], 3),
                         "Processing capacity (kbit/s)": round(m["throughput"] / 1e3, 1),
                         "System throughput (Mbit/s)": round(m["sys_throughput"] / 1e6, 3),
                         "Energy (J)": round(m["energy"], 1),
                         "P2 objective": round(m["objective"], 4),
                         "Offload ratio": round(m["offload"], 3),
                         "Deadline miss (%)": round(100 * m["deadline_miss"], 2),
                         "Consensus delay (ms)": round(m["cons_ms"], 1),
                         "Hidden consensus (%)": round(100 * m["hidden"], 1),
                         "Self-validation (%)": round(100 * m["self_val"], 1),
                         "Validator load": round(m["val_util"], 3),
                         "Tampered caught": round(m["tampered"], 1),
                         "Tampered committed": round(m["tamper_committed"], 1),
                         "Forgeries attempted": round(m["forge_attempts"], 1),
                         "Forgeries accepted": round(m["forge_accepted"], 1)})
    return pd.DataFrame(rows)


def with_order(algs, fn, *a, **kw):
    saved = list(ORDER)
    ORDER[:] = list(algs)
    try:
        return fn(*a, **kw)
    finally:
        ORDER[:] = saved


def plot_co_fig3(res, algs, path, title="Impact of different algorithms on train and test set"):
    """Fig. 3 layout (training | test) with room for many method names."""
    plt.rcParams.update(plt.rcParamsDefault)
    fig, axes = plt.subplots(1, 2, figsize=(max(14, 1.6 * len(algs)), 5.6))
    for ax, split, name in zip(axes, ("train", "test"), ("Training set", "Test set")):
        b1, b2 = with_order(algs, _bar_panel, ax, res[split], title=name, headroom=(1.12, 1.25))
        ax.set_xticklabels([a for a in algs if a in res[split]], rotation=35, ha="right")
    fig.legend([b1, b2], ["Completion time", "Processing capacity"], loc="lower center",
               ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(title, fontsize=14)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)


# Which published work each compared method stands for (all are re-implemented in this
# simulator, so every metric, including the new ones, is measured under identical settings).
PAPER_REF = {
    "CoSched-GNN": "Proposed",
    "TaBRL": "Yin et al., IEEE Commun. Stand. Mag. 2026 (base paper)",
    "FE-CTDE-style": "Feng et al., IEEE TMC 2026: joint offloading + reliability-based BFT committee",
    "HGAT-style": "Wu et al., IEEE IoT J. 2025: HGNN + DRL dependent-task offloading (no consensus-aware committee)",
    "MLP-Joint": "Joint offloading + committee DRL with a flat network (improved-PBFT / FE-CTDE line of work)",
    "CP-Greedy+Post-load": "Ablation A6: CoSched-GNN's rules without learning (ours)",
    "HEFT+Random": "HEFT, Topcuoglu et al., IEEE TPDS 2002 + random committee",
    "DAG-Local": "All subtasks on the vehicle (reference in TaBRL Fig. 3)",
    "MA-LHTO": "He et al., IEEE TMC 2026",
    "IOPO": "IOPO (TaBRL baseline)", "DECO": "DECO (TaBRL baseline)",
    "TaBRL-GNN": "TaBRL with a GNN actor (thesis variant)",
}
VALIDATOR_RULE = {
    "CoSched-GNN": "learned, post-offloading load, executors excluded",
    "HGAT-style": "lowest load before offloading",
    "MLP-Joint": "learned (flat MLP), executors excluded",
    "FE-CTDE-style": "n most reliable RESs (verification record)",
    "CP-Greedy+Post-load": "lowest post-offloading load, executors excluded",
    "HEFT+Random": "random per DAG",
}


def paper_table(res, algs):
    d = res["test"]
    return pd.DataFrame([{
        "Method": a, "Represents": PAPER_REF.get(a, ""),
        "Validator selection": VALIDATOR_RULE.get(a, "random committee fixed per episode"),
        "Completion time (s)": round(d[a]["completion"], 3),
        "Processing capacity (kbit/s)": round(d[a]["throughput"] / 1e3, 1),
        "Energy (J)": round(d[a]["energy"], 1),
        "P2 objective": round(d[a]["objective"], 4),
        "Deadline miss (%)": round(100 * d[a]["deadline_miss"], 2),
        "Consensus delay (ms)": round(d[a]["cons_ms"], 1),
        "Hidden consensus (%)": round(100 * d[a]["hidden"], 1),
        "Self-validation (%)": round(100 * d[a]["self_val"], 1),
        "Validator load (%)": round(100 * d[a]["val_util"], 2),
        "Tampered caught": round(d[a]["tampered"], 1),
        "Tampered committed": round(d[a]["tamper_committed"], 1),
        "Forgeries accepted": round(d[a]["forge_accepted"], 1)} for a in algs if a in d])


def plot_paper_comparison(res, algs, path):
    """Proposed method against the compared papers' methods on every metric (test set)."""
    rows = {a: res["test"][a] for a in algs if a in res["test"]}
    keys = [("completion", "Mean completion time (s)", 1), ("throughput", "Processing capacity (kbit/s)", 1e-3),
            ("energy", "Vehicle energy (J)", 1), ("objective", "P2 objective per DAG (higher is better)", 1),
            ("hidden", "Hidden consensus fraction (%)", 100), ("self_val", "Self-validation rate (%)", 100),
            ("val_util", "Validator load at selection (%)", 100), ("tampered", "Tampered results caught", 1)]
    fig, axes = plt.subplots(2, 4, figsize=(max(20, 2.6 * len(rows) * 1.5), 8.5))
    for i, (ax, (k, lab, sc)) in enumerate(zip(axes.ravel(), keys)):
        # consensus metrics do not apply to DAG-Local (nothing is offloaded or verified)
        labels = [a for a in rows if i < 4 or a != "DAG-Local"]
        v = [rows[a][k] * sc for a in labels]
        bars = ax.bar(labels, v, 0.6, color=[C_LINES.get(a, C_CAP) for a in labels])
        for b, x in zip(bars, v):
            ax.text(b.get_x() + b.get_width() / 2, x, f"{x:.2f}" if abs(x) < 10 else f"{x:.0f}",
                    ha="center", va="bottom", fontsize=8)
        ax.set_ylabel(lab); ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.suptitle("CoSched-GNN vs. compared methods (test set, same simulator and seeds)", fontsize=13)
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_consensus(res, algs, path):
    d = res["test"]
    algs = [a for a in algs if a in d and a != "DAG-Local"]
    panels = [("cons_ms", "Mean consensus delay per offloaded subtask (ms)", 1),
              ("hidden", "Hidden consensus fraction (%)", 100),
              ("self_val", "Self-validation rate (%)", 100),
              ("deadline_miss", "Deadline miss rate (%)", 100)]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for ax, (k, lab, sc) in zip(axes.ravel(), panels):
        v = [d[a][k] * sc for a in algs]
        bars = ax.bar(algs, v, 0.6, color=[C_LINES.get(a, C_CAP) for a in algs])
        for b, x in zip(bars, v):
            ax.text(b.get_x() + b.get_width() / 2, x, f"{x:.1f}", ha="center", va="bottom", fontsize=8)
        ax.set_ylabel(lab); ax.tick_params(axis="x", rotation=30)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.suptitle("Consensus and security metrics (test set)", fontsize=13)
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_metric_bars(rows, path, title, keys):
    """rows: {label: metrics}; keys: [(metric, axis label, scale)]."""
    labels = list(rows)
    fig, axes = plt.subplots(1, len(keys), figsize=(5.2 * len(keys), 4.6))
    for ax, (k, lab, sc) in zip(np.atleast_1d(axes), keys):
        v = [rows[a][k] * sc for a in labels]
        bars = ax.bar(labels, v, 0.6, color=[C_LINES.get(a, C_CAP) for a in labels])
        for b, x in zip(bars, v):
            ax.text(b.get_x() + b.get_width() / 2, x, f"{x:.2f}" if abs(x) < 10 else f"{x:.0f}",
                    ha="center", va="bottom", fontsize=8)
        ax.set_ylabel(lab); ax.tick_params(axis="x", rotation=30)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.suptitle(title, fontsize=13); fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_pqc(pq, methods, schemes, path):
    """pq: {method: {scheme: metrics}} under the quantum adversary."""
    panels = [("completion", "Mean completion time (s)"), ("cons_ms", "Consensus delay per subtask (ms)"),
              ("forge_accepted", "Forged certificates accepted"),
              ("tamper_committed", "Tampered results committed")]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))
    w = 0.8 / len(methods)
    for i, a in enumerate(methods):
        for ax, (k, _) in zip(axes, panels):
            ax.bar(np.arange(len(schemes)) + i * w, [pq[a][s][k] for s in schemes], w, label=a,
                   color=C_LINES.get(a))
    for ax, (_, lab) in zip(axes, panels):
        ax.set_xticks(np.arange(len(schemes)) + w * (len(methods) - 1) / 2)
        ax.set_xticklabels([SIG[sc][0] for sc in schemes], rotation=15); ax.set_ylabel(lab)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axes[0].legend(frameon=False, fontsize=8)
    fig.suptitle("ECDSA vs ML-DSA in PBFT against a quantum adversary (real signatures)")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


def plot_lines(xs, data, xlabel, path, title, keys):
    fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 4.5))
    for ax, (k, lab, sc) in zip(np.atleast_1d(axes), keys):
        for a, pts in data.items():
            xv = [x for x in xs if x in pts]
            ax.plot(xv, [pts[x][k] * sc for x in xv], marker="o", lw=2, label=a, color=C_LINES.get(a))
        ax.set_xlabel(xlabel); ax.set_ylabel(lab); ax.set_xticks(xs); ax.grid(alpha=0.3)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8)
    fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


GANTT_DAG = dict(   # 8-subtask driving example of the paper (Fig. 4/5), also used by the unit tests
    names="ABCDEFGH",
    preds=[[], [0], [0], [1, 2], [0], [0], [3, 4], [5, 6]],
    place=[0, 1, 1, 0, 1, 1, 0, 0],
    t_loc=[0.2, 1.5, 1.2, 0.6, 1.2, 0.8, 0.4, 0.2],
    t_edge=[0.2, 1.0, 0.8, 0.4, 0.8, 0.4, 0.3, 0.2],
    t_cons=[0.0, 0.5, 0.5, 0.0, 0.5, 0.5, 0.0, 0.0])


def plot_gantt(path):
    g = GANTT_DAG
    mk, st, fi, dn = static_schedule(g["preds"], g["place"], g["t_loc"], g["t_edge"], g["t_cons"])
    dur = [g["t_loc"][v] if g["place"][v] == 0 else g["t_edge"][v] + g["t_cons"][v] for v in range(8)]
    _, slack, _, _, _ = rank_slack(g["preds"], dur)
    fig, ax = plt.subplots(figsize=(9, 3.8))
    for v, nm in enumerate(g["names"]):
        row = 7 - v
        ax.barh(row, fi[v] - st[v], left=st[v], color="#0072B2" if g["place"][v] == 0 else "#E69F00")
        if g["place"][v]:
            ax.barh(row, dn[v] - fi[v], left=fi[v], color="#999999", hatch="//")
        ax.text(dn[v] + 0.03, row, f"{nm}  slack {max(slack[v], 0.0):.1f}s", va="center", fontsize=8)
    ax.set_yticks(range(8)); ax.set_yticklabels(list(reversed(g["names"])))
    ax.axvline(mk, color="k", ls="--", lw=1); ax.set_xlabel(f"Time (s)  -  makespan {mk:.1f} s")
    from matplotlib.patches import Patch
    ax.legend([Patch(color="#0072B2"), Patch(color="#E69F00"), Patch(facecolor="#999999", hatch="//")],
              ["vehicle CPU", "edge compute", "PBFT consensus"], loc="upper right", fontsize=8, frameon=False)
    ax.set_title("Consensus hidden in DAG slack: only the critical path pays it")
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)


# %% [markdown]
# ## 11. Experiments

# %%
def train_and_eval(names, cfg, train_jobs, wins, agents, hists, res):
    N = cfg["n_veh"]
    for name in names:
        if name in agents:
            continue
        print(f"\n=== {name} ===")
        agents[name], hists[name] = co_train_agent(name, train_jobs, cfg, N)
        for split, win in wins.items():
            res.setdefault(split, {})[name] = m = co_evaluate(agents[name], win, cfg, N)
            print(f"  eval {split:5s}: completion {m['completion']:.3f}s  capacity {m['throughput'] / 1e3:.0f} "
                  f"kbit/s  energy {m['energy']:.0f} J  offload {m['offload']:.2f}  cons {m['cons_ms']:.0f} ms  "
                  f"hidden {m['hidden']:.0%}  self-val {m['self_val']:.0%}  miss {m['deadline_miss']:.1%}  "
                  f"tampered caught/committed {m['tampered']:.0f}/{m['tamper_committed']:.0f}")


def co_main(cfg, experiments=("main", "ablation", "pqc", "hidden", "scaling"), methods=None):
    out = cfg["out_dir"]; os.makedirs(out, exist_ok=True)
    set_seed(cfg["seed"])
    print(f"Device: {DEVICE}")
    print_settings(cfg)
    print(f"CoSched settings: M={cfg['n_res']} RESs, f={cfg['pbft_f']} (n={3 * cfg['pbft_f'] + 1}, "
          f"q={2 * cfg['pbft_f'] + 1}), signature {cfg['sig_scheme']}, c_cons={c_cons_cycles(cfg) / 1e6:.2f} "
          f"Mcycles, PPO rollout {cfg['ppo_rollout']}")
    pc = pq_costs(cfg)
    print(f"Cryptography: {cfg['crypto']} | {pc['alg']} ({pc['level']}) via {pc['backend']}, signature "
          f"{pc['sig_bytes']} B, measured sign {pc['sign_s'] * 1e3:.3f} ms / verify {pc['verify_s'] * 1e3:.3f} ms; "
          f"sessions ML-KEM-768 + AES-256-GCM; hashes SHA3-256")
    train_jobs, test_jobs = load_dag_jobs(cfg)
    N = cfg["n_veh"]
    wins = dict(train=fixed_windows(train_jobs, N, cfg["horizon"], cfg, cfg["eval_windows"], seed=1),
                test=fixed_windows(test_jobs, N, cfg["horizon"], cfg, cfg["eval_windows"], seed=2))
    methods = [m for m in (methods or CO_ORDER) if m in CO_ALGOS]
    agents, hists, res, summary = {}, {}, {}, {}

    if "main" in experiments:
        print("\n######## Main comparison ########")
        train_and_eval(methods, cfg, train_jobs, wins, agents, hists, res)
        main_algs = [m for m in methods if m in res["test"]]
        plot_co_fig3(res, main_algs, os.path.join(out, "fig3_completion_capacity.png"))
        plot_consensus(res, main_algs, os.path.join(out, "consensus_security.png"))
        plot_paper_comparison(res, main_algs, os.path.join(out, "paper_comparison.png"))
        pt = paper_table(res, main_algs)
        pt.to_csv(os.path.join(out, "paper_comparison.csv"), index=False)
        print_table("Proposed vs. compared methods (test set)", pt.drop(columns=["Represents", "Validator selection"]))
        if any(hists.values()):
            with_order(main_algs, plot_training, hists, os.path.join(out, "training_curves.png"))
        res4 = {}
        for nv in cfg["fig4_veh"]:
            w4 = fixed_windows(test_jobs, nv, cfg["fig4_horizon"], cfg, cfg["eval_windows"], seed=3 + nv)
            res4[nv] = {a: co_evaluate(agents[a], w4, cfg, nv, trace=True) for a in main_algs if a != "DAG-Local"}
        with_order(main_algs, plot_fig4, res4, os.path.join(out, "fig4_backlog_energy.png"))
        t = co_table({s: {a: res[s][a] for a in main_algs} for s in res})
        t.to_csv(os.path.join(out, "results.csv"), index=False)
        fig4_table(res4).to_csv(os.path.join(out, "fig4_values.csv"), index=False)
        print_table("Main results (train and test)", t)
        summary["main"] = dict(res={s: {a: res[s][a] for a in main_algs} for s in res},
                               fig4={str(k): v for k, v in res4.items()})

    if "ablation" in experiments:
        print("\n######## Ablations A1-A6 ########")
        train_and_eval(ABLATIONS, cfg, train_jobs, {"test": wins["test"]}, agents, hists, res)
        rows = {a: res["test"][a] for a in ABLATIONS}
        plot_metric_bars(rows, os.path.join(out, "ablation.png"), "Ablations (test set)",
                         [("completion", "Mean completion time (s)", 1),
                          ("hidden", "Hidden consensus fraction (%)", 100),
                          ("self_val", "Self-validation rate (%)", 100),
                          ("val_util", "Validator load at selection (%)", 100)])
        t = co_table({"test": rows}).drop(columns="Split")
        t.to_csv(os.path.join(out, "ablation.csv"), index=False)
        print_table("Ablations (test set)", t)
        summary["ablation"] = rows

    pqc_methods = [m for m in ("CoSched-GNN", "TaBRL", "FE-CTDE-style") if m in CO_ALGOS]
    if "pqc" in experiments:
        print("\n######## PQC: classical vs post-quantum signatures under a quantum adversary ########")
        print("  (real signatures; the adversary runs Shor's algorithm, i.e. recovers ECDSA keys)")
        train_and_eval(pqc_methods, cfg, train_jobs, {"test": wins["test"]}, agents, hists, res)
        pq = {a: {} for a in pqc_methods}
        schemes = list(SIG)
        for sch in schemes:
            cs = dict(cfg, sig_scheme=sch, quantum_adversary=True)
            for a in pqc_methods:
                pq[a][sch] = co_evaluate(agents[a], wins["test"], cs, N)
            pc = pq_costs(cs)
            print(f"  {sch:10s} ({pc['alg']}, {pc['sig_bytes']} B, sign {pc['sign_s'] * 1e3:.3f} ms, "
                  f"verify {pc['verify_s'] * 1e3:.3f} ms)  " +
                  "  ".join(f"{a} {pq[a][sch]['completion']:.3f}s forged {pq[a][sch]['forge_accepted']:.0f}/"
                            f"{pq[a][sch]['forge_attempts']:.0f}" for a in pqc_methods))
        plot_pqc(pq, pqc_methods, schemes, os.path.join(out, "pqc_signatures.png"))
        rows = []
        for a in pqc_methods:
            for sc in schemes:
                m, pc = pq[a][sc], pq_costs(dict(cfg, sig_scheme=sc))
                rows.append({"Scheme": pc["alg"], "Security": pc["level"], "Algorithm": a,
                             "Signature (B)": pc["sig_bytes"], "Sign (ms)": round(pc["sign_s"] * 1e3, 4),
                             "Verify (ms)": round(pc["verify_s"] * 1e3, 4),
                             "Completion time (s)": round(m["completion"], 3),
                             "Consensus delay (ms)": round(m["cons_ms"], 1),
                             "Forgeries attempted": round(m["forge_attempts"], 1),
                             "Forgeries accepted": round(m["forge_accepted"], 1),
                             "Tampered committed": round(m["tamper_committed"], 1)})
        pd.DataFrame(rows).to_csv(os.path.join(out, "pqc.csv"), index=False)
        print_table("Signature schemes under a quantum adversary (test set)",
                    pd.DataFrame(rows).drop(columns=["Security"]))
        summary["pqc"] = pq

    hid_methods = [m for m in ("CoSched-GNN", "TaBRL", "FE-CTDE-style", "HGAT-style", "HEFT+Random")
                   if m in CO_ALGOS]
    if "hidden" in experiments:
        print("\n######## Hidden consensus fraction vs consensus-delay level ########")
        train_and_eval(hid_methods, cfg, train_jobs, {"test": wins["test"]}, agents, hists, res)
        levels = cfg["cons_levels"]
        hd = {a: {} for a in hid_methods}
        for lv in levels:
            cs = dict(cfg, cons_scale=lv)
            for a in hid_methods:
                hd[a][lv] = co_evaluate(agents[a], wins["test"], cs, N)
            print(f"  x{lv:<5} " + "  ".join(f"{a} {hd[a][lv]['completion']:.3f}s hid {hd[a][lv]['hidden']:.0%}"
                                             for a in hid_methods))
        plot_lines(list(levels), hd, "Consensus-delay level (x RTT and result-check cycles)",
                   os.path.join(out, "hidden_vs_level.png"), "Consensus hidden in DAG slack",
                   [("hidden", "Hidden consensus fraction (%)", 100), ("completion", "Mean completion time (s)", 1)])
        rows = [dict(Level=lv, Algorithm=a, **{"Completion time (s)": round(hd[a][lv]["completion"], 3),
                                               "Consensus delay (ms)": round(hd[a][lv]["cons_ms"], 1),
                                               "Hidden consensus (%)": round(100 * hd[a][lv]["hidden"], 1)})
                for a in hid_methods for lv in levels]
        pd.DataFrame(rows).to_csv(os.path.join(out, "hidden.csv"), index=False)
        print_table("Hidden consensus vs level (test set)", pd.DataFrame(rows))
        summary["hidden"] = hd

    if "scaling" in experiments:
        print("\n######## Zero-shot scaling (trained with N, M, k_max of the main run) ########")
        train_and_eval(methods, cfg, train_jobs, {"test": wins["test"]}, agents, hists, res)
        trained = [a for a in methods if a in agents]
        sc_rows = []
        veh = {a: {} for a in trained}
        for nv in cfg["scale_veh"]:
            w = fixed_windows(test_jobs, nv, cfg["horizon"], cfg, cfg["eval_windows"], seed=2)
            for a in trained:
                veh[a][nv] = co_evaluate(agents[a], w, cfg, nv)
            print(f"  N={nv:2d}  " + "  ".join(f"{a} {veh[a][nv]['completion']:.2f}s" for a in trained))
        plot_lines(list(cfg["scale_veh"]), veh, "Number of vehicles N", os.path.join(out, "scale_vehicles.png"),
                   f"Zero-shot: trained with N = {N}", [("completion", "Mean completion time (s)", 1),
                                                       ("throughput", "Processing capacity (bit/s)", 1)])
        size_free = [a for a in trained if a in GRAPH_METHODS]
        srv = {a: {} for a in size_free}
        for m in cfg["scale_res_co"]:
            cm = dict(cfg, n_res=m)
            w = fixed_windows(test_jobs, N, cfg["horizon"], cm, cfg["eval_windows"], seed=2)
            for a in size_free:
                srv[a][m] = co_evaluate(agents[a], w, cm, N)
            print(f"  M={m:2d}  " + "  ".join(f"{a} {srv[a][m]['completion']:.2f}s" for a in size_free))
        plot_lines(list(cfg["scale_res_co"]), srv, "Number of edge servers M", os.path.join(out, "scale_servers.png"),
                   f"Zero-shot: trained with M = {cfg['n_res']} (fixed-size FC agents cannot run at other M)",
                   [("completion", "Mean completion time (s)", 1), ("cons_ms", "Consensus delay (ms)", 1)])
        dag = {a: {} for a in size_free}
        for k in cfg["scale_k"]:
            ck = dict(cfg, k_max=k)
            _, te_k = load_dag_jobs(ck)
            w = fixed_windows(te_k, N, cfg["horizon"], ck, cfg["eval_windows"], seed=2)
            for a in size_free:
                dag[a][k] = co_evaluate(agents[a], w, ck, N)
            print(f"  k<={k:2d} " + "  ".join(f"{a} {dag[a][k]['completion']:.2f}s" for a in size_free))
        plot_lines(list(cfg["scale_k"]), dag, "Largest DAG size (subtasks)", os.path.join(out, "scale_dag_size.png"),
                   f"Zero-shot: trained with DAGs of up to {cfg['k_max']} subtasks",
                   [("completion", "Mean completion time (s)", 1), ("hidden", "Hidden consensus fraction", 1)])
        for tag, d in (("vehicles", veh), ("servers", srv), ("dag_size", dag)):
            for a, pts in d.items():
                for x, m in pts.items():
                    sc_rows.append({"Sweep": tag, "Size": x, "Algorithm": a,
                                    "Completion time (s)": round(m["completion"], 3),
                                    "Processing capacity (kbit/s)": round(m["throughput"] / 1e3, 1),
                                    "Consensus delay (ms)": round(m["cons_ms"], 1),
                                    "Hidden consensus (%)": round(100 * m["hidden"], 1)})
        pd.DataFrame(sc_rows).to_csv(os.path.join(out, "scaling.csv"), index=False)
        print_table("Zero-shot scaling (test set)", pd.DataFrame(sc_rows))
        summary["scaling"] = dict(vehicles=veh, servers=srv, dag_size=dag)

    plot_gantt(os.path.join(out, "gantt_example.png"))
    with open(os.path.join(out, "results.json"), "w") as fh:
        json.dump(dict(summary=summary, fig3=res, history=hists, cfg=cfg), fh, default=float)
    print(f"\nFigures and tables written to {out}/")
    return res, summary, agents


def multi_seed(cfg, seeds, experiments, methods):
    """Repeat the main comparison for several seeds; mean +- std and Welch's t-test against TaBRL."""
    base = cfg["out_dir"]
    per = {}
    for s in seeds:
        print(f"\n################ seed {s} ################")
        res, _, _ = co_main(dict(cfg, seed=s, out_dir=os.path.join(base, f"seed_{s}")), experiments, methods)
        per[s] = res.get("test", {})
    algs = [a for a in CO_ORDER + CO_EXTRA if all(a in per[s] for s in seeds)]
    rows = []
    for a in algs:
        ct = np.array([per[s][a]["completion"] for s in seeds])
        cp = np.array([per[s][a]["throughput"] for s in seeds]) / 1e3
        row = {"Algorithm": a, "Completion mean (s)": round(ct.mean(), 3), "Completion std": round(ct.std(ddof=1), 3),
               "Capacity mean (kbit/s)": round(cp.mean(), 1), "Capacity std": round(cp.std(ddof=1), 1)}
        if "TaBRL" in algs and a != "TaBRL" and len(seeds) > 1:
            tb = np.array([per[s]["TaBRL"]["completion"] for s in seeds])
            try:
                from scipy.stats import ttest_ind
                row["p vs TaBRL (completion)"] = round(float(ttest_ind(ct, tb, equal_var=False).pvalue), 4)
            except ImportError:
                pass
        rows.append(row)
    t = pd.DataFrame(rows)
    t.to_csv(os.path.join(base, "seeds_summary.csv"), index=False)
    print_table(f"Test set over seeds {list(seeds)}", t)
    return t


CFG.update(CO_CFG)
CFG.update(cons_levels=(0.5, 1, 2, 5, 10), scale_res_co=(6, 8, 10, 12), scale_k=(8, 12, 16, 20))


def apply_co_args(cfg):
    import argparse
    apply_cli_args(cfg)
    ap = argparse.ArgumentParser()
    ap.add_argument("--experiments", default="main,ablation,pqc,hidden,scaling",
                    help="comma list from main, ablation, pqc, hidden, scaling")
    ap.add_argument("--methods", help="comma list of methods for the main comparison (default: the 7 paper methods; "
                         "also available: TaBRL-GNN, MA-LHTO, IOPO, DECO)")
    ap.add_argument("--seeds", help="comma list, e.g. 42,43,44,45,46 (repeats every experiment per seed)")
    ap.add_argument("--servers", type=int, help="number of edge servers M (default 8)")
    ap.add_argument("--pbft-f", type=int, help="faults tolerated f (n = 3f + 1)")
    ap.add_argument("--sig", choices=list(SIG), help="signature scheme: ecdsa, dilithium2 (ML-DSA-44, "
                                                     "default), dilithium3 (ML-DSA-65), dilithium5 (ML-DSA-87)")
    ap.add_argument("--hidden", type=int, help="GNN hidden size (default 128)")
    ap.add_argument("--layers", type=int, help="GNN layers (default 3)")
    ap.add_argument("--rollout", type=int, help="PPO decisions per update (default 2048)")
    ap.add_argument("--synthetic", action="store_true", help="offline smoke test without the trace")
    ap.add_argument("--set", nargs="*", default=[], metavar="KEY=VALUE",
                    help="override any setting, e.g. --set ppo_gamma=0.95 lam_dl=2")
    ap.add_argument("--threads", type=int, help="CPU threads for PyTorch")
    args, _ = ap.parse_known_args()
    import ast
    for kv in args.set:
        k, v = kv.split("=", 1)
        if k not in cfg:
            raise ValueError(f"unknown setting {k}")
        try:
            cfg[k] = ast.literal_eval(v)
        except (ValueError, SyntaxError):
            cfg[k] = v
    if args.threads:
        torch.set_num_threads(args.threads)
    for flag, key in [("servers", "n_res"), ("pbft_f", "pbft_f"), ("sig", "sig_scheme"),
                      ("hidden", "co_hidden"), ("layers", "co_layers"), ("rollout", "ppo_rollout")]:
        if getattr(args, flag) is not None:
            cfg[key] = getattr(args, flag)
    if args.synthetic:
        cfg["allow_synthetic"] = True
    if cfg["out_dir"] == "results":
        cfg["out_dir"] = "results_cosched"
    exps = tuple(x.strip() for x in args.experiments.split(",") if x.strip())
    methods = [m.strip() for m in args.methods.split(",")] if args.methods else None
    seeds = [int(s) for s in args.seeds.split(",")] if args.seeds else None
    return cfg, exps, methods, seeds


def co_run(cfg, exps, methods, seeds):
    if seeds:
        return multi_seed(cfg, seeds, exps, methods)
    return co_main(cfg, exps, methods)


co_run = logged(co_run, "cosched_log.txt")


if __name__ == "__main__":
    co_run(*apply_co_args(CFG))


In [ ]:
%%writefile test_cosched.py
"""Unit tests for CoSched-GNN (spec section 11.4). Run: python test_cosched.py"""
import itertools
import numpy as np
import torch

from cosched_gnn import (GANTT_DAG, static_schedule, rank_slack, exposed_delay, t_cons_formula,
                         seq_logp, co_cfg, CoEnv, Task, VJob, Round, CoSchedGNN, CPGreedy, HEFTRandom,
                         synthetic_batch_task, load_dag_jobs, make_window, run_co_episode, set_seed)

import tabrl_iov
# the tests use small synthetic DAG jobs; never read the 4M-row Alibaba trace (slow on Kaggle)
tabrl_iov.find_trace_file = lambda path=None: None

G = GANTT_DAG
TOL = 1e-9


def dag_durations(place, t_cons=None):
    tc = G["t_cons"] if t_cons is None else t_cons
    return [G["t_loc"][v] if place[v] == 0 else G["t_edge"][v] + tc[v] for v in range(8)]


def test_driving_dag_makespan_and_slack():
    mk = static_schedule(G["preds"], G["place"], G["t_loc"], G["t_edge"], G["t_cons"])[0]
    assert abs(mk - 2.9) < TOL, mk
    _, slack, _, _, _ = rank_slack(G["preds"], dag_durations(G["place"]))
    want = dict(A=0, B=0, D=0, G=0, H=0, C=0.2, E=0.8, F=1.6)
    for nm, s in want.items():
        assert abs(slack["ABCDEFGH".index(nm)] - s) < 1e-9, (nm, slack)


def test_local_e_f_single_cpu():
    place = list(G["place"]); place[4] = place[5] = 0
    mk = static_schedule(G["preds"], place, G["t_loc"], G["t_edge"], G["t_cons"])[0]
    assert abs(mk - 3.4) < TOL, mk


def test_raised_consensus_on_slack_branch():
    tc = list(G["t_cons"]); tc[4] = 1.5
    mk = static_schedule(G["preds"], G["place"], G["t_loc"], G["t_edge"], tc)[0]
    assert abs(mk - 3.1) < TOL, mk
    ex = exposed_delay(G["preds"], G["place"], G["t_loc"], G["t_edge"], tc, 4)
    assert abs(ex - 0.2) < TOL, ex
    # at the base level E's consensus is fully hidden in its slack
    assert exposed_delay(G["preds"], G["place"], G["t_loc"], G["t_edge"], G["t_cons"], 4) < TOL


def _episode(agent_cls, seed=0, **kw):
    cfg = co_cfg(allow_synthetic=True, horizon=30, seed=seed, co_hidden=32, co_layers=2, **kw)
    set_seed(seed)
    jobs, _ = load_dag_jobs(cfg)
    env0 = CoEnv(cfg, cfg["n_veh"], 0)
    agent = agent_cls(env0, cfg)
    sched = make_window(jobs, cfg["n_veh"], cfg["horizon"], cfg, np.random.default_rng(seed))
    tasks = []
    orig = CoEnv.submit

    def spy(self, task, *a, **k):
        tasks.append(task)
        return orig(self, task, *a, **k)
    CoEnv.submit = spy
    try:
        m = run_co_episode(agent, sched, cfg, cfg["n_veh"], seed=seed, train=False)
    finally:
        CoEnv.submit = orig
    return cfg, tasks, m


def test_safety_invariant():
    for cls in (CoSchedGNN, CPGreedy, HEFTRandom):
        cfg, tasks, _ = _episode(cls)
        fmax = max(max(cfg["res_freq"]), cfg["f_local_max"])
        for task in tasks:
            if task.complete is None:
                continue
            for s in task.subs:
                if not s.preds:
                    continue
                parents = max(p.done_at for p in s.preds)
                end = task.info["cd"].get(s.k, s.done_at)
                assert end >= parents + s.b / fmax - 1e-6, (cls.__name__, end, parents)


def test_executor_mask():
    for cls in (CoSchedGNN, CPGreedy):
        cfg, tasks, m = _episode(cls)
        n = 3 * cfg["pbft_f"] + 1
        for task in tasks:
            ex = {y for x, y in zip(task.info["x"], task.info["y"]) if x}
            C = task.info["committee"]
            assert len(C) == n and len(set(C)) == n
            assert not ex & set(C), (cls.__name__, ex, C)
            assert cfg["n_res"] - len(ex) >= n
        assert m["self_val"] == 0.0


def test_fixed_committee_zero_load_constant():
    cfg = co_cfg()
    env = CoEnv(dict(cfg, allow_synthetic=True), 2, 0)
    C, u = [0, 1, 2, 3], np.zeros(env.M)
    vals = {round(t_cons_formula(cfg, env.rtt, env.Fk, u, C), 12) for _ in range(5)}
    assert len(vals) == 1


def test_validator_logp_bruteforce():
    torch.manual_seed(0)
    M, n = 6, 4
    s = torch.randn(1, M)
    mask = torch.ones(1, M)
    total = 0.0
    for seq in itertools.permutations(range(M), n):
        lp, _ = seq_logp(s, mask, torch.tensor([seq]))
        # brute force: product of softmax over the remaining servers
        p, left = 1.0, list(range(M))
        for k in seq:
            e = torch.exp(s[0, left]); p *= float(torch.exp(s[0, k]) / e.sum()); left.remove(k)
        assert abs(float(lp.exp()) - p) < 1e-6
        total += p
    assert abs(total - 1.0) < 1e-6


def test_ppo_logp_replay_matches():
    """The log-probability stored at act time equals the one recomputed in the PPO update."""
    cfg = co_cfg(allow_synthetic=True, horizon=20, co_hidden=32, co_layers=2, ppo_rollout=10 ** 9)
    jobs, _ = load_dag_jobs(cfg)
    for kw in ({}, dict(encoder="mlp"), dict(exec_mask=False), dict(val_mode="postload_rule")):
        agent = CoSchedGNN(CoEnv(cfg, 10, 0), cfg, **kw)
        sched = make_window(jobs, 10, 20, cfg, np.random.default_rng(0))
        run_co_episode(agent, sched, cfg, 10, seed=1, train=True)
        recs = agent.buf[:64]
        with torch.no_grad():
            lp, _, _ = agent.evaluate_batch(recs)
        old = np.array([r["logp"] for r in recs])
        assert np.abs(lp.cpu().numpy() - old).max() < 1e-3, kw


def test_pq_signatures_real():
    """Real ML-DSA / ECDSA: a signature verifies, any change to the message is rejected."""
    import pq_crypto as PQ
    for sch in PQ.SCHEMES:
        sg = PQ.signer(sch)
        pk, sk = sg.keygen()
        sig = sg.sign(sk, b"result")
        assert sg.verify(pk, b"result", sig) and not sg.verify(pk, b"resulT", sig)
        pk2, _ = sg.keygen()
        assert not sg.verify(pk2, b"result", sig)                  # wrong signer
        assert (sg.recover_secret(pk) is None) == (sch != "ecdsa")  # Shor breaks only ECDSA
    k = PQ.KEM()
    pk, sk = k.keygen()
    ct, ss = k.encaps(pk)
    assert k.decaps(sk, ct) == ss and k.open(ss, k.seal(ss, b"x", b"a"), b"a") == b"x"


def test_quantum_adversary_forgery():
    """A rejected malicious executor forges a quorum certificate. A quantum attacker succeeds
    against ECDSA and fails against ML-DSA."""
    kw = dict(p_tamper=1.0, p_forge=1.0, quantum_adversary=True)
    _, _, m = _episode(CPGreedy, sig_scheme="dilithium2", **kw)
    assert m["forge_attempts"] > 0 and m["forge_accepted"] == 0 and m["tamper_committed"] == 0, m
    _, _, m = _episode(CPGreedy, sig_scheme="ecdsa", **kw)
    assert m["forge_attempts"] > 0 and m["forge_accepted"] > 0, m
    _, _, m = _episode(CPGreedy, sig_scheme="ecdsa", p_tamper=1.0, p_forge=1.0, quantum_adversary=False)
    assert m["forge_accepted"] == 0, m                         # classical attacker cannot forge


def test_results_altered_in_transit_are_rejected():
    cfg, tasks, m = _episode(CPGreedy, p_mitm=1.0)
    assert m["mitm_detected"] > 0 and m["tamper_committed"] == 0
    assert all(not t.info["tc"] for t in tasks)                # nothing altered was committed
    assert m["kem_handshakes"] > 0 and m["sig_ops"] > 0


def test_load_raises_tcons():
    cfg = co_cfg()
    env = CoEnv(dict(cfg, allow_synthetic=True), 2, 0)
    C, u = [0, 1, 2, 3], np.zeros(env.M)
    base = t_cons_formula(cfg, env.rtt, env.Fk, u, C)
    for k in C:
        u2 = u.copy(); u2[k] = 0.9
        assert t_cons_formula(cfg, env.rtt, env.Fk, u2, C) >= base
    u2 = u.copy(); u2[C] = 0.9
    assert t_cons_formula(cfg, env.rtt, env.Fk, u2, C) > base
    # the same in the event simulator: verification jobs queue behind other work
    tc = [_one_subtask_tcons(extra) for extra in (0.0, 2e9)]
    assert tc[1] > tc[0], tc


def _one_subtask_tcons(extra_cycles):
    cfg = co_cfg(allow_synthetic=True, n_malicious=0)
    env = CoEnv(cfg, 1, 0)
    job = dict(b=[0.5e9], d=[1e6], preds=[[]], start=0.0)
    task = Task(0, 0.0, job); task.deadline = 5.0
    if extra_cycles:
        dummy = Round(None, 0, False, 0.0, [], 0.0); dummy.decided = True
        for k in (1, 2, 3, 4):
            for core in range(int(env.cores[k])):
                env.res[k].q[0].append(VJob(dummy, k, True, extra_cycles, 0.0))
    env.submit(task, [1], [0], cfg["f_local_max"], [1, 2, 3, 4])
    for _ in range(30):
        env.begin_slot(); env.run_slot()
        if task.complete is not None:
            break
    return task.info["tc"][0]


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    for t in tests:
        t()
        print(f"PASS {t.__name__}")
    print(f"all {len(tests)} tests passed")


In [ ]:
%%writefile combine_seeds.py
"""Merge the results of several runs (one seed each, possibly from different Kaggle sessions)
into the final table: mean +- std over seeds and Welch's t-test against TaBRL.

    python combine_seeds.py /kaggle/input/run-seed42/cosched /kaggle/input/run-seed43/cosched ...
    python combine_seeds.py "/kaggle/input/*/cosched*"          (glob patterns are allowed)

Each folder must contain the results.json written by cosched_gnn.py (--experiments main).
Writes seeds_summary.csv (and prints it) into --out (default: current folder)."""
import argparse
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import glob
import json
import os

import numpy as np
import pandas as pd

METRICS = [("completion", "Completion (s)", 1), ("throughput", "Capacity (kbit/s)", 1e-3),
           ("objective", "P2 objective", 1), ("energy", "Energy (J)", 1),
           ("deadline_miss", "Deadline miss (%)", 100), ("cons_ms", "Consensus (ms)", 1),
           ("hidden", "Hidden (%)", 100), ("self_val", "Self-validation (%)", 100),
           ("tampered", "Tampered caught", 1), ("tamper_committed", "Tampered committed", 1)]


def load(folder):
    path = os.path.join(folder, "results.json")
    with open(path) as fh:
        d = json.load(fh)
    c = d["cfg"]
    return c.get("seed"), d["fig3"]["test"], (c.get("n_veh"), c.get("n_res"), c.get("pbft_f"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folders", nargs="+")
    ap.add_argument("--out", default=".")
    args = ap.parse_args()
    folders = sorted({f for p in args.folders for f in (glob.glob(p) or [p])
                      if os.path.exists(os.path.join(f, "results.json"))})
    runs = [load(f) for f in folders]
    if not runs:
        raise SystemExit("no results.json found in the given folders")
    setups = {f: st for f, (_, _, st) in zip(folders, runs)}
    if len(set(setups.values())) > 1:
        for f, st in setups.items():
            print(f"  {f}: vehicles N={st[0]}, servers M={st[1]}, f={st[2]}")
        raise SystemExit("these runs use different settings (N, M or f); merge only runs with the same setup")
    runs = [(s, r) for s, r, _ in runs]
    seeds = [s for s, _ in runs]
    if len(set(seeds)) < len(seeds):
        raise SystemExit(f"the same seed appears twice: {seeds}")
    st = next(iter(setups.values()))
    print(f"{len(runs)} runs, seeds {seeds}, vehicles N={st[0]}, servers M={st[1]}")
    algs = [a for a in runs[0][1] if all(a in r for _, r in runs)]
    rows = []
    for a in algs:
        row = {"Method": a}
        for key, name, sc in METRICS:
            v = np.array([r[a].get(key, np.nan) for _, r in runs], float) * sc
            row[name] = f"{np.nanmean(v):.3f} +- {np.nanstd(v, ddof=1) if len(v) > 1 else 0:.3f}"
        if a != "TaBRL" and "TaBRL" in algs and len(runs) > 1:
            x = np.array([r[a]["completion"] for _, r in runs])
            y = np.array([r["TaBRL"]["completion"] for _, r in runs])
            try:
                from scipy.stats import ttest_ind
                row["p vs TaBRL (completion)"] = round(float(ttest_ind(x, y, equal_var=False).pvalue), 4)
            except ImportError:
                pass
        rows.append(row)
    t = pd.DataFrame(rows)
    os.makedirs(args.out, exist_ok=True)
    t.to_csv(os.path.join(args.out, "seeds_summary.csv"), index=False)
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", None)
    print(t.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile replot.py
"""Redraw the result figures of a finished run without some methods (default: CP-Greedy+Post-load,
which the paper reports as ablation A6 instead of as a compared method). Nothing is retrained.

    python replot.py RUN_FOLDER --out figures_clean
    python replot.py RUN_FOLDER --drop "CP-Greedy+Post-load,HEFT+Random"

RUN_FOLDER holds either the results.json written by cosched_gnn.py (a Kaggle run's seed_XX folder),
or the CSV tables main_tables.csv / pqc.csv / hidden.csv / scaling.csv (results_kaggle_seed42)."""
import argparse
import glob
import json
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

import cosched_gnn as C


def num(k):
    try:
        f = float(k)
        return int(f) if f.is_integer() else f
    except (TypeError, ValueError):
        return k


def numkeys(d):
    return {a: {num(k): v for k, v in pts.items()} for a, pts in d.items()}


def from_json(path):
    with open(path) as fh:
        d = json.load(fh)
    s = d.get("summary", {})
    out = dict(res=d.get("fig3", {}), hists=d.get("history", {}), cfg=d.get("cfg", {}))
    if "main" in s and s["main"].get("fig4"):
        out["fig4"] = {num(k): v for k, v in s["main"]["fig4"].items()}
    if "pqc" in s:
        out["pqc"] = s["pqc"]
    if "hidden" in s:
        out["hidden"] = numkeys(s["hidden"])
    if "scaling" in s:
        out["scaling"] = {k: numkeys(v) for k, v in s["scaling"].items()}
    if "ablation" in s:
        out["ablation"] = s["ablation"]
    return out


SCHEME_KEY = {v[0]: k for k, v in C.SIG.items()}   # display name -> scheme key


def from_csv(folder):
    """Tables transcribed from a run log (results_kaggle_seed42)."""
    def metrics(r):
        m = dict(completion=r["completion"])
        if "throughput_kbit" in r: m["throughput"] = r["throughput_kbit"] * 1e3
        for k in ("energy", "objective", "offload", "cons_ms", "tampered", "tamper_committed",
                  "forge_attempts", "forge_accepted"):
            if k in r: m[k] = r[k]
        if "sys_throughput" in r: m["sys_throughput"] = r["sys_throughput"] * 1e6
        for k in ("deadline_miss", "hidden", "self_val"):
            if f"{k}_pct" in r: m[k] = r[f"{k}_pct"] / 100
        if "val_load" in r: m["val_util"] = r["val_load"]
        return m
    out = {}
    mt = pd.read_csv(os.path.join(folder, "main_tables.csv"))
    out["res"] = {sp: {r.Algorithm: metrics(r._asdict()) for r in g.itertuples()} for sp, g in mt.groupby("Split")}
    p = os.path.join(folder, "pqc.csv")
    if os.path.exists(p):
        t = pd.read_csv(p); out["pqc"] = {}
        for r in t.itertuples():
            out["pqc"].setdefault(r.Algorithm, {})[SCHEME_KEY.get(r.Scheme, r.Scheme)] = metrics(r._asdict())
    p = os.path.join(folder, "hidden.csv")
    if os.path.exists(p):
        t = pd.read_csv(p); out["hidden"] = {}
        for r in t.itertuples():
            out["hidden"].setdefault(r.Algorithm, {})[num(r.Level)] = metrics(r._asdict())
    p = os.path.join(folder, "scaling.csv")
    if os.path.exists(p):
        t = pd.read_csv(p); sc = {}
        for r in t.itertuples():
            sc.setdefault(r.Sweep, {}).setdefault(r.Algorithm, {})[num(r.Size)] = metrics(r._asdict())
        out["scaling"] = sc
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folder")
    ap.add_argument("--out", default="figures_clean")
    ap.add_argument("--drop", default="CP-Greedy+Post-load")
    args = ap.parse_args()
    drop = {m.strip() for m in args.drop.split(",") if m.strip()}
    js = glob.glob(os.path.join(args.folder, "results.json"))
    d = from_json(js[0]) if js else from_csv(args.folder)
    os.makedirs(args.out, exist_ok=True)
    keep = lambda names: [a for a in C.CO_ORDER + C.CO_EXTRA + list(names) if a in names and a not in drop]
    order = lambda names: list(dict.fromkeys(keep(names)))
    o = lambda f: os.path.join(args.out, f)
    made = []

    res = {sp: {a: m for a, m in v.items() if a not in drop} for sp, v in d["res"].items()}
    algs = order(res.get("test", {}))
    if "train" in res and "test" in res:
        C.plot_co_fig3(res, algs, o("fig3_completion_capacity.png")); made.append("fig3_completion_capacity.png")
    C.plot_consensus(res, algs, o("consensus_security.png")); made.append("consensus_security.png")
    C.plot_paper_comparison(res, algs, o("paper_comparison.png")); made.append("paper_comparison.png")
    C.paper_table(res, algs).to_csv(o("paper_comparison.csv"), index=False)
    if d.get("fig4"):
        f4 = {nv: {a: m for a, m in v.items() if a not in drop} for nv, v in d["fig4"].items()}
        C.with_order(algs, C.plot_fig4, f4, o("fig4_backlog_energy.png")); made.append("fig4_backlog_energy.png")
    if d.get("hists"):
        h = {a: v for a, v in d["hists"].items() if a not in drop and a in algs}
        C.with_order(algs, C.plot_training, h, o("training_curves.png")); made.append("training_curves.png")
    if d.get("pqc"):
        pm = order(d["pqc"])
        schemes = [s for s in C.SIG if all(s in d["pqc"][a] for a in pm)]
        C.plot_pqc(d["pqc"], pm, schemes, o("pqc_signatures.png")); made.append("pqc_signatures.png")
    if d.get("hidden"):
        hd = {a: d["hidden"][a] for a in order(d["hidden"])}
        lv = sorted(next(iter(hd.values())))
        C.plot_lines(lv, hd, "Consensus-delay level (x RTT and result-check cycles)", o("hidden_vs_level.png"),
                     "Consensus hidden in DAG slack",
                     [("hidden", "Hidden consensus fraction (%)", 100), ("completion", "Mean completion time (s)", 1)])
        made.append("hidden_vs_level.png")
    sc = d.get("scaling", {})
    cfg = d.get("cfg", {})
    spec = {"vehicles": ("scale_vehicles.png", "Number of vehicles N",
                         f"Zero-shot: trained with N = {cfg.get('n_veh', 10)}",
                         [("completion", "Mean completion time (s)", 1), ("throughput", "Processing capacity (bit/s)", 1)]),
            "servers": ("scale_servers.png", "Number of edge servers M",
                        f"Zero-shot: trained with M = {cfg.get('n_res', 8)} (fixed-size FC agents cannot run at other M)",
                        [("completion", "Mean completion time (s)", 1), ("cons_ms", "Consensus delay (ms)", 1)]),
            "dag_size": ("scale_dag_size.png", "Largest DAG size (subtasks)",
                         f"Zero-shot: trained with DAGs of up to {cfg.get('k_max', 8)} subtasks",
                         [("completion", "Mean completion time (s)", 1), ("hidden", "Hidden consensus fraction", 1)])}
    for key, (fn, xl, title, keys) in spec.items():
        if key in sc:
            data = {a: sc[key][a] for a in order(sc[key])}
            keys = [k for k in keys if all(k[0] in m for pts in data.values() for m in pts.values())]
            if not data or not keys:
                continue
            xs = sorted({x for pts in data.values() for x in pts})
            C.plot_lines(xs, data, xl, o(fn), title, keys); made.append(fn)
    print(f"dropped {sorted(drop)}; wrote {len(made)} figures to {args.out}/:")
    for f in made:
        print("  " + f)


if __name__ == "__main__":
    main()


## Unit tests (must print `all 12 tests passed`)

In [ ]:
!python test_cosched.py

## Run
Results of each seed go to `OUT/seed_<seed>/`.

In [ ]:
!python -u cosched_gnn.py --epochs {EPOCHS} --seeds {SEEDS} --servers {SERVERS} --set n_veh={VEHICLES} --experiments {EXPS} --out {OUT}

## Result tables

In [ ]:
import glob, os
import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", None); pd.set_option("display.width", 250)
for f in ["paper_comparison.csv", "results.csv", "ablation.csv", "pqc.csv", "hidden.csv",
          "scaling.csv", "fig4_values.csv", "seeds_summary.csv"]:
    for p in sorted(glob.glob(os.path.join(OUT, "**", f), recursive=True)):
        print(p); display(pd.read_csv(p))

## Figures

In [ ]:
import glob, os
from IPython.display import Image, display
for f in ["paper_comparison.png", "fig3_completion_capacity.png", "consensus_security.png",
          "fig4_backlog_energy.png", "training_curves.png", "ablation.png", "pqc_signatures.png",
          "hidden_vs_level.png", "scale_vehicles.png", "scale_servers.png", "scale_dag_size.png",
          "gantt_example.png"]:
    for p in sorted(glob.glob(os.path.join(OUT, "**", f), recursive=True)):
        print(p); display(Image(filename=p, width=900))

## Save this run as a zip (download it from the Output panel)

In [ ]:
!cd /kaggle/working && zip -qr {OUT.split("/")[-1]}.zip {OUT.split("/")[-1]} && ls -lh *.zip

## Last step (separate small notebook, no GPU): merge the seeds
Add the outputs of Runs A, C and D as inputs (*Add Input -> Notebooks -> Your Work*), run the Code cells above that write `combine_seeds.py`, then this cell. It finds every seed's `results.json` under `/kaggle/input` (the ablation run is skipped).

In [ ]:
# !python combine_seeds.py $(find /kaggle/input -name results.json -path "*seed_*" -not -path "*ablation*" -exec dirname {} \;) --out /kaggle/working/final